In [ ]:
# ====================================================================================================================
# MT2KD ENHANCEMENT, STEP 1 (ASR half) — WavLM-Large -> Zipformer-M feature KD + CTC teacher comparison — SINGLE CELL, end to end, resumable
#
# WHAT THIS IS: the same pipeline as your verified ASR v4 cell (sanity_2h: 2 h of train-clean-100, ScaledAdam + Eden, same student, same
#   checks), with ONE change: the teacher is WavLM-Large instead of Whisper-large-v3. WavLM-Large is the teacher planned to replace BOTH
#   Whisper (ASR) and ECAPA-TDNN (speaker verification) in our enhancement; this cell tests the ASR half.
#   Then a CTC PROBE answers the real question: freeze each student, train the same small BiLSTM-CTC on top (10 h with transcripts),
#   and measure WER / CER on dev-clean for:  WavLM-taught student | Whisper-taught student (your v4 best.pt) | random-init student (control).
#
#   teacher : microsoft/wavlm-large (315M, frozen, fp16), layer 24 (configurable), 1024-D @ 50 Hz -> pairs (2j, 2j+1) -> 2048-D @ 25 Hz
#             raw waveform input, whole utterance (no 30 s window, no chunking, no padding)
#   student : Zipformer-M (icefall), 80-D fbank, 512-D @ 25 Hz, Linear 512 -> 2048 (as in v4, only the projection size changes)
#   loss    : masked L1 on time-aligned frames (alignment MEASURED from the WavLM and Zipformer code, as in v4)
#   GPU     : T4 works (fp16 + GradScaler, ~3-4 h end to end, resumable); A100 ~1.5 h. Peak GPU memory stays well below 16 GB.
# STORAGE: Drive gets ONLY 5 files per folder (best.pt, latest.pt, history.csv, final_report.json, config.json):
#            /content/drive/MyDrive/MT2KD/results/WavLM_ASR_Stage1_sanity_2h/     (KD training)
#            /content/drive/MyDrive/MT2KD/results/ASR_teacher_comparison_ctc/     (probe: final_report.json, history.csv, config.json)
#          Your Whisper run folder ASR_Stage1_v4_sanity_2h is only READ. Everything else lives in /content (rebuilt automatically).
# RESUME:  re-run this cell after ANY interruption (finished parts are skipped / resumed from the last checkpoint).
# ====================================================================================================================
import os
os.environ.setdefault("HF_HOME", "/content/hf_cache")      # local disk (must be set before transformers is imported)

# ####################################################################################################
# ---- asr1_common ----
# ####################################################################################################
# ===========================================================================================
# MT2KD ASR Stage 1 (WavLM-Large -> Zipformer-M) | shared configuration, paths, helpers, audit of existing work
# Every step of this pipeline is resumable: re-running a cell after a Colab disconnect continues where it stopped.
# ===========================================================================================
import os, sys, json, math, time, shutil, hashlib, subprocess, importlib.util
from pathlib import Path

CODE_VERSION = "asr1-wavlm-v1"              # bumped whenever a change affects features, alignment or training semantics

CFG = dict(
    # ---- READ-ONLY inputs on Drive (nothing is ever written below /content/drive except by class DriveStore)
    project_root="/content/drive/MyDrive/MT2KD",                       # only read (old v1 checkpoint, read-only, for an architecture comparison)
    librispeech_root="/content/drive/MyDrive/MT2KD/datasets/LibriSpeech",  # only read: LibriSpeech .tar.gz or extracted folders (searched first)
    librispeech_search=["/content/drive/MyDrive/MT2KD/datasets", "/content/drive/MyDrive/MT2KD", "/content/drive/MyDrive/LibriSpeech",
                        "/content/drive/MyDrive/datasets", "/content/drive/MyDrive"],   # also searched (read-only, 3 folder levels deep)
    librispeech_download=True,                       # not found on Drive -> download the official openslr archive to LOCAL disk (never to Drive)
    download_dir="/content/librispeech_dl",
    whisper_student_ckpt="/content/drive/MyDrive/MT2KD/results/ASR_Stage1_v4_sanity_2h/best.pt",   # your Whisper-taught student (read-only)
    # ---- Drive output folders (5 files each, written by DriveStore only)
    drive_out_root="/content/drive/MyDrive/MT2KD/results",   # KD run: <root>/WavLM_ASR_Stage1_<run>/ ; probe: <root>/ASR_teacher_comparison_ctc/
    drive_api=True,                                  # Google sign-in once per runtime: confirms uploads (md5) + deletes this folder's replaced checkpoints from the trash
    # ---- everything else goes to the Colab VM's local disk (lost when the runtime is recycled; regenerated automatically)
    write_root="/content/MT2KD_wavlm_work",
    local_audio="/content/ls_local",                 # session-local audio copy (rebuilt automatically after a restart)
    local_features="/content/wavlm_local",           # session-local copy of the teacher cache (fast random access)
    zipformer_dir="/content/MT2KD_zipformer_asr1",   # private copy of icefall's Zipformer (same files as the v4 cell)
    seed=42,
    subsets={"2h": 2.0, "10h": 10.0},                # hours (nested; the 2 h subset is the SAME as in the Whisper v4 run): 2 h = KD, 10 h = CTC probe
    # ---- teacher: WavLM-Large, 1024-D @ 50 Hz; two successive frames concatenated -> 2048-D @ 25 Hz
    teacher_model="microsoft/wavlm-large",
    teacher_layer=24,                                # 24 = final layer (= last_hidden_state), the analogue of Whisper's encoder output
    teacher_normalize=True,                          # WavLM-Large expects per-utterance normalised waveforms
    teacher_workers=2, shard_utts=256,
    # ---- student / KD
    align="shift",                                   # frame pairing from MEASURED frame centres; "truncate" = no offset (comparison only)
    target_norm="none",                              # paper: plain L1 on the raw teacher features
    init_bias="train_mean",                          # projection bias starts at the mean aligned teacher frame (engineering choice)
    # ---- optimisation: paper / icefall Zipformer recipe = ScaledAdam + Eden (identical to the Whisper v4 run)
    optimizer="scaled_adam",
    base_lr=0.045, lr_batches=7500, lr_epochs=3.5, eden_warmup_batches=500, eden_warmup_start=0.1, clipping_scale=2.0,
    lr=1.5e-4, weight_decay=1e-2, warmup_updates=500, min_lr_frac=0.05, grad_clip=1.0,      # used only when optimizer == "adamw"
    max_batch_seconds=1000.0, max_batch_utts=128, accum_steps=1, ref_duration=600.0,
    mem_fraction=0.85,                               # memory probe: largest batch whose measured peak stays below this share of the GPU
    max_consecutive_skips=25,
    max_session_hours=None,                          # e.g. 2.0: stop this cell run after ~2 h of training (state saved, resumable)
    # ---- validation / early stopping / checkpoints
    val_split="dev-clean", patience=5, min_delta=1e-4, overfit_gap_ratio=1.15,
    local_ckpt_every_updates=100, drive_ckpt_every_updates=500, ckpt_every_minutes=15.0,   # + both at every epoch end
    log_every=50, num_workers=min(8, os.cpu_count() or 2),
    # ---- CTC probe (identical for every student; fixed epochs, no model selection on dev-clean)
    probe_train_subset="10h", probe_epochs=30, probe_batch_seconds=200.0, probe_hidden=384, probe_layers=2, probe_dropout=0.1,
    probe_lr=1e-3, probe_seed=1234,
)
PRESETS = {
    # identical to the Whisper v4 "sanity_2h" run (same subset, epochs, batches, patience), so the two students are directly comparable
    "sanity_2h": dict(train_subset="2h", max_epochs=30, max_batch_seconds=120.0, patience=8, gate_ratio=0.90),
}
RUN_PRESETS = ["sanity_2h"]
def drive_dir(preset): return f"{CFG['drive_out_root']}/WavLM_ASR_Stage1_{preset}"
def probe_drive_dir(): return f"{CFG['drive_out_root']}/ASR_teacher_comparison_ctc"
OFFICIAL_UTTS = {"train-clean-100": 28539, "dev-clean": 2703, "test-clean": 2620}
SR = 16000

def log(*a): print(*a, flush=True)

DRIVE = "/content/drive"
def assert_local(p):
    """Every LOCAL write goes through here: refuses anything on Google Drive (Drive writes exist only in class DriveStore)."""
    rp = os.path.realpath(str(p))
    if rp.startswith(DRIVE) or str(p).startswith(DRIVE): raise PermissionError(f"refusing to write to Google Drive: {p}")
    return p

def paths(preset=None):
    R = Path(CFG["project_root"]); W = Path(assert_local(CFG["write_root"]))
    for k in ("local_audio", "local_features", "zipformer_dir", "download_dir"): assert_local(CFG[k])
    P = dict(
        root=R, ls=Path(CFG["librispeech_root"]),                         # read-only (Drive)
        meta=W / "datasets" / "LibriSpeech",                              # inventories, manifests, selection record (local)
        features=W / "features" / "wavlm",
        ckpt=W / "checkpoints" / "ASR_Stage1", logs=W / "logs" / "ASR_Stage1", results=W / "results" / "ASR_Stage1")
    if preset:
        run = f"{preset}_{PRESETS[preset]['train_subset']}"
        for k in ["ckpt", "logs", "results"]: P[k] = P[k] / run
        P.update(best=P["ckpt"] / "best", latest=P["ckpt"] / "latest")
    return P

def ensure_dirs(P):
    for k, v in P.items():
        if k not in ("root", "ls") and isinstance(v, Path): assert_local(v); v.mkdir(parents=True, exist_ok=True)

def pip_if_missing(*mods):
    for m in mods:
        mod, pkg = (m, m) if isinstance(m, str) else m
        if importlib.util.find_spec(mod) is None: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

def atomic_json(obj, p):
    p = Path(assert_local(p)); p.parent.mkdir(parents=True, exist_ok=True); tmp = p.with_name(p.name + ".tmp")
    json.dump(obj, open(tmp, "w"), indent=1, default=str); os.replace(tmp, p)

def atomic_torch_save(obj, p):
    import torch
    p = Path(assert_local(p)); p.parent.mkdir(parents=True, exist_ok=True); tmp = p.with_name(p.name + ".tmp"); torch.save(obj, tmp); os.replace(tmp, p)

def sha_of(items): return hashlib.sha1("\n".join(map(str, items)).encode()).hexdigest()[:12]

def file_sha1(p, chunk=1 << 22):
    h = hashlib.sha1()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""): h.update(b)
    return h.hexdigest()

def code_fingerprint():
    """Single-cell version: the smoke-test report is tied to CODE_VERSION (plus the cache folder and the measured alignment, checked separately)."""
    return CODE_VERSION

def set_seed(s):
    import random, numpy as np, torch
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def oom_error():
    """CUDA out-of-memory exception class (torch.OutOfMemoryError exists only in newer PyTorch)."""
    import torch
    return getattr(torch, "OutOfMemoryError", None) or torch.cuda.OutOfMemoryError

def gpu_mem():
    import torch
    if not torch.cuda.is_available(): return {}
    return dict(gpu_alloc_gib=torch.cuda.memory_allocated() / 2**30, gpu_peak_gib=torch.cuda.max_memory_allocated() / 2**30,
                gpu_reserved_gib=torch.cuda.memory_reserved() / 2**30, gpu_total_gib=torch.cuda.get_device_properties(0).total_memory / 2**30)

def disk_free_gib(p):
    p = Path(p)
    while not p.exists(): p = p.parent
    return shutil.disk_usage(p).free / 2**30

def require_space(path, need_gib, what, margin_gib=1.0):
    free = disk_free_gib(path)
    if need_gib > free - margin_gib: raise RuntimeError(f"not enough space for {what}: need {need_gib:.1f} GiB (+{margin_gib} margin), {free:.1f} GiB free at {path}")
    return free

def env_report():
    """Environment check (nothing is installed or downgraded)."""
    import torch, platform
    r = dict(python=platform.python_version(), torch=torch.__version__, cuda=torch.version.cuda, gpu=None, code_version=CODE_VERSION)
    try:
        import torchaudio; r["torchaudio"] = torchaudio.__version__
    except Exception as e: r["torchaudio"] = f"missing ({e})"
    try:
        import transformers; r["transformers"] = transformers.__version__
    except Exception: r["transformers"] = "missing (installed on first teacher use)"
    try:
        import k2; r["k2"] = getattr(k2, "__version__", "installed")
    except Exception: r["k2"] = "not installed (pure-PyTorch Swoosh port used; verified in the smoke test)"
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0); r.update(gpu=p.name, gpu_gib=round(p.total_memory / 2**30, 1), compute_capability=f"{p.major}.{p.minor}",
                                                           amp="bfloat16" if p.major >= 8 else "float16 + GradScaler")
    r["disk_free_local_gib"] = round(disk_free_gib("/content" if Path("/content").exists() else "/"), 1)
    r["disk_free_drive_gib"] = round(disk_free_gib(CFG["project_root"]), 1) if Path("/content/drive/MyDrive").exists() else None
    for k, v in r.items(): log(f"  {k:22s}: {v}")
    if r["gpu"] is None: log("  WARNING: no GPU. Runtime > Change runtime type > T4 GPU")
    return r

# ####################################################################################################
# ---- asr1_data ----
# ####################################################################################################
# ===========================================================================================
# MT2KD ASR Stage 1 | LibriSpeech: official-split inventory, strict integrity checks, reproducible nested 2 h / 10 h subsets,
#                    manifests, local audio
#   * the inventory reads only FLAC headers (streamed through the .tar.gz, nothing extracted) -> exact durations
#   * every selected file is then fully decoded once (corrupt / wrong-rate / silent files are excluded and the selection redone)
#   * no duration filter: utterances longer than 30 s are kept (WavLM encodes every utterance whole)
# ===========================================================================================
import os, json, time, shutil, tarfile, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd

SELECTION_VERSION = 2          # v1 dropped utterances > 30 s; v2 keeps them
MANIFEST_COLS = ["uid", "speaker", "chapter", "split", "n_samples", "duration_s", "relpath", "text"]
DTYPES = {"uid": str, "speaker": str, "chapter": str, "text": str, "relpath": str}

def flac_streaminfo(head):
    """First bytes of a .flac -> (sample_rate, channels, bits, total_samples) from the STREAMINFO block, or None."""
    if len(head) < 42 or head[:4] != b"fLaC" or (head[4] & 0x7F) != 0: return None
    x = int.from_bytes(head[18:26], "big")
    return x >> 44, ((x >> 41) & 0x7) + 1, ((x >> 36) & 0x1F) + 1, x & ((1 << 36) - 1)

LS_SHA256 = {   # official archives (same values as torchaudio.datasets.LIBRISPEECH)
    "train-clean-100": "d4ddd1d5a6ab303066f14971d768ee43278a5f2a0aa43dc716b0e64ecbbbf6e2",
    "dev-clean": "76f87d090650617fca0cac8f88b9416e0ebf80350acb97b343a85fa903728ab3"}
LS_MIRRORS = ["https://www.openslr.org/resources/12", "https://openslr.elda.org/resources/12", "https://openslr.trmal.net/resources/12",
              "https://us.openslr.org/resources/12"]
_SRC = {}

def _search(root, split):
    root = Path(root)
    if not root.is_dir(): return None, None
    for t in [root / f"{split}.tar.gz", root / "tars" / f"{split}.tar.gz"]:
        if t.exists(): return "tar", t
    for base in [root, root / "LibriSpeech", root / "LibriSpeech" / "LibriSpeech"]:
        if (base / split).is_dir() and any((base / split).iterdir()): return "dir", base
    for dp, dn, fn in os.walk(root):
        if f"{split}.tar.gz" in fn: return "tar", Path(dp) / f"{split}.tar.gz"
        if split in dn and any((Path(dp) / split).iterdir()): return "dir", Path(dp)
        if len(Path(dp).relative_to(root).parts) >= 3: dn[:] = []
        dn[:] = [d for d in dn if not d.startswith(".") and d not in ("teacher_reps", "results", "checkpoints", "features")]
    return None, None

def download_split(split):
    """Official openslr archive -> LOCAL disk (resumable wget, sha256 checked). Never written to Drive."""
    d = Path(assert_local(CFG["download_dir"])); d.mkdir(parents=True, exist_ok=True); dst = d / f"{split}.tar.gz"; okf = d / f"{split}.sha256.ok"
    if dst.exists() and okf.exists() and okf.read_text() == LS_SHA256[split]: return dst
    require_space(d, 7.0 if "train" in split else 0.5, f"the {split} download", 3.0); part = d / f"{split}.tar.gz.part"
    for m in LS_MIRRORS:
        url = f"{m}/{split}.tar.gz"; log(f"[{split}] downloading {url} -> {part} (local disk; resumable) ..."); t0 = time.time()
        r = subprocess.run(["wget", "-c", "--tries=5", "--timeout=60", "--progress=dot:giga", "-O", str(part), url])
        if r.returncode: log(f"  download from {m} failed (wget exit {r.returncode}); trying the next mirror"); continue
        log(f"  downloaded {part.stat().st_size / 2**30:.2f} GiB in {(time.time() - t0) / 60:.1f} min; verifying sha256 ...")
        h = hashlib.sha256()
        with open(part, "rb") as f:
            for b in iter(lambda: f.read(1 << 24), b""): h.update(b)
        if h.hexdigest() == LS_SHA256[split]: os.replace(part, dst); okf.write_text(LS_SHA256[split]); log(f"  sha256 OK -> {dst}"); return dst
        log(f"  sha256 MISMATCH ({h.hexdigest()[:16]}...): file removed, trying the next mirror"); part.unlink()
    raise RuntimeError(f"could not download a verified {split}.tar.gz from {LS_MIRRORS}")

def find_source(split, download=False):
    """('tar', archive) or ('dir', folder containing <split>). Searches the configured Drive locations (read-only), then the local download;
    with download=True and nothing found, downloads the official archive to local disk."""
    if split in _SRC: return _SRC[split]
    for root in [CFG["librispeech_root"]] + list(CFG.get("librispeech_search", [])) + [CFG["download_dir"]]:
        if split not in LS_SHA256 and root == CFG["download_dir"]: continue
        kind, src = _search(root, split)
        if kind == "tar" and str(src).startswith(str(CFG["download_dir"])):
            okf = Path(CFG["download_dir"]) / f"{split}.sha256.ok"
            if not (okf.exists() and okf.read_text() == LS_SHA256.get(split)): kind = None
        if kind: log(f"[{split}] source: {kind} {src}"); _SRC[split] = (kind, src); return kind, src
    if download and CFG.get("librispeech_download") and split in LS_SHA256:
        log(f"[{split}] not found on Drive under {[CFG['librispeech_root']] + CFG['librispeech_search']}: downloading it to local disk")
        _SRC[split] = ("tar", download_split(split)); return _SRC[split]
    return None, None

def _parse_member(name):
    """'LibriSpeech/train-clean-100/19/198/19-198-0000.flac' -> (uid, speaker, chapter) or None (official naming only)."""
    p = name.replace("\\", "/").split("/")
    if len(p) < 3 or not p[-1].endswith(".flac"): return None
    uid = p[-1][:-5]; sp = uid.split("-")
    if len(sp) != 3 or sp[0] != p[-3] or sp[1] != p[-2] or not all(s.isdigit() for s in sp): return None
    return uid, sp[0], sp[1]

def inventory(split, rebuild=False):
    """Valid utterances of an official split (exact sample counts, transcripts). Invalid ones -> inventory_v2_<split>_dropped.csv."""
    P = paths(); P["meta"].mkdir(parents=True, exist_ok=True); f = P["meta"] / f"inventory_v2_{split}.csv"; fd = P["meta"] / f"inventory_v2_{split}_dropped.csv"
    if f.exists() and not rebuild:
        inv = pd.read_csv(f, dtype=DTYPES, keep_default_na=False); nd = len(pd.read_csv(fd)) if fd.exists() and fd.stat().st_size > 30 else 0
        log(f"[{split}] inventory (cached): {len(inv):,} valid utterances, {inv.duration_s.sum() / 3600:.2f} h, {nd} invalid (see {fd.name})"); return inv
    kind, src = find_source(split, download=True)
    if kind is None: raise FileNotFoundError(f"LibriSpeech {split} not found under {[CFG['librispeech_root']] + CFG['librispeech_search']} "
                                              f"(expected {split}.tar.gz or an extracted folder) and automatic download is off")
    rows, texts, t0 = [], {}, time.time()
    def add_texts(raw):
        for line in raw.splitlines():
            if line.strip():
                k, _, t = line.partition(" ")
                if k in texts: raise RuntimeError(f"[{split}] transcript id {k} appears twice")
                texts[k] = t.strip()
    if kind == "tar":
        log(f"[{split}] reading FLAC headers and transcripts from {src.name} (streamed, nothing extracted) ...")
        with tarfile.open(src, "r|gz") as tf:
            for m in tf:
                if not m.isfile() or f"/{split}/" not in f"/{m.name}": continue
                if m.name.endswith(".trans.txt"): add_texts(tf.extractfile(m).read().decode("utf-8")); continue
                pm = _parse_member(m.name)
                if pm: rows.append(dict(uid=pm[0], speaker=pm[1], chapter=pm[2], relpath=m.name, header=flac_streaminfo(tf.extractfile(m).read(64))))
                elif m.name.endswith(".flac"): rows.append(dict(uid=os.path.basename(m.name)[:-5], speaker="", chapter="", relpath=m.name, header="badname"))
    else:
        base = src / split; log(f"[{split}] reading FLAC headers and transcripts under {base} ...")
        files = []
        for dp, _, fns in os.walk(base):
            for fn in fns:
                full = os.path.join(dp, fn)
                if fn.endswith(".trans.txt"): add_texts(open(full, encoding="utf-8").read())
                elif fn.endswith(".flac"): files.append(full)
        def head(p):
            try:
                with open(p, "rb") as fh: return flac_streaminfo(fh.read(64))
            except OSError: return None
        with ThreadPoolExecutor(32) as ex: infos = list(ex.map(head, files))
        for full, si in zip(files, infos):
            rel = os.path.relpath(full, src); pm = _parse_member(rel)
            rows.append(dict(uid=pm[0], speaker=pm[1], chapter=pm[2], relpath=rel, header=si) if pm else
                        dict(uid=os.path.basename(full)[:-5], speaker="", chapter="", relpath=rel, header="badname"))
    good, bad = [], []
    for r in rows:
        h = r.pop("header"); reason = ""
        if h == "badname": reason = "file name/folder does not follow the official speaker-chapter-utterance layout"
        elif h is None: reason = "FLAC header unreadable (corrupt or not FLAC)"
        elif h[0] != SR: reason = f"sample rate {h[0]} Hz (expected {SR})"
        elif h[1] != 1: reason = f"{h[1]} channels (expected mono)"
        elif h[3] <= 0: reason = "zero / unknown length in header"
        elif r["uid"] not in texts: reason = "missing transcript"
        if reason: bad.append(dict(r, reason=reason))
        else: good.append(dict(r, n_samples=int(h[3]), text=texts[r["uid"]]))
    inv = pd.DataFrame(good, columns=["uid", "speaker", "chapter", "relpath", "n_samples", "text"]).sort_values("uid").reset_index(drop=True)
    if inv.uid.duplicated().any(): raise RuntimeError(f"[{split}] duplicate utterance ids: {inv.uid[inv.uid.duplicated()].tolist()[:5]}")
    orphan = sorted(set(texts) - set(inv.uid) - {b["uid"] for b in bad})
    for u in orphan: bad.append(dict(uid=u, speaker=u.split("-")[0], chapter=u.split("-")[1] if "-" in u else "", relpath="", reason="transcript without audio"))
    inv["split"] = split; inv["duration_s"] = inv.n_samples / SR
    n_files = len(rows); exp = OFFICIAL_UTTS.get(split)
    if exp and n_files != exp: raise RuntimeError(f"[{split}] {n_files:,} audio files found, the official split has {exp:,}: the copy is incomplete")
    pd.DataFrame(bad, columns=["uid", "speaker", "chapter", "relpath", "reason"]).to_csv(fd, index=False)
    inv[MANIFEST_COLS].to_csv(f, index=False)
    log(f"[{split}] {n_files:,} files: {len(inv):,} valid ({inv.speaker.nunique()} speakers, {inv.duration_s.sum() / 3600:.2f} h), {len(bad)} invalid -> {fd.name} | "
        f"longest {inv.duration_s.max():.1f} s, {int((inv.duration_s > 30).sum())} longer than 30 s (kept) | {time.time() - t0:.0f} s")
    return inv

def select_subsets(inv, seed, hours, exclude=()):
    """Reproducible nested subsets of COMPLETE utterances: valid utterances (minus `exclude`), sorted by id, permuted with a fixed seed;
    each subset is the shortest prefix of that order whose total duration reaches its target (so 2 h is a prefix of 10 h).
    No duration filter (WavLM encodes long utterances whole)."""
    pool = inv[~inv.uid.isin(set(exclude))].sort_values("uid").reset_index(drop=True)
    order = np.random.default_rng(seed).permutation(len(pool)); cum = pool.duration_s.values[order].cumsum(); out = {}
    for name, h in sorted(hours.items(), key=lambda kv: kv[1]):
        k = len(pool) if not math.isfinite(h) else int(np.searchsorted(cum, h * 3600.0)) + 1
        if k > len(pool): raise ValueError(f"subset {name}: only {cum[-1] / 3600:.1f} h available")
        out[name] = pool.iloc[order[:k]].assign(sel_rank=np.arange(k)).reset_index(drop=True)
    return out

def local_path(relpath): return str(Path(CFG["local_audio"]) / relpath)

def materialize(df, split):
    """Puts exactly the manifest's files on local disk (only missing ones; safe to re-run after a restart)."""
    need = [r for r in df.relpath if not os.path.exists(local_path(r))]
    if not need: log(f"[{split}] local audio ready ({len(df):,} files)"); return
    est = df[df.relpath.isin(set(need))].n_samples.sum() * 2 * 0.7 / 2**30           # FLAC ~70 % of 16-bit PCM for speech (upper estimate)
    require_space(CFG["local_audio"], est, f"{split} audio on local disk", 2.0)
    kind, src = find_source(split); t0 = time.time(); Path(CFG["local_audio"]).mkdir(parents=True, exist_ok=True)
    log(f"[{split}] copying {len(need):,} selected files to local disk ({kind}: {src})")
    if kind == "tar":
        lst = Path(CFG["local_audio"]) / f".members_{split}.txt"; lst.write_text("\n".join(need) + "\n")
        r = subprocess.run(["tar", "-xzf", str(src), "-C", CFG["local_audio"], "-T", str(lst)], capture_output=True, text=True)
        if r.returncode: raise RuntimeError(f"tar extraction failed: {r.stderr[-300:]}")
    else:
        def cp(rel):
            dst = Path(local_path(rel)); dst.parent.mkdir(parents=True, exist_ok=True); tmp = str(dst) + ".part"
            shutil.copyfile(src / rel, tmp); os.replace(tmp, dst)
        with ThreadPoolExecutor(32) as ex: list(ex.map(cp, need))
    miss = [r for r in df.relpath if not os.path.exists(local_path(r))]
    if miss: raise RuntimeError(f"[{split}] {len(miss)} files still missing after the copy, e.g. {miss[:3]}")
    log(f"[{split}] {len(need):,} files ready in {time.time() - t0:.0f} s")

def decode_check(df, split):
    """Full decode of every file once: rate, channels, exact length, finite, not digital silence. Verified ids are remembered (local disk)."""
    import soundfile as sf
    fv = paths()["meta"] / f"decode_verified_v2_{split}.csv"
    ok = set(pd.read_csv(fv, dtype={"uid": str}).uid) if fv.exists() else set()
    todo = df[~df.uid.isin(ok)]
    def chk(r):
        try:
            w, sr = sf.read(local_path(r.relpath), dtype="float32", always_2d=True)
            if sr != SR: return r.uid, f"decoded rate {sr}"
            if w.shape[1] != 1: return r.uid, f"{w.shape[1]} channels"
            if len(w) != r.n_samples: return r.uid, f"decoded {len(w)} samples, header {r.n_samples}"
            if not np.isfinite(w).all(): return r.uid, "non-finite samples"
            if np.abs(w).max() == 0: return r.uid, "digital silence"
            return r.uid, ""
        except Exception as e: return r.uid, f"decode error: {type(e).__name__}: {e}"[:200]
    t0 = time.time()
    with ThreadPoolExecutor(8) as ex: res = list(ex.map(chk, todo.itertuples()))
    bad = {u: m for u, m in res if m}; good = sorted(ok | {u for u, m in res if not m})
    tmp = str(fv) + ".tmp"; pd.DataFrame(dict(uid=good)).to_csv(tmp, index=False); os.replace(tmp, fv)
    log(f"[{split}] full decode check: {len(todo):,} files decoded in {time.time() - t0:.0f} s ({len(df) - len(todo):,} verified earlier) | failures: {len(bad)}")
    return bad

def _record_failures(split, bad):
    f = paths()["meta"] / f"decode_failures_v2_{split}.csv"
    old = pd.read_csv(f, dtype={"uid": str}) if f.exists() else pd.DataFrame(columns=["uid", "reason"])
    new = pd.concat([old, pd.DataFrame([dict(uid=u, reason=m) for u, m in bad.items()])]).drop_duplicates("uid")
    new.to_csv(f, index=False); return set(new.uid)

def _failures(split):
    f = paths()["meta"] / f"decode_failures_v2_{split}.csv"
    return set(pd.read_csv(f, dtype={"uid": str}).uid) if f.exists() else set()

def prepare_manifests():
    """train_2h / train_10h / dev_clean manifests + subset_selection.json. Idempotent; never silently overwrites a different selection."""
    P = paths(); P["meta"].mkdir(parents=True, exist_ok=True); tr = inventory("train-clean-100"); dv = inventory(CFG["val_split"])
    assert not set(tr.uid) & set(dv.uid) and not set(tr.speaker) & set(dv.speaker), "train/dev overlap"
    kind, _ = find_source("test-clean"); te = inventory("test-clean") if kind else None
    if te is not None: assert not set(tr.uid) & set(te.uid) and not set(tr.speaker) & set(te.speaker), "train/test overlap"
    sel_f = P["meta"] / "subset_selection.json"
    if sel_f.exists() and json.load(open(sel_f)).get("selection_version") != SELECTION_VERSION:      # manifests from the previous code version
        arc = P["meta"] / "archive_manifests" / time.strftime("%Y%m%d_%H%M%S"); arc.mkdir(parents=True)
        for n in ["subset_selection.json", "train_2h_manifest.csv", "train_10h_manifest.csv", "dev_clean_manifest.csv"]:
            if (P["meta"] / n).exists(): shutil.move(str(P["meta"] / n), arc / n)
        log(f"manifests from an older selection rule moved to {arc} (nothing deleted)")
    # ---- training subsets: select, extract, decode every selected file; corrupt ones are excluded and the selection redone (deterministic)
    for it in range(20):
        excl = _failures("train-clean-100"); subs = select_subsets(tr, CFG["seed"], CFG["subsets"], excl)
        big = subs[max(CFG["subsets"], key=CFG["subsets"].get)]; materialize(big, "train-clean-100"); bad = decode_check(big, "train-clean-100")
        if not bad: break
        _record_failures("train-clean-100", bad); log(f"  excluded {len(bad)} corrupt files {sorted(bad)[:5]} and re-selected")
    else: raise RuntimeError("too many corrupt files in train-clean-100")
    sel = dict(selection_version=SELECTION_VERSION, seed=CFG["seed"], source_split="train-clean-100", inventory_sha=sha_of(tr.uid),
               rule="valid utterances of train-clean-100 (official naming, 16 kHz mono FLAC, transcript present, fully decodable), sorted by id, "
                    "permuted by numpy.random.default_rng(seed).permutation; each subset = shortest prefix reaching its target hours (nested). "
                    "No duration filter.", excluded_corrupt=sorted(_failures("train-clean-100")), subsets={})
    for name, df in subs.items():
        f = P["meta"] / f"train_{name}_manifest.csv"; uid_sha = sha_of(sorted(df.uid))
        if f.exists():
            old = pd.read_csv(f, dtype=DTYPES, keep_default_na=False)
            if sha_of(sorted(old.uid)) != uid_sha: raise RuntimeError(f"{f} exists with a DIFFERENT selection; not overwritten. Move it away if you mean to replace it.")
            log(f"train_{name}: existing manifest matches the selection (reused)")
        else: df[MANIFEST_COLS + ["sel_rank"]].to_csv(f, index=False)
        sel["subsets"][name] = dict(manifest=str(f), utterances=len(df), speakers=int(df.speaker.nunique()), chapters=int(df.chapter.nunique()),
                                    hours=round(df.duration_s.sum() / 3600, 4), min_s=round(df.duration_s.min(), 2), max_s=round(df.duration_s.max(), 2),
                                    longer_than_30s=int((df.duration_s > 30).sum()), uid_sha=uid_sha)
        log(f"train_{name}: {len(df):,} utterances | {df.speaker.nunique()} speakers | {df.duration_s.sum():.1f} s = {df.duration_s.sum() / 3600:.3f} h "
            f"(target {CFG['subsets'][name] * 3600:.0f} s) | "
            f"{int((df.duration_s > 30).sum())} longer than 30 s -> {f.name}")
    names = sorted(subs, key=lambda n: CFG["subsets"][n])
    for a, b in zip(names, names[1:]): assert set(subs[a].uid) <= set(subs[b].uid), f"{a} not nested in {b}"
    # ---- validation: the whole official dev-clean (only invalid/corrupt files excluded, and reported)
    for it in range(5):
        dvv = dv[~dv.uid.isin(_failures(CFG["val_split"]))]; materialize(dvv, CFG["val_split"]); bad = decode_check(dvv, CFG["val_split"])
        if not bad: break
        _record_failures(CFG["val_split"], bad)
    fdev = P["meta"] / "dev_clean_manifest.csv"; dvv[MANIFEST_COLS].to_csv(fdev, index=False)
    fdrop = P["meta"] / f"inventory_v2_{CFG['val_split']}_dropped.csv"
    n_drop = len(pd.read_csv(fdrop)) if fdrop.exists() and fdrop.stat().st_size > 30 else 0
    n_off = OFFICIAL_UTTS.get(CFG["val_split"], len(dv) + n_drop)
    sel["validation"] = dict(split=CFG["val_split"], manifest=str(fdev), utterances=len(dvv), official_utterances=n_off,
                             coverage=f"{len(dvv)}/{n_off}" + ("" if len(dvv) == n_off else " (FILTERED dev-clean: invalid/corrupt files excluded, see dropped lists)"),
                             hours=round(dvv.duration_s.sum() / 3600, 4), longer_than_30s=int((dvv.duration_s > 30).sum()), speakers=int(dvv.speaker.nunique()))
    sel["test"] = "test-clean is reserved for downstream (Stage 2) WER; Stage 1 reads only its headers for the overlap check" if te is not None else \
                  "test-clean not present on Drive (not needed for Stage 1)"
    atomic_json(sel, P["meta"] / "subset_selection.json")
    log(f"validation (official {CFG['val_split']}): {len(dvv):,}/{n_off} utterances, {dvv.duration_s.sum() / 3600:.2f} h, "
        f"{sel['validation']['longer_than_30s']} longer than 30 s (kept, encoded whole) | coverage {sel['validation']['coverage']}")
    log(f"no train/dev{'/test' if te is not None else ''} utterance or speaker overlap | selection record: {P['meta'] / 'subset_selection.json'}")
    return sel

def load_manifest(name):
    P = paths(); f = P["meta"] / ("dev_clean_manifest.csv" if name == "dev" else f"train_{name}_manifest.csv")
    if not f.exists(): raise FileNotFoundError(f"{f} missing: run the manifest cell first")
    sel = json.load(open(P["meta"] / "subset_selection.json"))
    if sel.get("selection_version") != SELECTION_VERSION: raise RuntimeError("manifests come from an older selection rule: re-run the manifest cell")
    df = pd.read_csv(f, dtype=DTYPES, keep_default_na=False)
    assert df.uid.is_unique and (df.text.str.len() > 0).all() and (df.n_samples > 0).all(), f"{f}: duplicate ids / empty transcripts / bad lengths"
    return df

def load_audio(relpath, n_expected):
    """16 kHz mono float32 in [-1, 1]; raises on wrong rate/length or non-finite samples."""
    import soundfile as sf
    wav, sr = sf.read(local_path(relpath), dtype="float32", always_2d=True)
    if sr != SR: raise ValueError(f"{relpath}: {sr} Hz (expected {SR})")
    if wav.shape[1] != 1: raise ValueError(f"{relpath}: {wav.shape[1]} channels (expected mono)")
    wav = wav[:, 0]
    if len(wav) != n_expected: raise ValueError(f"{relpath}: {len(wav)} samples, manifest says {n_expected}")
    if not np.isfinite(wav).all(): raise ValueError(f"{relpath}: non-finite samples")
    return np.ascontiguousarray(wav)

# ####################################################################################################
# ---- asr1_wavlm ----
# ####################################################################################################
# ===========================================================================================
# WavLM ASR Stage 1 | WavLM-Large teacher: loading, preprocessing, layer features, verification, timing measurement,
#                     resumable integrity-checked cache
#   WavLM-Large (microsoft/wavlm-large, 315M parameters, 24 Transformer layers, 1024-D @ 50 Hz, input = raw 16 kHz waveform).
#   Same representation recipe as the Whisper run: two successive 50 Hz frames concatenated -> 2048-D @ 25 Hz,
#   target frame j = [h_{2j} ; h_{2j+1}] (an odd last 50 Hz frame is dropped).
#   No 30 s window and no chunking: every utterance is encoded alone, at its exact length (no padding at all).
# ===========================================================================================
import os, json, math, time, shutil
from pathlib import Path
import numpy as np, pandas as pd

HOP_T = 640                                   # samples per 25 Hz target frame
CACHE_FORMAT = 3
WAVLM_DIM = 1024
WAVLM_CONV = [(10, 5), (3, 2), (3, 2), (3, 2), (3, 2), (2, 2), (2, 2)]     # (kernel, stride) of WavLM's convolutional feature encoder
def wavlm_frames50(n):
    """Exact frame count of the conv feature encoder (no padding): L -> (L - k) // s + 1 per layer (= (n - 400) // 320 + 1)."""
    L = int(n)
    for k, s in WAVLM_CONV: L = (L - k) // s + 1
    return max(L, 0)
def teacher_frames(n): return wavlm_frames50(n) // 2                       # 25 Hz target frames (pairs of 50 Hz frames)
def wavlm_centre50(m):
    """Analytic centre (in samples) of 50 Hz frame m: the middle of its receptive field (= 320 m + 199.5)."""
    x = float(m)
    for k, s in reversed(WAVLM_CONV): x = x * s + (k - 1) / 2
    return x

def wavlm_prep(wav):
    """Per-utterance zero-mean / unit-variance waveform (WavLM-Large was pre-trained on normalised input, cfg.normalize = True;
    same formula as transformers' Wav2Vec2FeatureExtractor.zero_mean_unit_var_norm)."""
    w = np.asarray(wav, np.float32)
    if not CFG["teacher_normalize"]: return np.ascontiguousarray(w)
    return np.ascontiguousarray(((w - w.mean()) / np.sqrt(w.var() + 1e-7)).astype(np.float32))

def _verification_file(): return paths()["logs"] / "wavlm_verification.json"

class WavLMTeacher:
    """Frozen WavLM-Large. One utterance per forward pass (no padding, so no attention mask is involved)."""
    def __init__(self, device="cuda", dtype=None):
        pip_if_missing("transformers")
        import torch, transformers
        from transformers import WavLMModel
        name = CFG["teacher_model"]; t0 = time.time()
        if dtype is None:          # the precision chosen by the verification of this runtime (fp32 only if fp16 was found inaccurate)
            vf = _verification_file()
            dtype = json.load(open(vf)).get("dtype_used", "float16") if vf.exists() else "float16"
        self.dtype_name = dtype; tdt = torch.float16 if dtype == "float16" else torch.float32
        major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
        kw = dict(low_cpu_mem_usage=True); kw["dtype" if (major, minor) >= (4, 56) else "torch_dtype"] = tdt
        model = WavLMModel.from_pretrained(name, **kw)
        cfg = model.config; self.commit = getattr(cfg, "_commit_hash", None) or "unknown"
        self.enc = model.to(device=device, dtype=tdt).eval()
        for p in self.enc.parameters(): p.requires_grad_(False)
        self.device = device; self.d_model = int(cfg.hidden_size); self.n_layers = int(cfg.num_hidden_layers); self.layer = int(CFG["teacher_layer"])
        self.hf_do_normalize = None
        try:
            from transformers import Wav2Vec2FeatureExtractor
            self.hf_do_normalize = bool(Wav2Vec2FeatureExtractor.from_pretrained(name).do_normalize)
        except Exception as e: log(f"  (feature-extractor config not read: {type(e).__name__}; the waveform normalisation is done by this code anyway)")
        conv = [(int(k), int(s)) for k, s in zip(cfg.conv_kernel, cfg.conv_stride)]
        self.info = dict(model=name, commit=self.commit, transformers=transformers.__version__, torch=torch.__version__,
                         sampling_rate=SR, d_model=self.d_model, encoder_layers=self.n_layers, conv=conv,
                         params_M=round(sum(p.numel() for p in self.enc.parameters()) / 1e6, 1), dtype=dtype,
                         layer=f"hidden_states[{self.layer}] of {self.n_layers}" + (" (= last_hidden_state, after the final LayerNorm)" if self.layer == self.n_layers else ""),
                         pairing="concatenate 50 Hz frames (2j, 2j+1) -> 25 Hz, 2 x d_model", frames_rule="T = ((n - 400) // 320 + 1) // 2",
                         normalize="per-utterance zero mean / unit variance" if CFG["teacher_normalize"] else "none", hf_feature_extractor_do_normalize=self.hf_do_normalize,
                         chunking="none (whole utterance, no padding)", load_s=round(time.time() - t0, 1))
        problems = []
        if conv != WAVLM_CONV: problems.append(f"conv feature encoder {conv}, expected {WAVLM_CONV}")
        if self.d_model != WAVLM_DIM or self.n_layers != 24: problems.append(f"hidden size {self.d_model}, layers {self.n_layers} (expected 1024 / 24: WavLM-Large)")
        if not 1 <= self.layer <= self.n_layers: problems.append(f"teacher_layer {self.layer} outside 1..{self.n_layers}")
        if any(p.requires_grad for p in self.enc.parameters()) or self.enc.training: problems.append("teacher not frozen / not in eval mode")
        if problems: raise RuntimeError("WavLM check failed: " + "; ".join(problems))
        log(f"WavLM loaded: {name} (commit {self.commit[:10]}) | {self.info['params_M']}M params, {self.n_layers} layers, {dtype}, frozen, eval | "
            f"layer {self.layer} | d_model {self.d_model} | {self.info['load_s']} s | " + ", ".join(f"{k} {v:.2f}" for k, v in gpu_mem().items()))

    def set_dtype(self, dtype):
        import torch
        self.enc.to(torch.float16 if dtype == "float16" else torch.float32); self.dtype_name = dtype; self.info["dtype"] = dtype

    def hidden(self, x):
        """x: normalised waveform (numpy [n]) -> the selected layer [T50, 1024] on the GPU (fp32 retry if fp16 overflows)."""
        import torch
        dt = next(self.enc.parameters()).dtype
        def run(dtype):
            inp = torch.as_tensor(x)[None].to(self.device, dtype=dtype)
            with torch.inference_mode():
                if self.layer == self.n_layers: return self.enc(input_values=inp).last_hidden_state[0]
                return self.enc(input_values=inp, output_hidden_states=True).hidden_states[self.layer][0]
        h = run(dt)
        if not torch.isfinite(h).all():                       # rare fp16 overflow: recompute this utterance in fp32
            self.enc.float(); h = run(torch.float32); self.enc.to(dt)
            if not torch.isfinite(h).all(): raise FloatingPointError("WavLM output non-finite even in fp32")
        return h

    def encode_wav(self, wav):
        """16 kHz mono float32 waveform -> float16 array [teacher_frames(n), 2048] (pairs of 50 Hz frames)."""
        import torch
        n = len(wav); h = self.hidden(wavlm_prep(wav)); T50 = wavlm_frames50(n)
        if h.shape[0] != T50 or h.shape[1] != self.d_model: raise RuntimeError(f"WavLM output {tuple(h.shape)} for {n} samples, expected ({T50}, {self.d_model})")
        T = T50 // 2
        return h[:2 * T].reshape(T, 2 * self.d_model).to(torch.float16).cpu().numpy()

class _WavDS:       # CPU workers only read audio (they never touch the GPU model)
    def __init__(self, rows): self.r = rows
    def __len__(self): return len(self.r)
    def __getitem__(self, i):
        r = self.r[i]; n = int(r["n_samples"]); return r["uid"], n, load_audio(r["relpath"], n)
def _collate(b): return b

# ------------------------------------------------------------------ verification on real audio
def _cos(a, b):
    a = np.asarray(a, np.float32).ravel(); b = np.asarray(b, np.float32).ravel(); return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

def pick_test_utts(train_df, dev_df):
    """short, medium, near-30 s (<= 30 s) and > 30 s (dev-clean, or two real training utterances joined if none) for the checks."""
    pool = pd.concat([train_df, dev_df], ignore_index=True)
    short = pool.loc[pool.duration_s.idxmin()]; med = pool.iloc[(pool.duration_s - pool.duration_s.median()).abs().argsort().iloc[0]]
    near = pool[pool.duration_s <= 30].iloc[(pool[pool.duration_s <= 30].duration_s - 29.5).abs().argsort().iloc[0]]
    longs = pool[pool.duration_s > 30]
    return dict(short=short, medium=med, near30=near, long=longs.iloc[0] if len(longs) else None)

def verify_wavlm(teacher, train_df, dev_df):
    """Preprocessing, dimensions, frame-count rule, layer identity, frame pairing, long-utterance handling, repeatability and precision.
    If fp16 is not accurate enough (cosine to fp32 < 0.999), the teacher switches to fp32 for the cache (recorded)."""
    import torch
    torch.cuda.reset_peak_memory_stats(); R = dict(info=dict(teacher.info)); U = pick_test_utts(train_df, dev_df)
    W = {k: (r.uid, load_audio(r.relpath, int(r.n_samples))) for k, r in U.items() if r is not None}
    if U["long"] is None:
        a, b = W["near30"][1], W["medium"][1]; W["long"] = ("joined:" + W["near30"][0] + "+" + W["medium"][0], np.concatenate([a, b]))
        log("  no utterance > 30 s in the manifests: a > 30 s test signal is built by joining two real utterances")
    for k, (u, w) in W.items():
        assert w.ndim == 1 and w.dtype == np.float32 and np.isfinite(w).all() and np.abs(w).max() <= 1.0
        log(f"  {k:7s} {u}: {len(w)} samples = {len(w) / SR:.2f} s, mono float32 @ {SR} Hz")
    # frame-count rule: the formula used for the cache == transformers' own output-length rule == closed form, for many lengths
    ns = list(range(400, 60000, 37)) + [480000, 523456]
    assert [wavlm_frames50(n) for n in ns] == [(n - 400) // 320 + 1 for n in ns], "WavLM frame-count rule mismatch"
    try:
        hf = [int(teacher.enc._get_feat_extract_output_lengths(torch.tensor(n)).item()) for n in ns]; hf_ok = hf == [wavlm_frames50(n) for n in ns]
    except Exception as e: hf_ok = f"not checked ({type(e).__name__})"
    assert hf_ok is not False, "transformers' WavLM output-length rule differs from (n - 400) // 320 + 1"
    log(f"  frame rule: 50 Hz frames = (n - 400) // 320 + 1 (transformers' own rule agrees: {hf_ok}); 25 Hz targets = that // 2; "
        f"checked against the real model output below")
    # preprocessing
    w0 = W["medium"][1]; x0 = wavlm_prep(w0)
    R["prep"] = dict(mean=float(x0.mean()), std=float(x0.std()), hf_do_normalize=teacher.hf_do_normalize)
    if CFG["teacher_normalize"]: assert abs(R["prep"]["mean"]) < 1e-3 and abs(R["prep"]["std"] - 1) < 1e-3, R["prep"]
    log(f"  preprocessing: raw 16 kHz waveform, {teacher.info['normalize']} (mean {R['prep']['mean']:+.1e}, std {R['prep']['std']:.4f}); "
        f"transformers' feature-extractor config for this model says do_normalize = {teacher.hf_do_normalize}")
    # layer identity + raw output geometry
    inp = torch.as_tensor(x0)[None].to(teacher.device, dtype=next(teacher.enc.parameters()).dtype)
    with torch.inference_mode(): o = teacher.enc(input_values=inp, output_hidden_states=True)
    hs = o.hidden_states; assert len(hs) == teacher.n_layers + 1, f"{len(hs)} hidden states for {teacher.n_layers} layers"
    lhs = o.last_hidden_state; d_last = float((hs[-1] - lhs).abs().max()); scale = max(1.0, float(lhs.abs().max()))
    if d_last <= 1e-3 * scale:
        how = f"hidden_states[{teacher.n_layers}] == last_hidden_state (|diff| {d_last:.1e}): stored after the final LayerNorm"
    else:                     # some transformers versions store the last layer's output BEFORE the encoder's final LayerNorm
        ln = getattr(getattr(teacher.enc, "encoder", None), "layer_norm", None)
        assert ln is not None, f"hidden_states[{teacher.n_layers}] differs from last_hidden_state by {d_last} and the encoder has no final LayerNorm"
        with torch.inference_mode(): d_ln = float((ln(hs[-1]) - lhs).abs().max())
        assert d_ln <= 1e-2 * scale, f"last_hidden_state is neither hidden_states[-1] (|diff| {d_last}) nor its LayerNorm (|diff| {d_ln})"
        how = (f"last_hidden_state == LayerNorm(hidden_states[{teacher.n_layers}]) (|diff| {d_ln:.1e}); this transformers version stores hidden_states[-1] "
               f"before the final LayerNorm")
    sel = lhs if teacher.layer == teacher.n_layers else hs[teacher.layer]          # exactly what WavLMTeacher.hidden() returns
    T50 = hs[0].shape[1]; assert T50 == wavlm_frames50(len(w0)) and sel.shape[-1] == teacher.d_model
    R["layer_check"] = dict(hidden_states=len(hs), used=teacher.info["layer"], detail=how)
    log(f"  layer: {how}; using {teacher.info['layer']} | {len(w0)} samples -> {T50} frames @ 50 Hz, {hs[0].shape[-1]}-D")
    # pairing / concatenation order (same forward path as the cache)
    out = teacher.encode_wav(w0); T = teacher_frames(len(w0)); hb = sel[0].float().cpu().numpy(); j = T // 2
    assert out.shape == (T, 2 * teacher.d_model) and np.isfinite(out).all()
    c_a, c_b = _cos(out[j, :teacher.d_model], hb[2 * j]), _cos(out[j, teacher.d_model:], hb[2 * j + 1])
    assert c_a > 0.9999 and c_b > 0.9999, (c_a, c_b)
    R["pairing"] = dict(frames50=T50, targets=T, dropped_odd_frame=bool(T50 % 2), cos=[c_a, c_b])
    log(f"  pairing: target[j] = [h(2j) ; h(2j+1)] verified (cos {c_a:.6f} / {c_b:.6f}); {T50} frames @ 50 Hz -> {out.shape} @ 25 Hz"
        + (" (odd last 50 Hz frame dropped)" if T50 % 2 else ""))
    del o, hs, hb, lhs, sel
    # every test length incl. > 30 s: shape rule, finite, no chunking
    for k, (u, w) in W.items():
        t = teacher.encode_wav(w); assert t.shape == (teacher_frames(len(w)), 2 * teacher.d_model) and np.isfinite(t).all(), (k, t.shape)
        R[f"frames_{k}"] = [len(w), len(t)]
    log(f"  all test lengths (incl. {len(W['long'][1]) / SR:.1f} s, encoded whole, no 30 s window): " + ", ".join(f"{k} {v[0]} -> {v[1]}" for k, v in R.items() if k.startswith("frames_")))
    # repeatability and precision
    again = teacher.encode_wav(w0); R["repeat_maxdiff"] = float(np.abs(again.astype(np.float32) - out).max())
    if teacher.dtype_name == "float16":
        teacher.set_dtype("float32"); t32 = teacher.encode_wav(w0); teacher.set_dtype("float16")
        R["fp16_vs_fp32_cos"] = _cos(out, t32); R["fp16_vs_fp32_rel_l1"] = float(np.abs(out.astype(np.float32) - t32).mean() / (np.abs(t32).mean() + 1e-12))
        if R["fp16_vs_fp32_cos"] < 0.999:
            log(f"  fp16 vs fp32 cosine only {R['fp16_vs_fp32_cos']:.5f}: the cache will be computed in fp32 (slower, exact)"); teacher.set_dtype("float32")
    else: R["fp16_vs_fp32_cos"] = None; R["fp16_vs_fp32_rel_l1"] = None
    R["dtype_used"] = teacher.dtype_name
    st = out.astype(np.float32); R.update(target_mean=float(st.mean()), target_std=float(st.std()), target_absmax=float(np.abs(st).max()), **gpu_mem())
    log(f"  repeat max |diff| {R['repeat_maxdiff']:.4f} | fp16 vs fp32 cosine {R['fp16_vs_fp32_cos']} (mean rel. L1 {R['fp16_vs_fp32_rel_l1']}) | cache dtype: {R['dtype_used']} | all finite")
    log(f"  target stats: mean {R['target_mean']:.3f}, std {R['target_std']:.3f}, |max| {R['target_absmax']:.1f} | GPU peak {R['gpu_peak_gib']:.2f} GiB")
    atomic_json(R, _verification_file()); log(f"  saved {_verification_file()}")
    return R

def measure_teacher_timing(teacher):
    """Measures, with the imported WavLM code, where each target frame sits in time: every input sample of a probe signal is perturbed
    (one at a time) and the 50 Hz frames of the REAL conv feature encoder that change are recorded (fp32, TF32 off). The centre of each frame's
    receptive field is fitted linearly; the pairing (2j, 2j+1) is then applied. Returns a linear map j -> centre (samples)."""
    import torch, copy
    fe = copy.deepcopy(teacher.enc.feature_extractor).float().eval(); dev = teacher.device
    tf = None
    try: tf = (torch.backends.cudnn.allow_tf32, torch.backends.cuda.matmul.allow_tf32); torch.backends.cudnn.allow_tf32 = False; torch.backends.cuda.matmul.allow_tf32 = False
    except Exception: pass
    try:
        L, K, B = 4000, 3200, 128; g = torch.Generator().manual_seed(0); x = (torch.randn(1, L, generator=g) * 0.1).to(dev)
        infl = None
        with torch.inference_mode():
            for s in range(0, K, B):
                ks = list(range(s, min(K, s + B))); xb = x.repeat(len(ks) + 1, 1)
                for i, k in enumerate(ks): xb[i + 1, k] += 0.5
                y = fe(xb)                                                                  # [b, C, M]; item 0 = unperturbed (same batch -> same kernels)
                if infl is None: infl = [[] for _ in range(y.shape[-1])]
                ch = (y[1:] - y[:1]).abs().amax(1) > 1e-5                                  # [b, M]
                for i, k in enumerate(ks):
                    for m in torch.nonzero(ch[i]).flatten().tolist(): infl[m].append(k)
    finally:
        try:
            if tf is not None: torch.backends.cudnn.allow_tf32, torch.backends.cuda.matmul.allow_tf32 = tf
        except Exception: pass
        del fe; torch.cuda.empty_cache()
    ms = [m for m in range(len(infl)) if infl[m] and min(infl[m]) > 0 and max(infl[m]) < K - 1]
    if len(ms) < 4:          # e.g. a GroupNorm feature encoder (every frame depends on every sample): use the layer-geometry map
        a_w, b_w = 320.0, wavlm_centre50(0)
        log(f"  teacher timing: the perturbation probe found only {len(ms)} local frames (feature encoder normalises over time?) -> "
            f"using the layer-geometry map 50 Hz frame m centre = {a_w:.0f} m + {b_w:.1f} samples")
        return dict(frame50_slope=a_w, frame50_intercept=b_w, fit_residual=None, measured=False,
                    centre_samples=dict(slope=2 * a_w, intercept=b_w + 0.5 * a_w))
    cm = np.array([(min(infl[m]) + max(infl[m])) / 2 for m in ms]); width = sorted({max(infl[m]) - min(infl[m]) + 1 for m in ms})
    a_w, b_w = np.polyfit(ms, cm, 1); res_w = float(np.abs(np.polyval([a_w, b_w], ms) - cm).max())
    slope = 2 * a_w; icpt = b_w + 0.5 * a_w                                                 # target j = 50 Hz frames (2j, 2j+1): centre index 2j + 0.5
    R = dict(frame50_slope=float(a_w), frame50_intercept=float(b_w), fit_residual=res_w, receptive_field_samples=width,
             influence_example={int(m): [min(infl[m]), max(infl[m])] for m in ms[:3]}, centre_samples=dict(slope=float(slope), intercept=float(icpt)))
    log(f"  teacher timing (measured): 50 Hz frame m <- samples centred at {a_w:.3f} m + {b_w:.2f} (receptive field {width} samples, fit residual {res_w:.3f}) "
        f"| target j centre = {slope:.1f} j + {icpt:.1f} samples = {slope / 16:.1f} j + {icpt / 16:.2f} ms")
    return R

# ------------------------------------------------------------------ cache
FEATURE_KEYS = ["model", "commit", "layer", "pairing", "dim", "frames_rule", "normalize", "sampling_rate", "cache_format", "dtype_stored"]

def cache_dir(split, teacher_commit):
    return paths()["features"] / f"{CFG['teacher_model'].split('/')[-1]}_{teacher_commit[:8]}_L{CFG['teacher_layer']}_cat2_v{CACHE_FORMAT}" / split

def _committed(d, full_check=False):
    """{uid -> (shard, offset, length)} for committed shards. A shard counts only if its .npy, .json and index .csv all exist and agree;
    inconsistent shards are moved to <cache>/corrupt/ (never deleted) and their utterances are recomputed."""
    parts = []
    for f in sorted(d.glob("index_*.csv")):
        k = f.stem.split("_")[1]; s, j = d / f"shard_{k}.npy", d / f"shard_{k}.json"; why = ""
        try:
            ix = pd.read_csv(f, dtype={"uid": str, "speaker": str}, keep_default_na=False); meta = json.load(open(j))
            if s.stat().st_size != meta["nbytes"]: why = f"size {s.stat().st_size} != {meta['nbytes']}"
            else:
                m = np.load(s, mmap_mode="r")
                if m.shape != (int(ix.length.sum()), meta["dim"]) or m.dtype != np.float16: why = f"shape {m.shape} / {m.dtype}"
                elif full_check and file_sha1(s) != meta["sha1"]: why = "sha1 mismatch"
        except Exception as e: why = f"{type(e).__name__}: {e}"
        if why:
            bad = d / "corrupt" / time.strftime("%Y%m%d_%H%M%S"); bad.mkdir(parents=True, exist_ok=True)
            for x in (f, s, j):
                if x.exists(): shutil.move(str(x), bad / x.name)
            log(f"  WARNING shard {k} failed the integrity check ({why}); moved to {bad}; its utterances will be recomputed"); continue
        parts.append(ix)
    idx = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["uid", "speaker", "n_samples", "shard", "offset", "length"])
    if idx.uid.duplicated().any(): raise RuntimeError(f"cache {d}: utterances in more than one shard: {idx.uid[idx.uid.duplicated()].tolist()[:5]}")
    return idx

def check_meta(d, meta):
    if (d / "meta.json").exists():
        old = json.load(open(d / "meta.json"))
        diff = {k: (old.get(k), meta.get(k)) for k in FEATURE_KEYS if old.get(k) != meta.get(k)}
        if diff: raise RuntimeError(f"existing cache in {d} was made differently: {diff}. Move that folder away (nothing is reused silently).")

def build_cache(teacher, df, split, encode_rows=None):
    """Resumable: committed utterances are skipped; an interrupted shard is redone (at most shard_utts utterances).
    Commit order per shard: .npy (atomic) -> .json (sha1, size, shape) -> index .csv (the commit marker)."""
    d = cache_dir(split, teacher.commit); d.mkdir(parents=True, exist_ok=True)
    meta = dict(teacher.info, split=split, dim=2 * teacher.d_model, rate_hz=25, sample_rate=SR, cache_format=CACHE_FORMAT, dtype_stored="float16",
                preprocessing="16 kHz mono float32 -> per-utterance zero-mean/unit-variance -> WavLM-Large (whole utterance, no padding) -> "
                              f"hidden_states[{teacher.layer}] -> frames (2j, 2j+1) concatenated (odd last frame dropped)")
    meta["sampling_rate"] = SR; check_meta(d, meta); atomic_json(meta, d / "meta.json")
    for f in list(d.glob("shard_*.npy")) + list(d.glob("shard_*.json")):          # orphans from an interrupted commit
        if not (d / f"index_{f.stem.split('_')[1]}.csv").exists(): f.unlink()
    for f in list(d.glob("*.tmp*")): f.unlink()
    done = _committed(d); todo = df[~df.uid.isin(set(done.uid))].sort_values("uid")
    need_gib = sum(teacher_frames(n) for n in todo.n_samples) * 2 * teacher.d_model * 2 / 2**30
    log(f"[{split}] teacher cache {d}\n  committed {len(done):,} | to compute {len(todo):,} utterances (~{need_gib:.2f} GiB)")
    if len(todo): require_space(d, need_gib, "the teacher cache on local disk", 3.0)
    used = [int(f.stem.split("_")[1]) for f in list(d.glob("index_*.csv")) + list(d.glob("corrupt/*/*.npy")) + list(d.glob("corrupt/*/*.csv"))]
    k = max(used, default=-1) + 1                                                  # never reuse a shard number
    t0 = time.time(); n_done = 0; s_done = 0.0; rows = todo.to_dict("records")
    for s in range(0, len(rows), CFG["shard_utts"]):
        chunk = rows[s:s + CFG["shard_utts"]]; arrs, recs, off = [], [], 0
        if encode_rows is not None: res = encode_rows(chunk)              # test hook
        else:
            import torch
            dl = torch.utils.data.DataLoader(_WavDS(chunk), batch_size=4, num_workers=CFG["teacher_workers"], collate_fn=_collate,
                                             prefetch_factor=2 if CFG["teacher_workers"] else None)
            res = [(u, n, teacher.encode_wav(w)) for b in dl for u, n, w in b]
        by = {u: (n, a) for u, n, a in res}
        for r in chunk:
            n, a = by[r["uid"]]
            if a.shape != (teacher_frames(n), 2 * teacher.d_model) or not np.isfinite(a).all(): raise RuntimeError(f"{r['uid']}: bad teacher output {a.shape}")
            arrs.append(a); recs.append(dict(uid=r["uid"], speaker=r["speaker"], n_samples=n, shard=k, offset=off, length=len(a))); off += len(a)
        A = np.concatenate(arrs).astype(np.float16); tmp = d / f"shard_{k:05d}.tmp.npy"; np.save(tmp, A); os.replace(tmp, d / f"shard_{k:05d}.npy")
        atomic_json(dict(sha1=file_sha1(d / f"shard_{k:05d}.npy"), nbytes=(d / f"shard_{k:05d}.npy").stat().st_size, rows=int(A.shape[0]), dim=int(A.shape[1]),
                         utterances=len(chunk)), d / f"shard_{k:05d}.json")
        tmpc = d / f"index_{k:05d}.tmp.csv"; pd.DataFrame(recs).to_csv(tmpc, index=False); os.replace(tmpc, d / f"index_{k:05d}.csv")
        n_done += len(chunk); s_done += sum(r["n_samples"] for r in chunk) / SR; el = time.time() - t0
        log(f"  shard {k:05d} committed | {n_done:,}/{len(rows):,} | {n_done / el:.1f} utt/s ({s_done / el:.0f} s of audio per s) | "
            f"ETA {(len(rows) - n_done) / max(n_done / el, 1e-9) / 60:.0f} min" + (f" | GPU peak {gpu_mem().get('gpu_peak_gib', 0):.1f} GiB" if encode_rows is None else ""))
        k += 1
    allidx = _committed(d); miss = set(df.uid) - set(allidx.uid)
    if miss: raise RuntimeError(f"{len(miss)} utterances still missing from the cache")
    allidx.to_csv(d / "index.csv", index=False)
    log(f"[{split}] cache complete: {len(allidx):,} utterances, {allidx.length.sum():,} frames x {2 * teacher.d_model} "
        f"({allidx.length.sum() * 2 * teacher.d_model * 2 / 2**30:.2f} GiB)")
    return d

def find_cache(split):
    """The (single) compatible teacher cache folder for a split."""
    c = sorted({Path(p).parent for p in paths()["features"].glob(f"*/{split}/meta.json")})
    ok = []
    for d in c:
        m = json.load(open(d / "meta.json"))
        if (m.get("model") == CFG["teacher_model"] and m.get("dim") == 2 * WAVLM_DIM and m.get("cache_format") == CACHE_FORMAT
                and m.get("layer", "").startswith(f"hidden_states[{CFG['teacher_layer']}] ")): ok.append(d)
        else: log(f"  (ignoring incompatible cache {d}: model {m.get('model')}, format {m.get('cache_format')}, dim {m.get('dim')}, layer {m.get('layer')})")
    if not ok: raise FileNotFoundError(f"no compatible WavLM {2 * WAVLM_DIM}-D cache for {split} under {paths()['features']}: run the teacher-cache step")
    if len(ok) > 1: raise RuntimeError(f"several compatible caches for {split}: {ok}. Keep one.")
    return ok[0]

class TeacherCache:
    """Read-only access by utterance id. Shards are memory-mapped (never loaded whole into RAM); copied to local disk and sha1-checked first."""
    def __init__(self, d, uids, local=True):
        d = Path(d); idx = _committed(d).set_index("uid"); uids = list(uids); miss = set(uids) - set(idx.index)
        if miss: raise RuntimeError(f"{len(miss)} utterances are not in the teacher cache {d} (e.g. {sorted(miss)[:3]}): run the cache step")
        self.meta = json.load(open(d / "meta.json")); self.idx = idx.loc[uids]; self.dim = int(self.meta["dim"]); self.src = d
        bad = self.idx[self.idx.length != [teacher_frames(n) for n in self.idx.n_samples]]
        if len(bad): raise RuntimeError(f"{len(bad)} cache entries have lengths that do not match ((n - 400) // 320 + 1) // 2, e.g. {bad.index[:3].tolist()}")
        shards = sorted(set(int(s) for s in self.idx.shard))
        if local and not str(d).startswith("/content/drive"):      # cache already on local disk: verify every shard's sha1 in place (once per shard)
            for s in shards:
                a, sha = d / f"shard_{s:05d}.npy", json.load(open(d / f"shard_{s:05d}.json"))["sha1"]; okf = d / f"shard_{s:05d}.ok"
                if okf.exists() and okf.read_text() == sha and okf.stat().st_mtime >= a.stat().st_mtime: continue
                if file_sha1(a) != sha: raise RuntimeError(f"shard {a} fails its sha1 check: move it to corrupt/ (or re-run after deleting it)")
                okf.write_text(sha)
            self.d = d
        elif local:
            loc = Path(CFG["local_features"]) / d.parent.name / d.name; loc.mkdir(parents=True, exist_ok=True)
            need = sum((d / f"shard_{s:05d}.npy").stat().st_size for s in shards if not (loc / f"shard_{s:05d}.npy").exists()) / 2**30
            if need: require_space(loc, need, "the local teacher-cache copy", 2.0)
            for s in shards:
                a, b, sha = d / f"shard_{s:05d}.npy", loc / f"shard_{s:05d}.npy", json.load(open(d / f"shard_{s:05d}.json"))["sha1"]
                ok_flag = loc / f"shard_{s:05d}.ok"
                if b.exists() and ok_flag.exists() and ok_flag.read_text() == sha: continue
                for attempt in range(2):
                    shutil.copyfile(a, str(b) + ".part"); os.replace(str(b) + ".part", b)
                    if file_sha1(b) == sha: ok_flag.write_text(sha); break
                else: raise RuntimeError(f"shard {a} fails its sha1 check after copying twice: move it to corrupt/ and rebuild")
            self.d = loc
        else: self.d = d
        self._m = {}
    def get(self, uid):
        r = self.idx.loc[uid]; s = int(r.shard)
        if s not in self._m: self._m[s] = np.load(self.d / f"shard_{s:05d}.npy", mmap_mode="r")
        return np.asarray(self._m[s][int(r.offset): int(r.offset) + int(r.length)])
    def baselines(self, ranges, cache_file=None, median_frames=20000, seed=0):
        """Constant-prediction baselines over the ALIGNED teacher frames only: ranges = {uid: (first, count)}.
        mean: per-dimension mean (exact, streamed); median: per-dimension median of a seeded random sample of frames
        (the median is the L1-optimal constant; the mean is what the projection bias is initialised to)."""
        if cache_file and Path(cache_file).exists():
            z = np.load(cache_file); return z["mean"], z["median"]
        s = np.zeros(self.dim, np.float64); n = 0; rng = np.random.default_rng(seed); tot = sum(c for _, c in ranges.values())
        p = min(1.0, median_frames / max(tot, 1)); sample = []
        for u, (a, c) in ranges.items():
            x = self.get(u)[a:a + c].astype(np.float32); s += x.astype(np.float64).sum(0); n += len(x)
            m = rng.random(len(x)) < p
            if m.any(): sample.append(x[m])
        mean = (s / n).astype(np.float32); med = np.median(np.concatenate(sample), 0).astype(np.float32)
        if cache_file: np.savez(cache_file, mean=mean, median=med)
        return mean, med

def verify_cache_vs_fresh(teacher, cache, df, n=8, seed=0):
    """Re-extracts n utterances (always including the longest) and compares them with the cache."""
    rng = np.random.default_rng(seed); rows = df.sort_values("duration_s").iloc[[-1] + list(rng.choice(len(df) - 1, min(n, len(df)) - 1, replace=False))]
    out = []
    for r in rows.itertuples():
        f = teacher.encode_wav(load_audio(r.relpath, int(r.n_samples))); c = cache.get(r.uid)
        assert f.shape == c.shape, (r.uid, f.shape, c.shape)
        out.append(dict(uid=r.uid, seconds=round(r.duration_s, 2), frames=len(c), cos=_cos(f, c), maxdiff=float(np.abs(f.astype(np.float32) - c).max())))
    worst = min(o["cos"] for o in out)
    log(f"  cache vs fresh extraction on {len(out)} utterances (longest {rows.duration_s.max():.1f} s): min cosine {worst:.6f}, "
        f"max |diff| {max(o['maxdiff'] for o in out):.4f}")
    return dict(items=out, min_cos=worst, passed=worst > 0.9995)

# ####################################################################################################
# ---- asr1_student ----
# ####################################################################################################
# ===========================================================================================
# MT2KD ASR Stage 1 | Zipformer-M student (your existing architecture, unchanged), source inspection, Swoosh port + verification,
#                    fbank frontend, projection, MEASURED teacher/student frame timing, alignment, masked L1, data loading
# ===========================================================================================
import os, re, sys, json, math, shutil, subprocess, importlib, hashlib
from pathlib import Path
import numpy as np

D_S = 512                                    # Zipformer-M final output (max of encoder_dim)
ZIP_ARGS = dict(output_downsampling_factor=2, downsampling_factor=(1, 2, 4, 8, 4, 2), num_encoder_layers=(2, 2, 3, 4, 3, 2),
                encoder_dim=(192, 256, 384, 512, 384, 256), encoder_unmasked_dim=(192, 192, 256, 256, 256, 192), query_head_dim=(32,) * 6,
                pos_head_dim=(4,) * 6, value_head_dim=(12,) * 6, num_heads=(4, 4, 4, 8, 4, 4), feedforward_dim=(512, 768, 1024, 1536, 1024, 768),
                cnn_module_kernel=(31, 31, 15, 15, 15, 31), pos_dim=48, causal=False)        # identical to your v1 code

# ------------------------------------------------------------------ frame counts (pure Python, unit-tested)
def fbank_frames(n): return (n + 80) // 160                         # kaldi fbank, 10 ms shift, snip_edges=False
def embed_frames(n): return (fbank_frames(n) - 7) // 2              # Conv2dSubsampling: T' = (T - 7) // 2  (~50 Hz)
def student_frames(n): return (embed_frames(n) + 1) // 2            # final Zipformer output (~25 Hz), = your v1 formula

# ------------------------------------------------------------------ alignment from measured frame centres (pure Python)
def centre(A, who, j): return A[who]["slope"] * j + A[who]["intercept"]          # samples
def offset_from(A):
    """Teacher frame index offset that puts frame centres closest: student j <-> teacher j + offset."""
    for k in ("student", "teacher"):
        if abs(A[k]["slope"] - 640.0) > 1.0: raise RuntimeError(f"{k} frame stride measured as {A[k]['slope']:.2f} samples, expected 640")
    return int(round((A["student"]["intercept"] - A["teacher"]["intercept"]) / 640.0))
def valid_pairs(n, Ts, Tt, A, off):
    """Number of usable pairs (student j, teacher j+off), j = 0..v-1: both frames exist and both frame centres lie inside the audio.
    Unmatched frames (teacher frames before off / after the last pair, frames centred beyond the end of the audio) are excluded."""
    if off < 0: raise RuntimeError("negative offsets are not supported (teacher frames would have to be dropped at the start of the student)")
    v = max(0, min(Ts, Tt - off))
    while v > 0 and (centre(A, "student", v - 1) > n or centre(A, "teacher", v - 1 + off) > n): v -= 1
    return v
def analytic_alignment():
    """Geometric cross-check from the layer definitions (kernel, stride, padding); the MEASURED values are what training uses."""
    def c(i, chain):
        x = float(i)
        for k, s, p in chain: x = x * s - p + (k - 1) / 2
        return x
    s0 = 160 * c(0, [(2, 2, 0), (3, 1, 0), (3, 2, 0), (3, 1, 0)]) + 80; s1 = 160 * c(1, [(2, 2, 0), (3, 1, 0), (3, 2, 0), (3, 1, 0)]) + 80
    t0 = wavlm_centre50(0.5); t1 = wavlm_centre50(2.5)            # WavLM target j = 50 Hz frames (2j, 2j+1): centre index 2j + 0.5
    return dict(student=dict(slope=s1 - s0, intercept=s0), teacher=dict(slope=t1 - t0, intercept=t0), source="analytic")

# ------------------------------------------------------------------ Swoosh: pure-PyTorch port of k2's implementation (k2/python/csrc/torch/v2/autograd/swoosh.cu)
K2_SWOOSH_SRC = r'''# Pure-PyTorch port of k2's Swoosh functions (k2/python/csrc/torch/v2/autograd/swoosh.cu), written by the MT2KD ASR Stage 1 pipeline.
# Same formulas, same fp32 compute, same 8-bit stochastically-rounded derivative in the autograd path. Differences from k2:
# the random numbers for the stochastic rounding come from torch's RNG (k2: its own torch::rand call) -> statistically identical, not bitwise.
import torch
_P = {"l": (4.0, 0.08, 0.035), "r": (1.0, 0.08, 0.313261687)}

def _forward(x, shift, coeff, offset):                    # k2 SwooshForward
    x = x.to(torch.float32); xo = x - shift
    l = torch.log1p(torch.exp(xo)); l = torch.where(torch.isinf(l), xo, l)
    return l - coeff * x - offset

def _forward_and_deriv(x, shift, coeff, offset):          # k2 SwooshForwardAndDeriv
    x = x.to(torch.float32); xo = x - shift
    denom = 1.0 + torch.exp(xo); d = (1.0 - coeff) - 1.0 / denom
    ld = torch.log(denom); ld = torch.where(torch.isinf(ld), xo, ld)
    return ld - coeff * x - offset, d

class _SwooshFunction(torch.autograd.Function):           # k2 SwooshFunction (autograd path)
    @staticmethod
    def forward(ctx, x, shift, coeff, offset, dropout_prob):
        x = x.to(torch.float32).contiguous(); xo = x - shift; e = torch.exp(xo)
        g = 1.0 - 1.0 / (1.0 + e)
        l = torch.log1p(e); l = torch.where(torch.isinf(l), xo, l); y = l - coeff * x - offset
        if dropout_prob != 0.0:
            drop = torch.rand_like(x) < dropout_prob
            y = torch.where(drop, torch.zeros_like(y), y * (1.0 / (1.0 - dropout_prob))); g = torch.where(drop, torch.full_like(g, coeff), g)
        g_int = (g * (255.0 / 1.005) + torch.rand_like(x)).to(torch.uint8)
        ctx.save_for_backward(g_int); ctx.coeff = coeff; ctx.dropout_prob = dropout_prob
        return y
    @staticmethod
    def backward(ctx, y_grad):
        (g,) = ctx.saved_tensors
        f = (g.to(torch.float32) * (1.005 / 255.0) - ctx.coeff) / (1.0 - ctx.dropout_prob)
        return y_grad.to(torch.float32) * f, None, None, None, None

def _swoosh(x, which, dropout_prob):
    shift, coeff, offset = _P[which]
    if not x.requires_grad:                               # k2: logaddexp path when no gradient is needed
        x = x.to(torch.float32); return torch.logaddexp(torch.zeros(1, device=x.device), x - shift) - coeff * x - offset
    return _SwooshFunction.apply(x, shift, coeff, offset, float(dropout_prob))

def swoosh_l(x, dropout_prob=0.0): return _swoosh(x, "l", dropout_prob)
def swoosh_r(x, dropout_prob=0.0): return _swoosh(x, "r", dropout_prob)
def swoosh_l_forward(x): return _forward(x, *_P["l"])
def swoosh_r_forward(x): return _forward(x, *_P["r"])
def swoosh_l_forward_and_deriv(x): return _forward_and_deriv(x, *_P["l"])
def swoosh_r_forward_and_deriv(x): return _forward_and_deriv(x, *_P["r"])
'''
STUB_FUNCS = {"swoosh_l", "swoosh_r", "swoosh_l_forward", "swoosh_r_forward", "swoosh_l_forward_and_deriv", "swoosh_r_forward_and_deriv"}

def setup_zipformer():
    """Private copy of icefall's Zipformer source (unchanged except `import k2` -> the Swoosh port). Returns source information."""
    ZDIR = CFG["zipformer_dir"]; SRC_Z = "/content/icefall/egs/librispeech/ASR/zipformer"
    if not os.path.isdir(SRC_Z):
        subprocess.run("rm -rf /content/icefall; git clone --depth 1 https://github.com/k2-fsa/icefall.git /content/icefall", shell=True, check=True)
    os.makedirs(ZDIR + "/icefall", exist_ok=True)
    for f in ["zipformer.py", "scaling.py", "subsampling.py", "encoder_interface.py", "optim.py"]:
        if not os.path.exists(f"{ZDIR}/{f}"): shutil.copy(f"{SRC_Z}/{f}", f"{ZDIR}/{f}")
    o_ = open(f"{ZDIR}/optim.py").read()             # ScaledAdam / Eden, unchanged; lhotse is used only by its self-tests (fix_random_seed)
    if "from lhotse.utils import fix_random_seed" in o_:
        o_ = o_.replace("from lhotse.utils import fix_random_seed",
                        "def fix_random_seed(seed):  # MT2KD: replaces lhotse.utils.fix_random_seed (used only by optim.py's self-tests)\n"
                        "    import random as _r, numpy as _np; _r.seed(seed); _np.random.seed(seed); torch.manual_seed(seed)", 1)
        open(f"{ZDIR}/optim.py", "w").write(o_)
    if re.search(r"^\s*(from|import) lhotse", o_, re.M): raise RuntimeError("icefall optim.py imports lhotse in an unexpected place")
    open(ZDIR + "/icefall/__init__.py", "a").close()
    open(ZDIR + "/icefall/utils.py", "w").write(
        'import torch\n\ndef torch_autocast(device_type="cuda", **kwargs):\n    try:\n        return torch.amp.autocast(device_type, **kwargs)\n'
        '    except (AttributeError, TypeError):\n        return torch.cuda.amp.autocast(**kwargs)\n')
    if not os.path.exists(f"{ZDIR}/k2_swoosh.py") or open(f"{ZDIR}/k2_swoosh.py").read() != K2_SWOOSH_SRC: open(f"{ZDIR}/k2_swoosh.py", "w").write(K2_SWOOSH_SRC)
    t_ = open(f"{ZDIR}/scaling.py").read()
    if "\nimport k2\n" in t_: t_ = t_.replace("\nimport k2\n", "\nimport k2_swoosh as k2\n", 1); open(f"{ZDIR}/scaling.py", "w").write(t_)
    used = set(re.findall(r"\bk2\.(\w+)", t_ + open(f"{ZDIR}/zipformer.py").read() + open(f"{ZDIR}/subsampling.py").read()))
    if used - STUB_FUNCS: raise RuntimeError(f"the Zipformer source uses k2 functions the port does not provide: {used - STUB_FUNCS}")
    sys.path = [p for p in sys.path if p != ZDIR]; sys.path.insert(0, ZDIR); importlib.invalidate_caches()
    for m_ in [k for k in list(sys.modules) if k in ("zipformer", "scaling", "subsampling", "encoder_interface", "k2_swoosh", "optim") or k == "icefall" or k.startswith("icefall.")]:
        if not (getattr(sys.modules[m_], "__file__", "") or "").startswith(ZDIR): del sys.modules[m_]          # e.g. imported earlier from another folder
    try: commit = subprocess.run(["git", "-C", "/content/icefall", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    except Exception: commit = "unknown"
    return dict(dir=ZDIR, icefall_commit=commit, k2_functions_used=sorted(used),
                files={f: file_sha1(f"{ZDIR}/{f}")[:12] for f in ["zipformer.py", "scaling.py", "subsampling.py", "k2_swoosh.py", "optim.py"]})

def build_models(teacher_dim):
    """Returns (student, proj) on CPU. Classes are created here because they need torch + icefall."""
    import torch, torch.nn as nn
    info = setup_zipformer()
    from subsampling import Conv2dSubsampling
    from zipformer import Zipformer2
    import zipformer as zmod, subsampling as smod
    assert zmod.__file__.startswith(CFG["zipformer_dir"]) and smod.__file__.startswith(CFG["zipformer_dir"]), "Zipformer imported from the wrong folder"
    class ZipformerStudent(nn.Module):
        """fbank (B, T, 80) -> Conv2dSubsampling (80 -> 192, ~50 Hz) -> Zipformer2 -> (B, ~T/4, 512) at ~25 Hz. Same keys as v1."""
        def __init__(self):
            super().__init__(); self.encoder_embed = Conv2dSubsampling(in_channels=80, out_channels=192); self.encoder = Zipformer2(**ZIP_ARGS)
        def forward(self, feats, feat_lens, return_embed=False):
            x, xl = self.encoder_embed(feats, feat_lens)
            enc, el = self.encoder(x.permute(1, 0, 2), xl, ~lengths_to_mask(xl, x.size(1)))
            return (enc.permute(1, 0, 2), el, x, xl) if return_embed else (enc.permute(1, 0, 2), el)
    class ASRProj(nn.Module):
        """Paper: a linear layer maps the student's ASR representation to the teacher dimension (512 -> 2048 for WavLM)."""
        def __init__(self): super().__init__(); self.lin = nn.Linear(D_S, teacher_dim)
        def forward(self, x): return self.lin(x)
    s = ZipformerStudent(); s.source_info = info
    return s, ASRProj()

def inspect_student(student, v1_ckpt=None):
    """Architecture actually instantiated from the imported source: stacks, layers, dims, parameters; optional key/shape match with v1."""
    enc = student.encoder; stacks = []
    for i, m in enumerate(enc.encoders):
        inner = getattr(m, "encoder", m); stacks.append(dict(type=type(m).__name__, downsample=int(enc.downsampling_factor[i]), layers=len(inner.layers),
                                                             dim=int(enc.encoder_dim[i]), params=sum(p.numel() for p in m.parameters())))
    R = dict(source=student.source_info, embed=type(student.encoder_embed).__name__, embed_in=80, embed_out=int(student.encoder_embed.out_channels)
             if hasattr(student.encoder_embed, "out_channels") else None, stacks=stacks, output_downsampling=int(enc.output_downsampling_factor),
             output_dim=int(max(enc.encoder_dim)), total_layers=sum(s["layers"] for s in stacks), params=sum(p.numel() for p in student.parameters()))
    exp = ZIP_ARGS
    assert [s["layers"] for s in stacks] == list(exp["num_encoder_layers"]) and [s["dim"] for s in stacks] == list(exp["encoder_dim"])
    assert [s["downsample"] for s in stacks] == list(exp["downsampling_factor"]) and R["output_dim"] == D_S and R["output_downsampling"] == 2
    if v1_ckpt and Path(v1_ckpt).exists():
        import torch
        st = torch.load(v1_ckpt, map_location="cpu", weights_only=False).get("student", {}); mine = student.state_dict()
        R["v1_match"] = dict(path=str(v1_ckpt), same_keys=set(st) == set(mine), same_shapes=all(tuple(st[k].shape) == tuple(mine[k].shape) for k in set(st) & set(mine)),
                             tensors=len(mine))
        if not (R["v1_match"]["same_keys"] and R["v1_match"]["same_shapes"]): raise RuntimeError(f"architecture differs from your v1 student: {R['v1_match']}")
    log(f"  Zipformer-M from {student.source_info['dir']} (icefall {student.source_info['icefall_commit'][:10]}): stacks "
        + ", ".join(f"{s['layers']}x{s['dim']}/{s['downsample']}" for s in stacks) + f" | {R['total_layers']} layers | output {R['output_dim']}-D, /2 -> 25 Hz | "
        f"{R['params'] / 1e6:.2f}M params" + (f" | identical keys+shapes to {Path(v1_ckpt).parent.name}/{Path(v1_ckpt).name}" if R.get("v1_match") else ""))
    return R

def verify_swoosh(device="cuda"):
    """Swoosh port vs reference: forward exactness (fp32/fp16 inputs, extremes), derivative, 8-bit stochastic backward (bounded, unbiased),
    the icefall modules that call it, and the real k2 if it happens to be installed."""
    import torch, torch.nn.functional as F
    setup_zipformer(); import k2_swoosh as P, scaling as S
    dev = torch.device(device if torch.cuda.is_available() else "cpu"); R = dict(source="k2/python/csrc/torch/v2/autograd/swoosh.cu (ported)")
    xs = torch.cat([torch.linspace(-60, 60, 200001), torch.tensor([-65504., -1e4, -100., 100., 1e4, 65504.])]).to(dev)
    for w, (sh, co, of) in {"l": (4.0, 0.08, 0.035), "r": (1.0, 0.08, 0.313261687)}.items():
        x64 = xs.double(); ref = torch.logaddexp(torch.zeros(1, device=dev, dtype=torch.float64), x64 - sh) - co * x64 - of; dref = torch.sigmoid(x64 - sh) - co
        tol = 2e-6 * (1 + ref.abs())
        y = getattr(P, f"swoosh_{w}_forward")(xs); yd, d = getattr(P, f"swoosh_{w}_forward_and_deriv")(xs)
        e_fwd = float(((y.double() - ref).abs() / tol).max()); e_fd = float(((yd.double() - ref).abs() / tol).max()); e_d = float((d.double() - dref).abs().max())
        y16 = getattr(P, f"swoosh_{w}_forward")(xs.half()); y16_ok = bool(torch.isfinite(y16).all()) and y16.dtype == torch.float32
        xg = xs.clone().requires_grad_(True); yg = getattr(P, f"swoosh_{w}")(xg); yg.backward(torch.ones_like(yg)); g = xg.grad.double()
        q = 1.005 / 255.0; e_q = float((g - dref).abs().max()); bias = float((g - dref).mean())
        xr = torch.randn(2_000_000, device=dev) * 6; xr.requires_grad_(True); yr = getattr(P, f"swoosh_{w}")(xr); yr.backward(torch.ones_like(yr))
        bias_r = float((xr.grad.double() - (torch.sigmoid(xr.detach().double() - sh) - co)).mean())
        e_nog = float(((getattr(P, f"swoosh_{w}")(xs).double() - ref).abs() / tol).max())
        mod = (S.SwooshL if w == "l" else S.SwooshR)(); e_mod = float(((mod(xs).double() - ref).abs() / tol).max())
        own = (S.SwooshLForward if w == "l" else S.SwooshRForward)(xs); e_own = float(((own.double() - ref).abs() / tol)[torch.isfinite(own)].max())
        R[w] = dict(forward_err_rel=e_fwd, forward_and_deriv_err_rel=e_fd, deriv_maxerr=e_d, fp16_input_finite_fp32_output=y16_ok, no_grad_path_err_rel=e_nog,
                    backward_maxerr=e_q, backward_quant_step=q, backward_bias_grid=bias, backward_bias_random=bias_r, icefall_module_err_rel=e_mod,
                    icefall_reference_SwooshForward_err_rel=e_own)
        ok = e_fwd <= 1 and e_fd <= 1 and e_d < 1e-6 and y16_ok and e_nog <= 1 and e_q <= q + 1e-6 and abs(bias_r) < 2e-4 and e_mod <= 1 and e_own <= 1
        R[w]["passed"] = ok
        log(f"  swoosh_{w}: forward rel.err {e_fwd:.2f} (<=1 = within 2e-6 relative) | deriv {e_d:.1e} | fp16 inputs finite {y16_ok} | backward |err| {e_q:.4f} "
            f"<= 8-bit step {q:.4f}, mean bias {bias_r:+.1e} | icefall SwooshL/R module {e_mod:.2f}, icefall reference {e_own:.2f} -> {'OK' if ok else 'FAIL'}")
    # the fused ActivationDropoutAndLinear path (uses forward_and_deriv in backward) vs autograd through the reference, dropout 0
    torch.manual_seed(0); lin = S.ActivationDropoutAndLinear(64, 32, bias=True, activation="SwooshL", dropout_p=0.0).to(dev)
    x = (torch.randn(50, 64, device=dev) * 4).requires_grad_(True); y = lin(x); y.square().sum().backward(); gx = x.grad.clone(); gw = lin.weight.grad.clone()
    x2 = x.detach().double().requires_grad_(True); W = lin.weight.detach().double().requires_grad_(True); b = lin.bias.detach().double()
    y2 = F.linear(torch.logaddexp(torch.zeros(1, device=dev, dtype=torch.float64), x2 - 4) - 0.08 * x2 - 0.035, W, b); y2.square().sum().backward()
    R["fused_linear"] = dict(out_rel=float((y.double() - y2).abs().max() / y2.abs().max()), dx_rel=float((gx.double() - x2.grad).abs().max() / x2.grad.abs().max()),
                             dw_rel=float((gw.double() - W.grad).abs().max() / W.grad.abs().max()))
    R["fused_linear"]["passed"] = max(R["fused_linear"].values()) < 1e-4
    log(f"  ActivationDropoutAndLinear(SwooshL) vs float64 autograd: output {R['fused_linear']['out_rel']:.1e}, dx {R['fused_linear']['dx_rel']:.1e}, "
        f"dW {R['fused_linear']['dw_rel']:.1e} (relative)")
    try:
        import k2 as real
        if not hasattr(real, "swoosh_l_forward"): raise ImportError
        xr = torch.randn(100000, device=dev) * 8
        R["real_k2"] = dict(forward_l_maxdiff=float((real.swoosh_l_forward(xr) - P.swoosh_l_forward(xr)).abs().max()),
                            forward_r_maxdiff=float((real.swoosh_r_forward(xr) - P.swoosh_r_forward(xr)).abs().max()),
                            deriv_l_maxdiff=float((real.swoosh_l_forward_and_deriv(xr)[1] - P.swoosh_l_forward_and_deriv(xr)[1]).abs().max()))
        log(f"  compared with the installed k2: {R['real_k2']}")
    except Exception: R["real_k2"] = "k2 not installed for this PyTorch (not downgraded); compared against k2's published formulas and algorithm instead"
    R["passed"] = R["l"]["passed"] and R["r"]["passed"] and R["fused_linear"]["passed"]
    R["claim"] = ("forward and derivative: same formulas as k2 in fp32 (verified to <= 2e-6 relative against float64); autograd backward: same 8-bit "
                  "stochastic quantisation of the derivative as k2 (bounded by one step, unbiased); random numbers differ, so results are not bit-identical to a k2 run")
    return R

def measure_student_timing(student):
    """Measures, with the imported code, where each student output frame sits in time:
    (1) kaldi fbank frame centre from impulse probes through the real fbank() below, (2) Conv2dSubsampling index map from perturbation probes,
    (3) the output downsampler (SimpleDownsample) index map. Returns a linear map j -> centre (samples)."""
    import torch
    S = np.arange(8000, 8000 + 1280, 4); arg = []
    for s in S:
        w = np.zeros(16000, np.float32); w[s] = 0.5; arg.append(int(fbank(w)[:100].sum(1).argmax()))
    arg = np.array(arg); cen = {int(k): float(S[arg == k].mean()) for k in np.unique(arg) if (arg == k).sum() >= 8}
    c_fb = float(np.median([c - 160 * k for k, c in cen.items()]))
    dev = next(student.parameters()).device; was = student.training; student.eval(); L = 80
    g = torch.Generator().manual_seed(0); x = torch.randn(1, L, 80, generator=g).to(dev); xl = torch.tensor([L], device=dev)
    with torch.no_grad():
        y0, _ = student.encoder_embed(x, xl); infl = [[] for _ in range(y0.shape[1])]
        for k in range(L):
            x2 = x.clone(); x2[:, k] += 5.0; ch = (student.encoder_embed(x2, xl)[0] - y0).abs().amax(-1)[0] > 1e-5
            for i in torch.nonzero(ch).flatten().tolist(): infl[i].append(k)
        ii = [i for i in range(6, y0.shape[1] - 6) if infl[i]]; ci = np.array([(min(infl[i]) + max(infl[i])) / 2 for i in ii])
        a_e, b_e = np.polyfit(ii, ci, 1); res_e = float(np.abs(np.polyval([a_e, b_e], ii) - ci).max())
        ds = student.encoder.downsample_output; C = max(ZIP_ARGS["encoder_dim"]); T2 = 20
        z = torch.randn(T2, 1, C, generator=g).to(dev); o0 = ds(z); inf2 = [[] for _ in range(o0.shape[0])]
        for t in range(T2):
            z2 = z.clone(); z2[t] += 3.0; ch = (ds(z2) - o0).abs().amax(-1)[:, 0] > 1e-6
            for j in torch.nonzero(ch).flatten().tolist(): inf2[j].append(t)
        jj = [j for j in range(o0.shape[0] - 1) if inf2[j]]; cj = np.array([(min(inf2[j]) + max(inf2[j])) / 2 for j in jj])
        a_d, b_d = np.polyfit(jj, cj, 1); res_d = float(np.abs(np.polyval([a_d, b_d], jj) - cj).max())
    student.train(was)
    slope = 160 * a_e * a_d; icpt = c_fb + 160 * (a_e * b_d + b_e)
    R = dict(fbank_centre_offset_samples=c_fb, embed_slope=float(a_e), embed_intercept=float(b_e), embed_fit_residual=res_e, embed_influence_example={int(i): infl[i] for i in ii[:2]},
             down_slope=float(a_d), down_intercept=float(b_d), down_fit_residual=res_d, down_influence_example={int(j): inf2[j] for j in jj[:3]},
             centre_samples=dict(slope=float(slope), intercept=float(icpt)))
    log(f"  student timing (measured): fbank frame k centre = 160 k + {c_fb:.1f} samples | Conv2dSubsampling: embed frame i <- fbank frames centred at "
        f"{a_e:.3f} i + {b_e:.3f} (residual {res_e:.3f}) | output downsample: j <- embed {a_d:.3f} j + {b_d:.3f} | student j centre = {slope / 16:.1f} j + {icpt / 16:.2f} ms")
    return R

def lengths_to_mask(lens, T):
    import torch
    return torch.arange(T, device=lens.device)[None, :] < lens[:, None]

def set_batch_count(model, bc):
    for name, mod in model.named_modules():
        if hasattr(mod, "batch_count"): mod.batch_count = bc
        if hasattr(mod, "name"): mod.name = name

def fbank(wav):
    """Student frontend = your v1 ASR fbank (80 bins, 25 ms / 10 ms, povey window, snip_edges=False)."""
    import torch, torchaudio.compliance.kaldi as kaldi
    return kaldi.fbank(torch.from_numpy(wav)[None], num_mel_bins=80, frame_length=25.0, frame_shift=10.0, dither=0.0, energy_floor=1e-10,
                       sample_frequency=float(SR), snip_edges=False, low_freq=20.0, high_freq=-400.0, remove_dc_offset=True,
                       preemphasis_coefficient=0.97, window_type="povey", use_energy=False)

# ------------------------------------------------------------------ alignment and loss
def align_targets(t, t_lens, s_lens, n_samples, S_T, A, off):
    """t [B, Tt, D] padded teacher -> (targets [B, S_T, D] fp32, valid pairs [B]): student frame j <-> teacher frame j + off for j < v,
    v = valid_pairs(...). Frames without a valid partner are excluded from the loss by the mask (no interpolation, no padding as target)."""
    import torch
    B, _, Dm = t.shape; out = torch.zeros(B, S_T, Dm, device=t.device, dtype=torch.float32); v = torch.zeros(B, dtype=torch.long, device=t.device)
    for b in range(B):
        n_ = valid_pairs(int(n_samples[b]), int(s_lens[b]), int(t_lens[b]), A, off)
        out[b, :n_] = t[b, off:off + n_].float(); v[b] = n_
    return out, v

def masked_l1(pred, tgt, valid):
    """Paper's ASR KD loss: L1 between projected student and teacher features, mean over the teacher dims and over VALID frames only."""
    mask = lengths_to_mask(valid, pred.size(1)).float(); nf = mask.sum().clamp(min=1)
    return ((pred.float() - tgt).abs().mean(-1) * mask).sum() / nf, nf

# ------------------------------------------------------------------ data
def make_batches(lens, epoch, seed, shuffle, max_sec, max_utts, idx=None):
    """Duration batches over `idx` (default: all): padded seconds <= max_sec and <= max_utts items; sorted inside shuffled pools of 1000;
    deterministic for (seed, epoch, idx)."""
    lens = np.asarray(lens); idx = np.arange(len(lens)) if idx is None else np.asarray(sorted(idx))
    rng = np.random.default_rng(seed + 1000 * epoch + (0 if len(idx) == len(lens) else 7919 * len(idx)))
    order = idx[rng.permutation(len(idx))] if shuffle else idx; out = []
    for s in range(0, len(order), 1000):
        ids = order[s:s + 1000]; ids = ids[np.argsort(lens[ids], kind="stable")]; cur, mx = [], 0
        for i in ids:
            nm = max(mx, lens[i])
            if cur and (nm * (len(cur) + 1) / SR > max_sec or len(cur) >= max_utts): out.append(cur); cur, nm = [], lens[i]
            cur.append(int(i)); mx = nm
        if cur: out.append(cur)
    if shuffle: out = [out[j] for j in rng.permutation(len(out))]
    return out

class KDData:
    def __init__(self, df, cache): self.r = df.to_dict("records"); self.c = cache
    def __len__(self): return len(self.r)
    def __getitem__(self, i):
        import torch
        r = self.r[i]; w = load_audio(r["relpath"], int(r["n_samples"])); t = self.c.get(r["uid"])
        return dict(uid=r["uid"], n=len(w), fb=fbank(w), t=torch.from_numpy(t))

def collate(b):
    import torch, math
    from torch.nn.utils.rnn import pad_sequence
    return dict(uid=[x["uid"] for x in b], n=torch.tensor([x["n"] for x in b]),
                fb=pad_sequence([x["fb"] for x in b], batch_first=True, padding_value=math.log(1e-10)), fb_lens=torch.tensor([x["fb"].size(0) for x in b]),
                t=pad_sequence([x["t"] for x in b], batch_first=True), t_lens=torch.tensor([x["t"].size(0) for x in b]))

def loader(df, cache, batches, num_workers, seed=0):
    import torch
    def wi(_): torch.set_num_threads(1)
    return torch.utils.data.DataLoader(KDData(df, cache), batch_sampler=batches, num_workers=num_workers, collate_fn=collate, pin_memory=True,
                                       worker_init_fn=wi, prefetch_factor=4 if num_workers else None,
                                       generator=torch.Generator().manual_seed(int(seed)))      # own generator: the global RNG streams are not consumed

# ####################################################################################################
# ---- asr1_train ----
# ####################################################################################################
# ===========================================================================================
# MT2KD ASR Stage 1 | alignment measurement, smoke test (critical checks, machine-readable report), resumable KD training,
#                    validation, early stopping, memory probe, plots, report, single-file export to MT2KD/ASR_outputs/
#   Resume semantics (stated honestly): training resumes from the last saved UPDATE BOUNDARY (local / Drive checkpoints),
#   with model, projection, optimiser, GradScaler, schedule position, epoch, batch index, seen utterances, RNG states and history restored.
#   Updates made after that save are recomputed; replays use the same batches and RNG state but GPU kernels are not bit-deterministic.
# ===========================================================================================
import os, json, math, time, shutil
from pathlib import Path
import numpy as np, pandas as pd

def ALIGN_F(): return paths()["logs"] / "alignment.json"
def SMOKE_F(): return paths()["logs"] / "smoke" / "smoke_report.json"
V1_CKPT = lambda: paths()["root"] / "stage1_asr_kd" / "best.pt"

# ===========================================================================================
# alignment: measured with the imported WavLM and Zipformer code
# ===========================================================================================
def measure_alignment(teacher, train_df, dev_df):
    """Measures both frame-centre maps, checks them against the layer-geometry analysis, derives the offset, and tests the pairing on
    short / medium / near-30 s / > 30 s utterances. Writes logs/ASR_Stage1/alignment.json (training reads it)."""
    import torch
    T = measure_teacher_timing(teacher); student, _ = build_models(2 * WAVLM_DIM); S = measure_student_timing(student)
    M = dict(student=S["centre_samples"], teacher=T["centre_samples"]); An = analytic_alignment()
    dev_s = {k: abs(M[k]["intercept"] - An[k]["intercept"]) for k in ("student", "teacher")}
    off = offset_from(M); res_ms = (centre(M, "student", 0) - centre(M, "teacher", off)) / 16
    log(f"  measured vs analytic frame centres: student {M['student']['intercept'] / 16:.2f} vs {An['student']['intercept'] / 16:.2f} ms, "
        f"teacher {M['teacher']['intercept'] / 16:.2f} vs {An['teacher']['intercept'] / 16:.2f} ms | both strides {M['student']['slope'] / 16:.1f} / "
        f"{M['teacher']['slope'] / 16:.1f} ms")
    log(f"  => student frame j <-> teacher frame j + {off}; remaining centre difference {res_ms:+.2f} ms (for offset 0 it would be "
        f"{(centre(M, 'student', 0) - centre(M, 'teacher', 0)) / 16:+.1f} ms)")
    table = []
    for k, r in pick_test_utts(train_df, dev_df).items():
        if r is None: continue
        n = int(r.n_samples); Ts, Tt = student_frames(n), teacher_frames(n); v = valid_pairs(n, Ts, Tt, M, off)
        table.append(dict(kind=k, uid=r.uid, seconds=round(n / SR, 3), student_frames=Ts, teacher_frames=Tt, pairs=v, unmatched_teacher=Tt - v,
                          unmatched_student=Ts - v, first_pair_ms=[centre(M, "student", 0) / 16, centre(M, "teacher", off) / 16],
                          last_pair_ms=[centre(M, "student", v - 1) / 16, centre(M, "teacher", v - 1 + off) / 16], audio_end_ms=1000 * n / SR))
        log(f"  {k:7s} {r.uid}: {n / SR:6.2f} s | student {Ts} | teacher {Tt} | pairs {v} (unmatched: teacher {Tt - v}, student {Ts - v}) | "
            f"last pair centres {centre(M, 'student', v - 1) / 16:.0f} / {centre(M, 'teacher', v - 1 + off) / 16:.0f} ms, audio ends {1000 * n / SR:.0f} ms")
    ok = (max(dev_s.values()) <= 16 and abs(res_ms) <= 20 and all(t["pairs"] > 0 and t["unmatched_student"] <= 1 for t in table))
    A = dict(measured=M, analytic=An, measured_minus_analytic_samples=dev_s, offset=off, residual_ms=res_ms, teacher_detail=T, student_detail=S,
             table=table, passed=ok, note="training uses the MEASURED maps; the analytic values are an independent cross-check")
    atomic_json(A, ALIGN_F()); log(f"  alignment {'PASSED' if ok else 'FAILED'} -> {ALIGN_F()}")
    del student; torch.cuda.empty_cache()
    return A

def load_alignment():
    if not ALIGN_F().exists(): raise FileNotFoundError(f"{ALIGN_F()} missing: run the alignment cell")
    A = json.load(open(ALIGN_F()))
    if not A.get("passed"): raise RuntimeError("the alignment measurement did not pass; see alignment.json")
    return A

# ===========================================================================================
# shared context
# ===========================================================================================
def context(preset, local=True, with_alignment=True):
    """Manifests, local audio, teacher caches, and (with_alignment) the measured alignment and aligned-frame baselines."""
    sub = PRESETS[preset]["train_subset"]; P = paths(preset); ensure_dirs(P)
    tr = load_manifest(sub); dv = load_manifest("dev")
    materialize(tr, "train-clean-100"); materialize(dv, CFG["val_split"])
    dtr, ddv = find_cache("train-clean-100"), find_cache(CFG["val_split"])
    TCtr, TCdv = TeacherCache(dtr, tr.uid, local), TeacherCache(ddv, dv.uid, local)
    base = dict(preset=preset, sub=sub, P=P, tr=tr, dv=dv, TCtr=TCtr, TCdv=TCdv, teacher_meta=TCtr.meta, manifest_sha=sha_of(sorted(tr.uid)), dev_sha=sha_of(sorted(dv.uid)))
    if not with_alignment: return base
    A = load_alignment(); M = A["measured"]; off = A["offset"] if CFG["align"] == "shift" else 0
    rng = {}
    for name, df, TC in [("train", tr, TCtr), ("dev", dv, TCdv)]:
        Tt = TC.idx.loc[df.uid, "length"].values; Ts = np.array([student_frames(n) for n in df.n_samples])
        v = np.array([valid_pairs(int(n), int(s), int(t), M, off) for n, s, t in zip(df.n_samples, Ts, Tt)])
        if (v <= 0).any(): raise RuntimeError(f"{name}: utterances without a valid frame pair: {df.uid[v <= 0].tolist()[:3]}")
        log(f"{name}: {len(df):,} utts | pairs {v.sum():,} of {Ts.sum():,} student / {Tt.sum():,} teacher frames | unmatched per utterance: "
            f"student max {int((Ts - v).max())}, teacher {int((Tt - v).min())}-{int((Tt - v).max())}")
        if name == "train": rng = {u: (off, int(x)) for u, x in zip(df.uid, v)}
    mean, med = TCtr.baselines(rng, cache_file=P["logs"] / f"baselines_{sha_of(sorted(tr.uid))}_{off}.npz")
    return dict(base, A=M, offset=off, align=A, mean=mean, median=med)

def amp_dtype():
    import torch
    return torch.bfloat16 if torch.cuda.get_device_properties(0).major >= 8 else torch.float16

def forward_loss(student, proj, b, C, dev, amp):
    import torch
    fb, fl = b["fb"].to(dev, non_blocking=True), b["fb_lens"].to(dev); t = b["t"].to(dev, non_blocking=True)
    with torch.autocast(device_type="cuda", dtype=amp):
        enc, el = student(fb, fl); pred = proj(enc)
    tgt, v = align_targets(t, b["t_lens"], el, b["n"], enc.size(1), C["A"], C["offset"])
    l1, nf = masked_l1(pred, tgt, v)
    with torch.no_grad():
        mask = lengths_to_mask(v, enc.size(1)).float()
        mu, md = torch.as_tensor(C["mean"], device=dev), torch.as_tensor(C["median"], device=dev)
        b_mean = ((tgt - mu).abs().mean(-1) * mask).sum() / nf; b_med = ((tgt - md).abs().mean(-1) * mask).sum() / nf
        cos = (torch.nn.functional.cosine_similarity(pred.float(), tgt, dim=-1) * mask).sum() / nf
    return l1, dict(base_mean=float(b_mean), base_median=float(b_med), cos=float(cos), frames=int(nf), enc=enc, el=el, tgt=tgt, v=v, pred=pred)

def evaluate(student, proj, C, dev, amp):
    """Frame-weighted validation metrics over the whole dev manifest (no parameter updates)."""
    import torch
    was = student.training; student.eval(); proj.eval(); s = bm = bd = c_ = 0.0; n = 0; nu = 0
    bt = make_batches(C["dv"].n_samples.values, 0, CFG["seed"], False, 300.0, CFG["max_batch_utts"])
    with torch.no_grad():
        for b in loader(C["dv"], C["TCdv"], bt, CFG["num_workers"]):
            l1, i = forward_loss(student, proj, b, C, dev, amp)
            if not torch.isfinite(l1): raise FloatingPointError("non-finite validation loss")
            f = i["frames"]; s += float(l1) * f; bm += i["base_mean"] * f; bd += i["base_median"] * f; c_ += i["cos"] * f; n += f; nu += len(b["uid"])
    student.train(was); proj.train(was)
    assert nu == len(C["dv"]), f"validation covered {nu} of {len(C['dv'])} utterances"
    return dict(val_l1=s / n, val_base_mean_l1=bm / n, val_base_median_l1=bd / n, val_ratio_mean=s / bm, val_ratio_median=s / bd, val_cos=c_ / n,
                val_frames=n, val_utts=nu)

def new_models(C, dev):
    import torch
    set_seed(CFG["seed"]); student, proj = build_models(int(C["teacher_meta"]["dim"]))
    if CFG["init_bias"] == "train_mean":
        with torch.no_grad(): proj.lin.bias.copy_(torch.as_tensor(C["mean"]))
    return student.to(dev), proj.to(dev)

def make_optimizer(student, proj, cfg):
    """(optimizer, scheduler). Paper / icefall: ScaledAdam(named params, clipping_scale=2.0) + Eden(lr_batches, lr_epochs, warmup_start=0.1).
    icefall groups parameters by per-module 'lr_scale'; no Zipformer module defines one, so a single group is identical to the recipe.
    Eden's lr is a pure function of (global step, epoch), both stored in the training state, so a resume reproduces it exactly."""
    import torch
    if cfg["optimizer"] == "scaled_adam":
        setup_zipformer(); from optim import ScaledAdam, Eden
        named = [("student." + n, p) for n, p in student.named_parameters()] + [("proj." + n, p) for n, p in proj.named_parameters()]
        opt = ScaledAdam([dict(named_params=named, lr=cfg["base_lr"])], lr=cfg["base_lr"], clipping_scale=cfg["clipping_scale"])
        sch = Eden(opt, cfg["lr_batches"], cfg["lr_epochs"], warmup_batches=cfg["eden_warmup_batches"], warmup_start=cfg["eden_warmup_start"])
        sch.step_epoch(0); sch.step_batch(0)
        return opt, sch
    if cfg["optimizer"] == "adamw":
        params = list(student.parameters()) + list(proj.parameters())
        return torch.optim.AdamW([dict(params=[p for p in params if p.ndim >= 2], weight_decay=cfg["weight_decay"]),
                                  dict(params=[p for p in params if p.ndim < 2], weight_decay=0.0)], lr=cfg["lr"]), None
    raise ValueError(f"unknown optimizer {cfg['optimizer']}")

def memory_probe(student, proj, C, dev, amp, max_sec):
    """Measured (not assumed) peak GPU memory of the worst-case training batch: the longest training utterances, padded, forward + backward.
    The batch limit is lowered by 20 % steps until the peak is below mem_fraction of the GPU."""
    import torch
    L = int(C["tr"].n_samples.max()); total = torch.cuda.get_device_properties(0).total_memory; scaler = torch.amp.GradScaler("cuda", enabled=amp == torch.float16)
    while max_sec >= 20:
        nb = max(1, min(CFG["max_batch_utts"], int(max_sec * SR // L))); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(); peak = float("nan")
        try:
            b = dict(fb=torch.randn(nb, fbank_frames(L), 80), fb_lens=torch.full((nb,), fbank_frames(L)), t=torch.randn(nb, teacher_frames(L), int(C["teacher_meta"]["dim"])).half(),
                     t_lens=torch.full((nb,), teacher_frames(L)), n=torch.full((nb,), L))
            student.train(); proj.train(); l1, _ = forward_loss(student, proj, b, C, dev, amp); scaler.scale(l1).backward()
            peak = torch.cuda.max_memory_allocated(); ok = peak < CFG["mem_fraction"] * total
        except oom_error(): ok = False
        for p in list(student.parameters()) + list(proj.parameters()): p.grad = None
        b = l1 = None; torch.cuda.empty_cache()
        log(f"  memory probe: {nb} x {L / SR:.1f} s (longest training utterance) = {nb * L / SR:.0f} s padded -> peak {peak / 2**30:.2f} GiB of "
            f"{total / 2**30:.1f} GiB {'OK' if ok else '-> too much, reducing'}")
        if ok: return max_sec, peak / 2**30
        max_sec *= 0.8
    raise RuntimeError("even a 20 s batch does not fit in GPU memory")

# ===========================================================================================
# smoke test
# ===========================================================================================
import re
UNUSED_ICEFALL = re.compile(r"(^|\.)layers\.\d+\.bypass_scale$")     # Zipformer2EncoderLayer.bypass_scale: defined, never used by icefall

def smoke_test(preset="pilot", n_utts=4):
    """All checks; critical failures block training. Writes logs/ASR_Stage1/smoke/smoke_report.json."""
    import torch, gc
    dev = torch.device("cuda"); amp = amp_dtype(); R = []; st = {}; out = SMOKE_F().parent; out.mkdir(parents=True, exist_ok=True)
    sc = 1024.0 if amp == torch.float16 else 1.0
    def _sbw(loss):          # fp16 (T4): scaled backward + unscaled gradients (what GradScaler does); bf16 (A100): plain backward
        (loss * sc).backward()
        if sc != 1.0:
            for p in list(st["S"].parameters()) + list(st["Pj"].parameters()):
                if p.grad is not None: p.grad.div_(sc)
    def chk(no, name, fn, critical=True):
        t0 = time.time()
        try: info = fn(); R.append(dict(no=no, check=name, critical=critical, status="PASS", info=info, seconds=round(time.time() - t0, 1))); log(f"[{no}] PASS  {name}: {info}")
        except Exception as e:
            R.append(dict(no=no, check=name, critical=critical, status="FAIL", info=f"{type(e).__name__}: {e}", seconds=round(time.time() - t0, 1)))
            log(f"[{no}] {'FAIL' if critical else 'WARN'}  {name}: {type(e).__name__}: {e}")
    def d1():
        P = paths(); sel = json.load(open(P["meta"] / "subset_selection.json")); s2, dv = load_manifest("2h"), load_manifest("dev"); s10 = s2
        tgt = 3600 * CFG["subsets"]["2h"]; assert tgt <= s2.duration_s.sum() < tgt + s2.duration_s.max() + 1e-6, f"2 h subset is {s2.duration_s.sum():.1f} s"
        assert not set(s10.uid) & set(dv.uid) and not set(s10.speaker) & set(dv.speaker), "train/dev overlap"
        kind, _ = find_source("test-clean")
        if kind: te = inventory("test-clean"); assert not set(s10.uid) & set(te.uid) and not set(s10.speaker) & set(te.speaker), "train/test overlap"
        for split, df in [("train-clean-100", s10), (CFG["val_split"], dv)]:
            ver = set(pd.read_csv(P["meta"] / f"decode_verified_v2_{split}.csv", dtype={"uid": str}).uid); miss = set(df.uid) - ver
            assert not miss, f"{len(miss)} {split} manifest files were never fully decoded"
        assert all(df.split.eq(sp).all() for df, sp in [(s10, "train-clean-100"), (dv, CFG["val_split"])])
        st["dfs"] = (s2, dv)
        return (f"2h {len(s2)} utts / {s2.duration_s.sum():.1f} s | dev {sel['validation']['coverage']} | "
                f"ids unique, transcripts present, all files fully decoded, no train/dev{'/test' if kind else ''} overlap")
    def d2():
        C = context(preset, with_alignment=False); st["C"] = C
        rows = C["tr"].sort_values("duration_s").iloc[np.linspace(0, len(C["tr"]) - 1, n_utts).astype(int)].reset_index(drop=True); st["rows"] = rows
        ws = [load_audio(r.relpath, int(r.n_samples)) for r in rows.itertuples()]; st["w"] = ws
        return f"{n_utts} utterances {[len(w) for w in ws]} samples, mono float32 {SR} Hz, finite, exact lengths"
    def w1():
        st["T"] = WavLMTeacher(); T = st["T"]; return f"{T.info['model']} commit {T.commit[:10]}, {T.n_layers} layers, using layer {T.layer}, d_model {T.d_model}, {T.dtype_name}, eval, frozen"
    def w2():
        s2, dv = st["dfs"]; Rw = verify_wavlm(st["T"], st["C"]["tr"], dv)
        return f"frame rule OK, layer identity OK, pairing cos {min(Rw['pairing']['cos']):.6f}, fp16 vs fp32 cos {Rw['fp16_vs_fp32_cos']}, long utterances whole, cache dtype {Rw['dtype_used']}"
    def w3():
        C = st["C"]; nt, nd = len(C["TCtr"].idx), len(C["TCdv"].idx)
        assert C["TCtr"].idx.index.is_unique and C["TCdv"].idx.index.is_unique and nt == len(C["tr"]) and nd == len(C["dv"])
        for TC in (C["TCtr"], C["TCdv"]):
            for k in ["model", "layer", "pairing", "dim", "frames_rule", "normalize", "cache_format"]: assert TC.meta[k] == st["T"].info.get(k, TC.meta[k]), k
            assert TC.meta["commit"] == st["T"].commit, "cache made with a different WavLM revision"
        return f"one entry per manifest utterance (train {nt}, dev {nd}); shards sha1-verified; metadata matches the loaded WavLM (commit, layer, pairing, normalisation)"
    def w4():
        C = st["C"]; a = verify_cache_vs_fresh(st["T"], C["TCtr"], C["tr"], 6); b = verify_cache_vs_fresh(st["T"], C["TCdv"], C["dv"], 4, seed=1)
        assert a["passed"] and b["passed"]; return f"min cosine train {a['min_cos']:.6f}, dev {b['min_cos']:.6f} (includes the longest utterances)"
    def a1():
        s2, dv = st["dfs"]; A = measure_alignment(st["T"], st["C"]["tr"], dv); assert A["passed"]; st["T"].enc.cpu(); del st["T"]; gc.collect(); torch.cuda.empty_cache()
        st["C"] = context(preset)                 # re-read: uses the alignment just measured
        return f"student j <-> teacher j+{A['offset']}, centre difference {A['residual_ms']:+.2f} ms, measured = analytic within {max(A['measured_minus_analytic_samples'].values()):.1f} samples"
    def s1():
        Rs = verify_swoosh(); st["swoosh"] = Rs; assert Rs["passed"]; return "forward exact to 2e-6 rel., derivative exact, backward = k2's 8-bit stochastic scheme (bounded, unbiased)"
    def z1():
        C = st["C"]; st["S"], st["Pj"] = new_models(C, dev); Ri = inspect_student(st["S"], V1_CKPT()); st["arch"] = Ri
        return f"{Ri['total_layers']} layers, stacks {[s['layers'] for s in Ri['stacks']]}, dims {[s['dim'] for s in Ri['stacks']]}, {Ri['params'] / 1e6:.2f}M params" + \
               (", same keys/shapes as v1" if Ri.get("v1_match") else "")
    def z2():
        b = collate([KDData(st["rows"], st["C"]["TCtr"])[i] for i in range(n_utts)]); st["b"] = b
        assert b["fb"].shape[-1] == 80 and b["fb_lens"].tolist() == [fbank_frames(n) for n in b["n"].tolist()] and torch.isfinite(b["fb"]).all()
        return f"fbank {tuple(b['fb'].shape)} (80-D, 10 ms), lens {b['fb_lens'].tolist()}"
    def z3():
        S = st["S"]; b = st["b"]; S.eval()
        with torch.no_grad():
            enc, el, emb, xl = S(b["fb"].to(dev), b["fb_lens"].to(dev), return_embed=True); diffs = []
            for k in range(n_utts):                                         # padding mask: padded batch == each utterance alone
                L = int(b["fb_lens"][k]); e1, l1_ = S(b["fb"][k:k + 1, :L].to(dev), b["fb_lens"][k:k + 1].to(dev)); n_ = int(l1_[0]) - 4
                # last 4 frames excluded: icefall's stack downsampling pairs the last valid frame with a padded one (inherent, also in icefall training)
                diffs.append(float((enc[k, :n_] - e1[0, :n_]).abs().max() / e1[0, :n_].abs().max()))
        S.train()
        assert emb.shape[-1] == 192 and xl.tolist() == [embed_frames(n) for n in b["n"].tolist()]
        assert enc.shape[-1] == D_S and el.tolist() == [student_frames(n) for n in b["n"].tolist()] and torch.isfinite(enc).all() and max(diffs) < 2e-3, diffs
        return f"embed {tuple(emb.shape)} -> encoder {tuple(enc.shape)}, lens {el.tolist()}; padded batch == single utterances except the last 4 frames (max rel. diff {max(diffs):.1e})"
    def l1():
        C = st["C"]; b = st["b"]; S, Pj = st["S"], st["Pj"]; set_batch_count(S, 100000)   # regularisers at end values: every layer in the graph
        loss, i = forward_loss(S, Pj, b, C, dev, amp); st["loss"] = loss; st["i"] = i
        pred = i["pred"].detach().float().cpu().numpy(); tl = b["t"].float().numpy(); vals = []
        for k in range(n_utts):                                             # independent numpy re-implementation of pairing + mask + L1
            n = int(b["n"][k]); v = valid_pairs(n, student_frames(n), teacher_frames(n), C["A"], C["offset"])
            vals.append(np.abs(pred[k, :v] - tl[k, C["offset"]:C["offset"] + v]).mean(-1))
        man = float(np.concatenate(vals).mean()); assert torch.isfinite(loss) and abs(float(loss) - man) < 1e-3 * max(1.0, man), (float(loss), man)
        b2 = dict(b); t2 = b["t"].clone()
        for k, L in enumerate(b["t_lens"].tolist()): t2[k, L:] = 1e4; t2[k, :C["offset"]] = -1e4      # padding and unmatched leading frames
        b2["t"] = t2; S.eval(); Pj.eval()
        with torch.no_grad(): la, _ = forward_loss(S, Pj, b, C, dev, amp); lb, _ = forward_loss(S, Pj, b2, C, dev, amp)
        S.train(); Pj.train(); assert abs(float(la) - float(lb)) < 1e-6, (float(la), float(lb))
        return (f"L1 {float(loss):.4f} == numpy {man:.4f} | pred {tuple(i['pred'].shape)} vs target {tuple(i['tgt'].shape)} | pairs {i['v'].tolist()} | "
                f"garbage in padding/unmatched teacher frames: no effect | baselines mean {i['base_mean']:.4f}, median {i['base_median']:.4f}")
    def g1(): _sbw(st["loss"]); return "backward OK" + (" (fp16: loss-scaled)" if sc != 1.0 else "")
    def g2():
        S, Pj = st["S"], st["Pj"]; out_ = []
        for g, m in {"encoder_embed": S.encoder_embed, **{f"block{k + 1}": S.encoder.encoders[k] for k in range(6)}, "downsample_output": S.encoder.downsample_output, "projection": Pj}.items():
            gs = [p.grad for p in m.parameters() if p.grad is not None]; assert gs, f"{g}: no gradient"
            nrm = float(torch.sqrt(sum((x.float() ** 2).sum() for x in gs))); assert all(torch.isfinite(x).all() for x in gs) and nrm > 0, g; out_.append(f"{g} {nrm:.1e}")
        return " | ".join(out_)
    def g3():
        S, Pj = st["S"], st["Pj"]; none = [n for n, p in list(S.named_parameters()) + list(Pj.named_parameters()) if p.requires_grad and p.grad is None]
        known = [n for n in none if UNUSED_ICEFALL.search(n)]; other = sorted(set(none) - set(known)); st["unused"] = set(known)
        assert not other, f"{len(other)} trainable tensors received no gradient (not a known unused icefall parameter): {other[:8]}"
        return ("every trainable student/projection tensor received a gradient" if not known else
                f"every USED tensor received a gradient; {len(known)} layer-level 'bypass_scale' tensors get none because icefall's Zipformer2EncoderLayer "
                f"defines but never uses them ('# TODO: remove it'; the bypass uses bypass.bypass_scale / bypass_mid.bypass_scale)")
    def g4():
        S, Pj = st["S"], st["Pj"]; params = list(S.parameters()) + list(Pj.parameters())
        assert all(p.requires_grad for p in params), "some student/projection parameters are frozen"
        return "no teacher in the graph (cached features); WavLM was frozen (eval, requires_grad False) and is released"
    def o1():
        S, Pj = st["S"], st["Pj"]; params = list(S.parameters()) + list(Pj.parameters())
        opt, _ = make_optimizer(S, Pj, dict(CFG, **PRESETS[preset])); st["opt"] = opt
        named = dict(list(S.named_parameters()) + [("proj." + k, v) for k, v in Pj.named_parameters()]); before = {k: v.detach().clone() for k, v in named.items()}
        unused = {k for k, v in named.items() if v.grad is None}
        assert all(UNUSED_ICEFALL.search(k) for k in unused), f"tensors without gradient: {sorted(unused)[:8]}"
        if CFG["optimizer"] == "adamw": torch.nn.utils.clip_grad_norm_(params, CFG["grad_clip"])
        opt.step(); opt.zero_grad(set_to_none=True)
        unchanged = [k for k, v in named.items() if torch.equal(v.detach(), before[k]) and k not in unused]
        must = ["proj.lin.weight", "proj.lin.bias"] + [k for k in named if k.startswith("encoder_embed.")][:1] + \
               [next(k for k in named if k.startswith(f"encoder.encoders.{i}.") and k not in unused) for i in range(6)]
        assert not set(must) & set(unchanged), f"key tensors not updated: {sorted(set(must) & set(unchanged))}"
        assert len(unchanged) <= 0.01 * len(named), f"{len(unchanged)} of {len(named)} tensors not updated: {unchanged[:8]}"
        return (f"{type(opt).__name__}: {len(named) - len(unused) - len(unchanged)}/{len(named) - len(unused)} used student/projection tensors updated"
                + (f" (unchanged: {unchanged[:5]})" if unchanged else "") + f"; {len(unused)} unused icefall 'bypass_scale' leftovers have no gradient")
    def c1():
        f = out / "smoke_ckpt.pt"; atomic_torch_save(dict(student=st["S"].state_dict(), asr_proj=st["Pj"].state_dict()), f)
        ck = torch.load(f, map_location=dev, weights_only=False); S2, P2 = new_models(st["C"], dev)
        S2.load_state_dict(ck["student"], strict=True); P2.load_state_dict(ck["asr_proj"], strict=True); b = st["b"]; outs = []
        for S_, P_ in [(st["S"], st["Pj"]), (S2, P2)]:
            S_.eval(); P_.eval()
            with torch.no_grad(): e, _ = S_(b["fb"].to(dev), b["fb_lens"].to(dev)); outs.append(P_(e))
            S_.train(); P_.train()
        d = float((outs[0] - outs[1]).abs().max()); os.remove(f); assert d < 1e-5, d; return f"atomic save + strict reload, identical outputs (max diff {d:.1e})"
    def c2():
        """Recovery: state saved after one update; the next update is done, the state restored, the same update redone -> same result."""
        S, Pj, opt = st["S"], st["Pj"], st["opt"]; b = st["b"]; C = st["C"]; f = out / "smoke_resume.pt"
        atomic_torch_save(dict(s=S.state_dict(), p=Pj.state_dict(), o=opt.state_dict(), rng=torch.get_rng_state(), rngc=torch.cuda.get_rng_state()), f)
        def step():
            set_batch_count(S, 100000); l, _ = forward_loss(S, Pj, b, C, dev, amp); _sbw(l); opt.step(); opt.zero_grad(set_to_none=True)
            return float(l), float(Pj.lin.weight.detach().double().sum())
        a = step(); ck = torch.load(f, map_location=dev, weights_only=False)
        S.load_state_dict(ck["s"]); Pj.load_state_dict(ck["p"]); opt.load_state_dict(ck["o"]); torch.set_rng_state(ck["rng"].cpu()); torch.cuda.set_rng_state(ck["rngc"].cpu())
        b_ = step(); os.remove(f)
        assert abs(a[0] - b_[0]) < 1e-3 * abs(a[0]) and abs(a[1] - b_[1]) < 1e-3 * max(1, abs(a[1])), (a, b_)
        return f"replayed update after restore: loss {a[0]:.5f} vs {b_[0]:.5f}, projection checksum {a[1]:.4f} vs {b_[1]:.4f}"
    def m1():
        mb, peak = memory_probe(st["S"], st["Pj"], st["C"], dev, amp, CFG["max_batch_seconds"]); st["mem"] = (mb, peak)
        return f"worst-case batch fits: max_batch_seconds {mb:.0f} -> measured peak {peak:.2f} GiB"
    def t2():
        set_batch_count(st["S"], 100000); l, _ = forward_loss(st["S"], st["Pj"], st["b"], st["C"], dev, amp); _sbw(l); st["opt"].step(); st["opt"].zero_grad(set_to_none=True)
        assert torch.isfinite(l); return f"another training step OK, L1 {float(l):.4f} | " + ", ".join(f"{k} {v:.2f}" for k, v in gpu_mem().items())
    plan = [("D1", "dataset integrity (splits, nesting, ids, transcripts, decoded files, overlap)", d1), ("D2", "LibriSpeech audio loads", d2),
            ("W1", "WavLM-Large loads (frozen, eval, 24 layers, 1024-D)", w1), ("W2", "WavLM preprocessing / frame rule / layer / pairing / long utterances / precision", w2),
            ("W3", "teacher cache integrity and metadata", w3), ("W4", "cached features == freshly extracted", w4),
            ("A1", "teacher/student temporal alignment measured from the imported code", a1), ("S1", "Swoosh port == k2 algorithm", s1),
            ("Z1", "Zipformer-M source and configuration", z1), ("Z2", "80-D fbank frontend", z2), ("Z3", "student shapes, lengths, padding masks", z3),
            ("L1", "masked L1 == independent calculation; padding/unmatched frames ignored", l1), ("G1", "backpropagation", g1),
            ("G2", "non-zero finite gradients in every block and the projection", g2), ("G3", "every trainable tensor gets a gradient", g3),
            ("G4", "teacher frozen / not in the graph", g4), ("O1", "optimizer updates student and projection", o1),
            ("C1", "checkpoint save and strict reload", c1), ("C2", "checkpoint recovery (restore + replay)", c2), ("M1", "measured GPU memory of the worst-case batch", m1),
            ("T2", "another training step", t2)]
    for no, name, fn in plan:
        chk(no, name, fn, critical=(no != "G3"))
        if no in ("D2", "W1") and R[-1]["status"] == "FAIL": log("cannot continue without this"); break
    if "T" in st: del st["T"]
    gc.collect(); torch.cuda.empty_cache()
    crit_fail = [r["no"] for r in R if r["critical"] and r["status"] == "FAIL"]; ran = {r["no"] for r in R}; missing = [n for n, _, _ in plan if n not in ran]
    rep = dict(preset=preset, code_version=CODE_VERSION, code_fingerprint=code_fingerprint(), time=time.strftime("%Y-%m-%d %H:%M:%S"), checks=R,
               critical_failed=crit_fail + missing, all_critical_passed=not crit_fail and not missing,
               teacher_cache=str(st["C"]["TCtr"].src) if "C" in st else None, alignment_sha=sha_of([json.dumps(load_alignment()["measured"], sort_keys=True)]) if ALIGN_F().exists() else None,
               max_batch_seconds_probe=st.get("mem", (None, None))[0], swoosh=st.get("swoosh"), architecture=st.get("arch"))
    atomic_json(rep, SMOKE_F())
    log(f"\nSMOKE TEST: {sum(r['status'] == 'PASS' for r in R)}/{len(plan)} passed | critical failures: {rep['critical_failed'] or 'none'} -> {SMOKE_F()}"
        + ("" if rep["all_critical_passed"] else "\nTRAINING IS BLOCKED until every critical check passes"))
    return rep

def _gate(C):
    if not SMOKE_F().exists(): raise RuntimeError("smoke test has not been run: run the smoke-test cell first")
    r = json.load(open(SMOKE_F()))
    if not r.get("all_critical_passed"): raise RuntimeError(f"smoke test has critical failures {r.get('critical_failed')}: training blocked")
    if r.get("code_fingerprint") != code_fingerprint(): raise RuntimeError("the code changed since the smoke test: run the smoke-test cell again")
    if r.get("teacher_cache") != str(C["TCtr"].src): raise RuntimeError("the teacher cache differs from the one the smoke test checked: run the smoke test again")
    if r.get("alignment_sha") != sha_of([json.dumps(C["align"]["measured"], sort_keys=True)]): raise RuntimeError("alignment changed since the smoke test: run it again")

# ===========================================================================================
# training
# ===========================================================================================

# ####################################################################################################
# ---- Google Drive output store: the ONLY code in this cell that writes to Google Drive
# ####################################################################################################
import threading, random, hashlib, itertools
DRIVE_FILES = ("best.pt", "latest.pt", "history.csv", "final_report.json", "config.json")
RESUME_KEYS = {"sig", "student", "asr_proj", "opt", "scaler", "scheduler", "state", "rng_cpu", "rng_cuda", "rng_np", "rng_py"}

class SimulatedDisconnect(Exception):
    """Raised by the resume test to imitate a runtime that dies: nothing is saved at that moment."""

class DriveStore:
    """Writes ONLY into one folder (drive_dir(<run>)) and ONLY the five files in DRIVE_FILES (plus '<name>.tmp' while a write is in progress).
    * small files (config.json, history.csv, final_report.json): written synchronously, atomically (.tmp then rename)
    * checkpoints (latest.pt, best.pt): copied in a background thread from the local checkpoint (training does not wait),
      written to '<name>.tmp' first and renamed only when the copy is complete -> an interrupted upload never replaces a good file
    * with the Drive API (CFG['drive_api']): each upload is confirmed by comparing Drive's md5 with the local md5, and the copies that
      Drive moves to the trash when a checkpoint is replaced are deleted permanently (only files of THIS folder with THESE names),
      because trashed files keep using Drive quota for 30 days.
    fake=True: a local stand-in folder (used by the resume test; nothing goes to Drive)."""
    def __init__(self, folder, use_api=True, fake=False):
        self.dir, self.fake = Path(folder), fake
        if fake: assert_local(folder)
        else:
            if not os.path.isdir(DRIVE + "/MyDrive"): raise RuntimeError("Google Drive is not mounted")
            # the path as written must be inside MyDrive; its resolved location may be elsewhere on the Drive mount, because a folder
            # such as MT2KD can be a Drive SHORTCUT (resolved to /content/drive/.shortcut-targets-by-id/...)
            if not os.path.normpath(os.path.abspath(str(folder))).startswith(DRIVE + "/MyDrive/"): raise PermissionError(f"Drive output folder must be inside MyDrive: {folder}")
            if not os.path.realpath(str(folder)).startswith(os.path.realpath(DRIVE) + "/"): raise PermissionError(f"{folder} resolves outside Google Drive")
        self.dir.mkdir(parents=True, exist_ok=True)
        self.api = self.fid = None; self.api_note = "not used (stand-in folder)" if fake else "disabled (CFG['drive_api'] = False)"
        if use_api and not fake: self._init_api()
        self.cv = threading.Condition(); self.pending = {}; self.busy = None; self.errors = []; self.done = {}; self._ids = itertools.count()
        threading.Thread(target=self._worker, daemon=True).start()

    # ---------------------------------------------------------------- guards / API
    def _p(self, name):
        base = name[:-4] if name.endswith(".tmp") else name
        if base not in DRIVE_FILES: raise PermissionError(f"'{name}' is not one of the files allowed on Drive: {DRIVE_FILES}")
        return self.dir / name
    def _init_api(self):
        try:
            from google.colab import auth; auth.authenticate_user()
            from googleapiclient.discovery import build
            self.api = build("drive", "v3", cache_discovery=False)
            self.api_note = "ON: uploads confirmed by md5; trashed copies of this folder's checkpoints are deleted permanently"
        except Exception as e:
            self.api = None; self.api_note = (f"UNAVAILABLE ({type(e).__name__}: {e}): uploads are not confirmed and replaced checkpoints stay in the Drive "
                                              f"trash (empty it to recover the space)")
    def _folder_id(self):
        if self.fid or self.api is None: return self.fid
        rel = os.path.relpath(os.path.normpath(os.path.abspath(str(self.dir))), DRIVE + "/MyDrive").split(os.sep); parent = "root"
        try:
            for part in rel:                                  # walk the path as written; follow Drive shortcuts to their target folder
                q = (f"name = '{part}' and '{parent}' in parents and trashed = false and "
                     f"(mimeType = 'application/vnd.google-apps.folder' or mimeType = 'application/vnd.google-apps.shortcut')")
                r = self.api.files().list(q=q, fields="files(id,mimeType,shortcutDetails)", pageSize=10, supportsAllDrives=True,
                                          includeItemsFromAllDrives=True).execute().get("files", [])
                if not r: return None                         # folder not synced to Drive yet: retried later
                x = r[0]; parent = x["shortcutDetails"]["targetId"] if x["mimeType"].endswith("shortcut") else x["id"]
            self.fid = parent
        except Exception: return None
        return self.fid

    # ---------------------------------------------------------------- storage information
    def quota(self):
        if self.api:
            try:
                q = self.api.about().get(fields="storageQuota").execute()["storageQuota"]; lim = int(q.get("limit", 0)); use = int(q.get("usage", 0))
                return dict(limit=lim or None, usage=use, trash=int(q.get("usageInDriveTrash", 0)), free=(lim - use) if lim else None, source="Drive API")
            except Exception: pass
        du = shutil.disk_usage(self.dir if self.fake else DRIVE + "/MyDrive")
        return dict(limit=du.total, usage=du.used, trash=None, free=du.free, source="mounted file system (approximate)")
    def files(self): return {f.name: f.stat().st_size for f in sorted(self.dir.iterdir()) if f.is_file()}
    def status(self, expected_bytes=None):
        q = self.quota(); fl = self.files(); g = lambda b: "?" if b is None else f"{b / 2**30:.2f} GiB"
        log(f"  Drive output folder : {self.dir}")
        log(f"  folder size now     : {g(sum(fl.values()))} | files: " + (", ".join(f"{k} ({v / 2**20:.1f} MiB)" for k, v in fl.items()) or "none"))
        log(f"  Drive storage       : free {g(q['free'])} of {g(q['limit'])} (used {g(q['usage'])}" + (f", in trash {g(q['trash'])}" if q["trash"] is not None else "")
            + f") [{q['source']}]")
        log(f"  Drive API           : {self.api_note}")
        if expected_bytes: log(f"  this run will keep  : about {g(expected_bytes)} in this folder (best.pt + latest.pt + small files)")
        unexpected = [k for k in fl if (k[:-4] if k.endswith(".tmp") else k) not in DRIVE_FILES]
        if unexpected: log(f"  NOTE: files not written by this code are in the folder (left untouched): {unexpected}")
        return dict(quota=q, files=fl)
    def check_space(self, need_bytes, what):
        """Refuses to start writing large files when the free Drive space is below what the next writes need."""
        q = self.quota()
        if q["free"] is not None and q["free"] < need_bytes:
            raise RuntimeError(f"Google Drive has {q['free'] / 2**30:.2f} GiB free; {what} needs about {need_bytes / 2**30:.2f} GiB. Nothing large was written. "
                               f"Free at least {(need_bytes - q['free']) / 2**30:.2f} GiB more on Drive (deleted files only free space once the Drive trash is "
                               f"emptied) and re-run the cell.")
        free = "?" if q["free"] is None else f"{q['free'] / 2**30:.2f}"
        log(f"  Drive space check OK for {what}: need ~{need_bytes / 2**30:.2f} GiB, free {free} GiB")

    # ---------------------------------------------------------------- writes
    def write_text(self, name, text):
        tmp, dst = self._p(name + ".tmp"), self._p(name)
        with open(tmp, "w") as fh: fh.write(text); fh.flush(); os.fsync(fh.fileno())
        os.replace(tmp, dst)
    def write_json(self, name, obj): self.write_text(name, json.dumps(obj, indent=1, default=str))
    def push(self, local_file, name):
        """Queues an upload of a local checkpoint file. The file is snapshotted NOW (hard link), so later local saves cannot change what is
        uploaded; a newer request for the same name replaces a queued (not yet started) one."""
        self._p(name); snap = f"{local_file}.upload{next(self._ids)}"
        try: os.link(local_file, snap)
        except OSError: shutil.copyfile(local_file, snap)
        with self.cv:
            if self.errors: os.remove(snap); raise RuntimeError(f"a Google Drive upload failed: {self.errors[-1]} (training stopped; local state is saved)")
            old = self.pending.get(name); self.pending[name] = snap; self.cv.notify_all()
        if old and os.path.exists(old): os.remove(old)
    def _worker(self):
        while True:
            with self.cv:
                while not self.pending: self.cv.wait()
                name = next(iter(self.pending)); src = self.pending.pop(name); self.busy = name
            try: self._upload(src, name)
            except Exception as e: self.errors.append(f"{name}: {type(e).__name__}: {e}"); log(f"  GOOGLE DRIVE UPLOAD FAILED ({name}): {type(e).__name__}: {e}")
            finally:
                try: os.remove(src)
                except OSError: pass
                with self.cv: self.busy = None; self.cv.notify_all()
    def _upload(self, src, name):
        t0 = time.time(); tmp, dst = self._p(name + ".tmp"), self._p(name); h = hashlib.md5(); n = 0
        with open(src, "rb") as fi, open(tmp, "wb") as fo:          # the open handle keeps a consistent snapshot even if the local file is replaced meanwhile
            for b in iter(lambda: fi.read(1 << 24), b""): fo.write(b); h.update(b); n += len(b)
            fo.flush(); os.fsync(fo.fileno())
        os.replace(tmp, dst); md5 = h.hexdigest()
        self.done[name] = dict(bytes=n, md5=md5, written=time.strftime("%Y-%m-%d %H:%M:%S"), confirmed_by_drive_api=None)
        log(f"  Drive: {name} written ({n / 2**20:.0f} MiB in {time.time() - t0:.0f} s)" + ("; waiting for Drive to confirm the upload" if self.api else ""))
        if self.api:
            self.done[name]["confirmed_by_drive_api"] = self._confirm(name, md5); self.purge_trash()
    def _confirm(self, name, md5, timeout=1800):
        t0 = time.time()
        while time.time() - t0 < timeout:
            with self.cv:
                if name in self.pending: return "superseded by a newer upload"
            fid = self._folder_id()
            if fid:
                try:
                    r = self.api.files().list(q=f"name = '{name}' and '{fid}' in parents and trashed = false", fields="files(id,md5Checksum)").execute().get("files", [])
                    if any(x.get("md5Checksum") == md5 for x in r): log(f"  Drive: {name} upload CONFIRMED (md5 {md5[:10]} matches)"); return True
                except Exception: pass
            time.sleep(15)
        log(f"  Drive: {name} upload not confirmed within {timeout // 60} min (it may still be syncing)"); return False
    def purge_trash(self):
        """Deletes permanently the TRASHED copies of this folder's own files (replaced checkpoints); nothing else is touched."""
        fid = self._folder_id()
        if not fid: return 0
        freed = 0
        try:
            for x in self.api.files().list(q=f"'{fid}' in parents and trashed = true", fields="files(id,name,size)", pageSize=200).execute().get("files", []):
                base = x["name"][:-4] if x["name"].endswith(".tmp") else x["name"]
                if base in DRIVE_FILES: self.api.files().delete(fileId=x["id"]).execute(); freed += int(x.get("size", 0))
        except Exception as e: log(f"  (could not clean this folder's trashed copies: {type(e).__name__}: {e})")
        if freed: log(f"  Drive: {freed / 2**20:.0f} MiB of replaced copies removed from the trash")
        return freed
    def wait(self, timeout=None):
        """Blocks until every queued upload is written (and confirmed, with the API)."""
        t0 = time.time()
        with self.cv:
            while self.pending or self.busy:
                if timeout and time.time() - t0 > timeout: return False
                self.cv.wait(5)
        if self.api: self.purge_trash()
        if self.errors: raise RuntimeError(f"Google Drive upload failed: {self.errors[-1]}")
        return True

    # ---------------------------------------------------------------- reading back
    def candidates(self, name): return [f for f in (self._p(name), self._p(name + ".tmp")) if f.exists()]
    def promote_tmp(self, name):
        """A complete '<name>.tmp' whose rename was interrupted becomes '<name>'."""
        os.replace(self._p(name + ".tmp"), self._p(name)); return self._p(name)
    def clean_stale_tmp(self):
        for f in self.dir.glob("*.tmp"):
            if f.name[:-4] in DRIVE_FILES: f.unlink(); log(f"  removed an incomplete upload left by an interrupted run: {f.name}")
    def best_info(self): return best_meta(self._p("best.pt")) if self._p("best.pt").exists() else None

def drive_need_bytes(n_par, drive):
    """Drive space the run needs beyond what its own files already use: best.pt (fp32 weights) + latest.pt (weights + 2 AdamW moments),
    each written as a temporary copy before the old file is replaced (peak = 2 x each)."""
    best_b, latest_b = 4.05 * n_par, 20.0 * n_par          # latest.pt: weights + optimizer state (ScaledAdam keeps more per weight than AdamW)
    have = sum(v for k, v in drive.files().items() if k in ("best.pt", "latest.pt"))
    return max(0.0, 2 * (best_b + latest_b) - have) + 50e6, best_b + latest_b

def best_meta(f):
    import torch
    try: ck = torch.load(f, map_location="cpu", weights_only=False, mmap=True)
    except Exception:
        try: ck = torch.load(f, map_location="cpu", weights_only=False)
        except Exception as e: return dict(error=f"{type(e).__name__}: {e}")
    r = dict(val_l1=float(ck["val"]["val_l1"]), epoch=int(ck["epoch"]), update=int(ck["update"]), sig=ck.get("sig")); del ck
    return r

def discard_summary():
    """Every utterance excluded from the data, with its reason (inventory checks + full decode)."""
    m = paths()["meta"]; out = {}
    for split in ["train-clean-100", CFG["val_split"]]:
        d = {}; f = m / f"inventory_v2_{split}_dropped.csv"
        if f.exists() and f.stat().st_size > 30:
            x = pd.read_csv(f, dtype=str); d["invalid_files"] = {u: r for u, r in zip(x.uid, x.reason)}
        f = m / f"decode_failures_v2_{split}.csv"
        if f.exists():
            x = pd.read_csv(f, dtype=str); d["failed_full_decode"] = {u: r for u, r in zip(x.uid, x.reason)}
        out[split] = d or "none excluded"
    return out

def _load_resumable(f):
    import torch
    try: ck = torch.load(f, map_location="cpu", weights_only=False)
    except Exception as e: return None, f"unreadable ({type(e).__name__}: {str(e)[:120]})"
    if not isinstance(ck, dict) or not RESUME_KEYS <= set(ck): return None, "not a complete resumable checkpoint"
    return ck, ""

def _position(S): return (S["update"], S["epoch"], len(S["seen"]))

# ####################################################################################################
# ---- training (resumable from local disk or Google Drive)
# ####################################################################################################
def train(preset="pilot", C=None, P=None, drive=None, overrides=None, interrupt_after_update=None, probe=True):
    """KD training. Resumes automatically from the most advanced VALID checkpoint among: local latest, Drive latest.pt, Drive latest.pt.tmp."""
    import torch
    C = C or context(preset); _gate(C); P = P or paths(preset); ensure_dirs(P); dev = torch.device("cuda"); amp = amp_dtype()
    cfg = dict(CFG, **PRESETS[preset]); cfg.update(overrides or {})
    LATEST, BEST = P["latest"] / "checkpoint.pt", P["best"] / "best.pt"; N = len(C["tr"]); lens = C["tr"].n_samples.values
    student, proj = new_models(C, dev)
    params = list(student.parameters()) + list(proj.parameters())
    opt, sch = make_optimizer(student, proj, cfg)
    scaler = torch.amp.GradScaler("cuda", init_scale=2.0 ** 12, enabled=amp == torch.float16)
    sig_keys = ["seed", "optimizer", "base_lr", "lr_batches", "lr_epochs", "eden_warmup_batches", "eden_warmup_start", "clipping_scale",
                "lr", "weight_decay", "warmup_updates", "min_lr_frac", "grad_clip", "accum_steps", "align", "target_norm", "init_bias", "max_batch_utts"]
    # (max_epochs is NOT in the signature: raising it later continues the same run instead of refusing its checkpoints)
    SIG = sha_of([json.dumps({k: cfg[k] for k in sig_keys}, sort_keys=True), C["manifest_sha"], C["dev_sha"], str(C["TCtr"].src), C["offset"], CODE_VERSION])
    S = dict(epoch=0, batch=0, update=0, best=None, es_counter=0, train_hist=[], val_hist=[], elapsed=0.0, skips=0, finished=None, seen=[], plan_remaining=None,
             ep_sum=0.0, ep_fr=0.0, max_batch_seconds=None, sessions=0, resumes=[])

    # ---- 1) look for the most advanced valid checkpoint (local disk first, then Drive, including an upload whose rename was interrupted)
    for f in list(P["ckpt"].rglob("*.upload*")): f.unlink()                 # snapshots left by an interrupted upload
    cands = [("local disk", LATEST)] + ([("Google Drive", f) for f in drive.candidates("latest.pt")] if drive else []); chosen = None
    for src, f in cands:
        if not Path(f).exists(): continue
        ck, why = _load_resumable(f)
        if ck is None: log(f"  ignored {src} file {f}: {why} (e.g. an interrupted save or upload)"); continue
        if ck["sig"] != SIG and src == "local disk":
            dst = P["ckpt"].parent / f"{P['ckpt'].name}_other_config_{time.strftime('%Y%m%d_%H%M%S')}"; del ck; os.replace(P["ckpt"], dst); ensure_dirs(P)
            log(f"  local checkpoints of a DIFFERENT configuration moved aside (local disk): {dst}"); continue
        if ck["sig"] != SIG:
            raise RuntimeError(f"{f} ({src}) is a checkpoint of a DIFFERENT configuration / data / code version (sig {ck['sig']}, this run {SIG}). "
                               f"It is left untouched. Move it out of the folder yourself to start a new run, or restore the settings it was made with.")
        k = _position(ck["state"]); log(f"  valid checkpoint on {src}: {Path(f).name}, epoch {ck['state']['epoch'] + 1}, batch {ck['state']['batch']}, global step {k[0]}")
        if chosen is None or k > chosen[2]: chosen = (src, f, k, ck)
        else: del ck
    if chosen:
        src, f, k, ck = chosen
        if src == "Google Drive" and str(f).endswith(".tmp"): f = drive.promote_tmp("latest.pt"); log("  (the newest Drive checkpoint was a complete upload whose rename was interrupted: renamed)")
        student.load_state_dict(ck["student"], strict=True); proj.load_state_dict(ck["asr_proj"], strict=True); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        S = ck["state"]; torch.set_rng_state(ck["rng_cpu"].cpu()); torch.cuda.set_rng_state(ck["rng_cuda"].cpu()); np.random.set_state(ck["rng_np"]); random.setstate(ck["rng_py"])
        log("\n" + "#" * 70 + "\n#  RESUMING FROM CHECKPOINT\n" + "#" * 70)
        log(f"  source {src}: {f}\n  epoch {S['epoch'] + 1}/{cfg['max_epochs']} | batch position {S['batch']} ({len(S['seen'])}/{N} utterances of this epoch done) | "
            f"global optimizer step {S['update']} | lr schedule at step {ck['scheduler']['update']} (lr {ck['scheduler']['lr']:.3e}) | best val L1 "
            f"{S['best']['val_l1'] if S['best'] else None} | early-stop counter {S['es_counter']}/{cfg['patience']}")
        S.setdefault("resumes", []).append(dict(source=src, update=S["update"], epoch=S["epoch"], batch=S["batch"], seen=len(S["seen"]), time=time.strftime("%Y-%m-%d %H:%M:%S")))
        if src == "Google Drive":
            shutil.copyfile(f, str(LATEST) + ".part"); os.replace(str(LATEST) + ".part", LATEST)       # local copy (later local saves start from here)
        del ck
        if cfg["max_batch_seconds"] < S["max_batch_seconds"]:
            log(f"  max_batch_seconds lowered {S['max_batch_seconds']:.0f} -> {cfg['max_batch_seconds']:.0f}: the REMAINING utterances of this epoch are re-batched")
            S["max_batch_seconds"] = cfg["max_batch_seconds"]; S["plan_remaining"] = sorted(set(range(N)) - set(S["seen"])); S["batch"] = 0
    else:
        log("\n" + "#" * 70 + "\n#  STARTING NEW TRAINING RUN\n" + "#" * 70 + "\n  (no valid checkpoint on local disk" + (" or in " + str(drive.dir) if drive else "") + ")")
    if drive: drive.clean_stale_tmp()

    # ---- 2) the best checkpoint is never replaced by a worse model. A best.pt (Drive or local) better than the resumed state's best
    #         (saved during work lost in the interruption) becomes a floor: best.pt is only replaced by a model that beats it.
    lb = best_meta(BEST) if BEST.exists() else None
    if lb and lb.get("sig") != SIG:
        dst = BEST.with_name(f"best_other_config_{time.strftime('%Y%m%d_%H%M%S')}.pt"); os.replace(BEST, dst); lb = None
        log(f"  local best.pt of a different configuration moved aside: {dst}")
    db = drive.best_info() if drive else None
    if db and "error" in db: log(f"  Drive best.pt is unreadable ({db['error']}): it will be replaced by the next best model"); db = None
    if db and db.get("sig") != SIG:
        raise RuntimeError(f"{drive.dir / 'best.pt'} is not a best checkpoint of this configuration ({db}); left untouched. Move it away to start a new run.")
    floor = None
    for src_, m in (("Google Drive", db), ("local disk", lb)):
        if m and (S["best"] is None or m["val_l1"] < S["best"]["val_l1"]) and (floor is None or m["val_l1"] < floor["val_l1"]): floor = dict(m, source=src_)
    if floor: log(f"  existing best.pt on {floor['source']} (val L1 {floor['val_l1']:.5f}, epoch {floor['epoch']}) is newer than the resumed state: "
                  f"it is only replaced by a model with a lower validation loss")
    if drive:
        if db and (lb is None or lb["val_l1"] > db["val_l1"]): shutil.copyfile(drive.dir / "best.pt", str(BEST) + ".part"); os.replace(str(BEST) + ".part", BEST)
        elif lb and (db is None or lb["val_l1"] < db["val_l1"]): drive.push(BEST, "best.pt"); log("  local best.pt is newer than the Drive copy (upload was interrupted): re-uploading")
    S["sessions"] += 1

    # ---- 3) Drive space (estimated from the parameter count: fp32 weights; AdamW keeps two more fp32 tensors per weight)
    if drive and not drive.fake: drive.check_space(drive_need_bytes(sum(p.numel() for p in params), drive)[0], "best.pt + latest.pt (each replaced via a temporary copy)")
    if S["max_batch_seconds"] is None:
        S["max_batch_seconds"] = memory_probe(student, proj, C, dev, amp, cfg["max_batch_seconds"])[0] if probe else cfg["max_batch_seconds"]
    MBS = S["max_batch_seconds"]

    def lr_at(update, prog):
        """linear warm-up over the first warmup_updates, then cosine over training PROGRESS (fraction of max_epochs x utterances)."""
        w = cfg["lr"] * min(1.0, (update + 1) / cfg["warmup_updates"])
        cosv = cfg["min_lr_frac"] + (1 - cfg["min_lr_frac"]) * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
        return min(w, cfg["lr"] * cosv) if update < cfg["warmup_updates"] else cfg["lr"] * cosv
    def lr_now(update):
        """learning rate for the next optimizer step (global step `update` + 1), also written into the optimizer.
        ScaledAdam/Eden: as icefall (scheduler.step_epoch(epoch - 1) at each epoch start, step_batch(batch_idx_train) before each step)."""
        if sch is not None:
            sch.step_epoch(S["epoch"]); sch.step_batch(update + 1); return sch.get_last_lr()[0]
        lr = lr_at(update, progress())
        for g in opt.param_groups: g["lr"] = lr
        return lr
    progress = lambda: (S["epoch"] + len(S["seen"]) / N) / cfg["max_epochs"]
    tr = C["tr"]; dvf = C["dv"]
    run_cfg = dict(code_version=CODE_VERSION, sig=SIG, preset=preset, seed=cfg["seed"], amp=str(amp), updated=time.strftime("%Y-%m-%d %H:%M:%S"),
                   data=dict(source="LibriSpeech train-clean-100 (official)", train_subset=PRESETS[preset]["train_subset"], target_seconds=3600 * CFG["subsets"].get(PRESETS[preset]["train_subset"], 0),
                             train_utterances=N, train_seconds=round(float(tr.duration_s.sum()), 2), train_hours=round(float(tr.duration_s.sum() / 3600), 4),
                             train_speakers=int(tr.speaker.nunique()), train_manifest_sha=C["manifest_sha"],
                             selection_rule="valid utterances sorted by id, permuted with numpy default_rng(seed); shortest prefix reaching the target duration",
                             validation=dict(split=CFG["val_split"], utterances=len(dvf), hours=round(float(dvf.duration_s.sum() / 3600), 4), manifest_sha=C["dev_sha"]),
                             test="test-clean is not used in Stage 1", discarded=discard_summary(), train_uids=sorted(tr.uid.tolist())),
                   teacher=dict(model=C["teacher_meta"].get("model"), commit=C["teacher_meta"].get("commit"), layer=C["teacher_meta"].get("layer"),
                                representation=f"{C['teacher_meta'].get('layer')}, 1024-D @ 50 Hz; frames (2j, 2j+1) concatenated -> 2048-D @ 25 Hz", frozen=True,
                                preprocessing=C["teacher_meta"].get("normalize"), dtype=C["teacher_meta"].get("dtype"),
                                long_utterances="encoded whole (no window, no chunking, no padding)", cache="local disk only (/content), regenerated after a runtime recycle"),
                   student=dict(model="Zipformer-M (icefall zipformer.py)", input="80-D log-mel fbank, 10 ms", frontend="Conv2dSubsampling (icefall)", zip_args=ZIP_ARGS,
                                output="512-D @ 25 Hz", projection=f"Linear 512 -> {C['teacher_meta']['dim']}", params_student=sum(p.numel() for p in student.parameters()),
                                params_projection=sum(p.numel() for p in proj.parameters()), swoosh="pure-PyTorch port of k2's Swoosh (k2 not required), verified numerically"),
                   alignment=dict(rule=f"student frame j <-> teacher frame j+{C['offset']} (measured frame centres)", residual_ms=C["align"]["residual_ms"], offset=C["offset"]),
                   loss=f"masked L1 between projected student and teacher features (mean over {C['teacher_meta']['dim']} dims and valid aligned frames)",
                   training={k: cfg[k] for k in ["lr", "weight_decay", "warmup_updates", "min_lr_frac", "grad_clip", "accum_steps", "max_epochs", "patience", "min_delta",
                                                  "max_batch_utts", "local_ckpt_every_updates", "drive_ckpt_every_updates", "ckpt_every_minutes", "init_bias", "target_norm"]}
                            | ({k: cfg[k] for k in ["optimizer", "base_lr", "lr_batches", "lr_epochs", "eden_warmup_batches", "eden_warmup_start", "clipping_scale"]}
                               if cfg["optimizer"] == "scaled_adam" else dict(optimizer="AdamW", schedule="linear warm-up, then cosine over progress"))
                            | dict(max_batch_seconds_effective=MBS),
                   drive_files=dict(folder=str(drive.dir) if drive else None, files=list(DRIVE_FILES)))
    atomic_json(run_cfg, P["logs"] / "run_config.json")
    if drive: drive.write_json("config.json", run_cfg)
    log(f"  train {N:,} utts / {tr.duration_s.sum():.0f} s / {run_cfg['data']['train_speakers']} speakers | val {len(dvf):,} utts / {dvf.duration_s.sum() / 3600:.2f} h | "
        f"max {cfg['max_epochs']} epochs | " + (f"ScaledAdam + Eden (base lr {cfg['base_lr']}, lr_batches {cfg['lr_batches']}, lr_epochs {cfg['lr_epochs']})"
        if cfg["optimizer"] == "scaled_adam" else f"AdamW lr {cfg['lr']}") + f" | {str(amp).split('.')[-1]}{' + GradScaler' if scaler.is_enabled() else ''} | batches <= {MBS:.0f} s "
        f"| checkpoints: local every {cfg['local_ckpt_every_updates']} steps / {cfg['ckpt_every_minutes']:.0f} min, Drive every {cfg['drive_ckpt_every_updates']} steps, both every epoch")

    t_ref = [time.time()]; last_local = [time.time()]; t_session = time.time(); budget = cfg.get("max_session_hours")
    def sched(): return dict(kind=("Eden: a pure function of (global step, epoch), both restored from 'state'" if sch is not None else
                                   "warm-up + cosine on progress: a pure function of (global step, progress), both restored from 'state'"),
                             update=S["update"], progress=progress(), lr=lr_now(S["update"]), eden=sch.state_dict() if sch is not None else None)
    def save_latest(to_drive=False):
        S["elapsed"] += time.time() - t_ref[0]; t_ref[0] = time.time()
        atomic_torch_save(dict(sig=SIG, student=student.state_dict(), asr_proj=proj.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(), scheduler=sched(),
                               state=S, rng_cpu=torch.get_rng_state(), rng_cuda=torch.cuda.get_rng_state(), rng_np=np.random.get_state(), rng_py=random.getstate(),
                               zip_args=ZIP_ARGS, code_version=CODE_VERSION), LATEST)
        pd.DataFrame(S["train_hist"]).to_csv(P["logs"] / "train_history.csv", index=False); last_local[0] = time.time()
        if to_drive and drive: drive.push(LATEST, "latest.pt")
    def write_history():
        h = pd.DataFrame([dict(epoch=v["epoch"], global_step=v["update"], train_loss=v["train_l1"], val_loss=v["val_l1"], lr=v["lr"], best=v["best"]) for v in S["val_hist"]],
                         columns=["epoch", "global_step", "train_loss", "val_loss", "lr", "best"])
        h.to_csv(P["logs"] / "history.csv", index=False)
        if drive: drive.write_text("history.csv", h.to_csv(index=False))

    student.train(); proj.train(); win = []; pending = []; stop_reason = None
    try:
        while S["epoch"] < cfg["max_epochs"] and not S["finished"]:
            ep = S["epoch"]; batches = make_batches(lens, ep, cfg["seed"], True, MBS, cfg["max_batch_utts"], idx=S["plan_remaining"])
            k0 = S["batch"]; t_ep = time.time(); torch.cuda.reset_peak_memory_stats()
            nb = [len(x) for x in batches]
            log(f"epoch {ep + 1}: {len(batches)} batches, {min(nb)}-{max(nb)} utterances per batch (mean {np.mean(nb):.1f})"
                + (f" | continuing at batch {k0}/{len(batches)} ({len(S['seen'])}/{N} utterances already seen)" if k0 or S["plan_remaining"] is not None else ""))
            for k, b in enumerate(loader(C["tr"], C["TCtr"], batches[k0:], cfg["num_workers"], seed=cfg["seed"] + ep), start=k0):
                set_batch_count(student, S["update"] * MBS * cfg["accum_steps"] / cfg["ref_duration"])
                l1, i = forward_loss(student, proj, b, C, dev, amp)
                if not torch.isfinite(l1):
                    S["skips"] += 1; log(f"  non-finite loss at step {S['update']} (batch skipped, {S['skips']} in a row)"); opt.zero_grad(set_to_none=True)
                    for kk, _, _ in pending: S["seen"] += batches[kk]
                    pending = []
                    if S["skips"] > cfg["max_consecutive_skips"]: save_latest(True); raise FloatingPointError("too many non-finite losses; state saved")
                    S["seen"] += batches[k]; S["batch"] = k + 1; continue
                scaler.scale(l1 / cfg["accum_steps"]).backward(); win.append((float(l1), i["base_mean"], i["cos"], i["frames"])); pending.append((k, float(l1), i["frames"]))
                if (k + 1) % cfg["accum_steps"] and k + 1 < len(batches): continue
                lr = lr_now(S["update"])
                # ScaledAdam clips internally (clipping_scale x median recent norm): here the norm is only measured (max_norm = inf)
                scaler.unscale_(opt); gn = float(torch.nn.utils.clip_grad_norm_(params, cfg["grad_clip"] if sch is None else float("inf")))
                if math.isfinite(gn): scaler.step(opt); S["skips"] = 0
                else:
                    S["skips"] += 1; log(f"  non-finite gradient at step {S['update']} (update skipped, {S['skips']} in a row)")
                    if scaler.is_enabled(): scaler.step(opt)
                    if S["skips"] > cfg["max_consecutive_skips"]: save_latest(True); raise FloatingPointError("too many non-finite gradients; state saved")
                scaler.update(); opt.zero_grad(set_to_none=True); S["update"] += 1
                for kk, lv, fr in pending: S["seen"] += batches[kk]; S["ep_sum"] += lv * fr; S["ep_fr"] += fr
                pending = []; S["batch"] = k + 1; u = S["update"]
                if u % cfg["log_every"] == 0:
                    w = np.array(win); fr = w[:, 3]; win = []
                    rec = dict(update=u, epoch=ep + 1, batch=k + 1, progress=round(progress(), 5), l1=float(np.average(w[:, 0], weights=fr)),
                               base_mean=float(np.average(w[:, 1], weights=fr)), cos=float(np.average(w[:, 2], weights=fr)), grad_norm=gn, lr=lr,
                               loss_scale=scaler.get_scale() if scaler.is_enabled() else 1.0, **{a: round(v, 3) for a, v in gpu_mem().items()})
                    S["train_hist"].append(rec); el = time.time() - t_ep
                    log(f"[ep {ep + 1}/{cfg['max_epochs']} | b {k + 1}/{len(batches)} | step {u}] L1 {rec['l1']:.4f} | mean-baseline {rec['base_mean']:.4f} "
                        f"(ratio {rec['l1'] / rec['base_mean']:.3f}) | cos {rec['cos']:.3f} | |g| {gn:.2f} | lr {lr:.2e} | GPU peak {rec.get('gpu_peak_gib', 0):.1f} GiB | "
                        f"epoch ETA {el / (k + 1 - k0) * (len(batches) - k - 1) / 60:.1f} min")
                if drive and u % cfg["drive_ckpt_every_updates"] == 0: save_latest(True); log(f"  checkpoint saved at step {u}: local disk + Google Drive (upload in background)")
                elif u % cfg["local_ckpt_every_updates"] == 0 or time.time() - last_local[0] > 60 * cfg["ckpt_every_minutes"]: save_latest(); log(f"  checkpoint saved at step {u} (local disk)")
                if interrupt_after_update is not None and u == interrupt_after_update: raise SimulatedDisconnect(f"simulated runtime loss after step {u}")
                if budget and time.time() - t_session > 3600 * budget:
                    save_latest(True); write_history(); stop_reason = f"session time budget of {budget} h reached"; break
            if stop_reason: break
            assert len(set(S["seen"])) == N == len(S["seen"]), f"epoch {ep + 1} saw {len(set(S['seen']))} distinct of {N} utterances ({len(S['seen'])} total)"
            v = evaluate(student, proj, C, dev, amp); tr_l1 = S["ep_sum"] / max(S["ep_fr"], 1)
            v.update(epoch=ep + 1, update=S["update"], train_l1=tr_l1, lr=lr_now(S["update"]), epoch_minutes=(time.time() - t_ep) / 60,
                     **{a: round(x, 3) for a, x in gpu_mem().items()})
            improved = S["best"] is None or v["val_l1"] < S["best"]["val_l1"] - cfg["min_delta"]
            if improved and floor and v["val_l1"] >= floor["val_l1"]:
                S["best"] = {k: floor[k] for k in ("val_l1", "epoch", "update")}; S["es_counter"] = 0; improved = False
                log(f"  (val L1 {v['val_l1']:.5f} is not lower than the existing best.pt ({floor['val_l1']:.5f}): best.pt kept)")
                floor = None
            elif improved:
                S["best"] = dict(val_l1=v["val_l1"], epoch=ep + 1, update=S["update"], val_cos=v["val_cos"], val_ratio_mean=v["val_ratio_mean"]); S["es_counter"] = 0; floor = None
                atomic_torch_save(dict(student=student.state_dict(), asr_proj=proj.state_dict(), zip_args=ZIP_ARGS, cfg=cfg, val=v, epoch=ep + 1, update=S["update"],
                                       teacher=C["teacher_meta"], manifest_sha=C["manifest_sha"], alignment=run_cfg["alignment"], seed=cfg["seed"], sig=SIG,
                                       code_version=CODE_VERSION), BEST)
                if drive: drive.push(BEST, "best.pt")
            else: S["es_counter"] += 1
            vh = S["val_hist"]; flags = []
            if len(vh) >= 2 and v["val_l1"] > vh[-1]["val_l1"] > vh[-2]["val_l1"] and tr_l1 < vh[-1]["train_l1"] < vh[-2]["train_l1"]:
                flags.append("validation rose for 2 checks while training fell")
            if v["val_l1"] / tr_l1 > cfg["overfit_gap_ratio"]: flags.append(f"val/train L1 = {v['val_l1'] / tr_l1:.2f} > {cfg['overfit_gap_ratio']}")
            v["overfit_flags"] = "; ".join(flags); v["best"] = improved; v["es_counter"] = S["es_counter"]; vh.append(v)
            log(f"=== epoch {ep + 1}: train L1 {tr_l1:.4f} | VAL L1 {v['val_l1']:.4f} (mean-baseline {v['val_base_mean_l1']:.4f}, median-baseline "
                f"{v['val_base_median_l1']:.4f}) | cos {v['val_cos']:.3f} | step {S['update']} | {v['epoch_minutes']:.1f} min"
                + (" | NEW BEST (saved)" if improved else f" | no improvement ({S['es_counter']}/{cfg['patience']})")
                + (f" | POSSIBLE OVERFITTING: {v['overfit_flags']}" if flags else "") + " ===")
            S["epoch"] = ep + 1; S["batch"] = 0; S["seen"] = []; S["plan_remaining"] = None; S["ep_sum"] = S["ep_fr"] = 0.0
            if S["es_counter"] >= cfg["patience"]: S["finished"] = f"early stopping after epoch {ep + 1} (best epoch {S['best']['epoch']})"
            elif S["epoch"] >= cfg["max_epochs"]: S["finished"] = f"max_epochs {cfg['max_epochs']} reached"
            save_latest(True); write_history(); plots(P)
    except oom_error():
        opt.zero_grad(set_to_none=True); torch.cuda.empty_cache(); save_latest(True); write_history()
        if drive: drive.wait()
        raise RuntimeError(f"CUDA out of memory. State saved at step {S['update']}. Set CFG['max_batch_seconds'] lower (effective {MBS:.0f}) and re-run: "
                           f"the rest of the epoch is re-batched.")
    except KeyboardInterrupt:
        save_latest(True); write_history(); log("interrupted: state saved locally; waiting for the Google Drive upload ...")
        if drive: drive.wait()
        make_report(preset, C, P, drive, verify_best=False); raise
    if S["finished"]:
        ck = torch.load(BEST, map_location=dev, weights_only=False); student.load_state_dict(ck["student"]); proj.load_state_dict(ck["asr_proj"])
        log(f"finished: {S['finished']}. Best checkpoint (epoch {ck['epoch']}) restored; kept at {BEST}" + (f" and {drive.dir / 'best.pt'}" if drive else "")); del ck
    elif stop_reason: log(f"{stop_reason}: state saved at step {S['update']}. Re-run the cell to continue.")
    del student, proj, opt, params; torch.cuda.empty_cache()
    if drive: log("waiting for the Google Drive uploads to finish ..."); drive.wait()
    return make_report(preset, C, P, drive, verify_best=bool(S["finished"]))

# ####################################################################################################
# ---- resume test: deliberately interrupted short runs must continue from the correct epoch, batch and optimizer step
# ####################################################################################################
def resume_test(C, n_train=24, n_dev=8):
    """Short runs on 24 training / 8 validation utterances, with a LOCAL stand-in folder for Drive (nothing is written to Google Drive):
       reference : uninterrupted run
       run A     : killed after step c1 without saving (as a dying runtime would) ...
       recycle   : local checkpoints deleted (= runtime recycled) and an incomplete 'latest.pt.tmp' planted in the Drive folder
       run B     : must resume from the Drive copy (the corrupt upload ignored), and is killed again after step c2
       run C     : must resume from the local checkpoint (more advanced than the Drive one) and finish.
    Checks: resume source / epoch / batch / global step; every step done exactly once with the reference's (epoch, batch); losses equal the
    reference's (same data order and RNG; GPU kernels are not bit-exact); same final state; no utterance repeated or skipped (asserted per epoch)."""
    import torch, gc
    root = Path(CFG["write_root"]) / "resume_test"
    if root.exists(): shutil.rmtree(root)
    trs = C["tr"].sort_values(["duration_s", "uid"]); tr = trs.iloc[np.linspace(0, len(trs) // 2, n_train).astype(int)].drop_duplicates("uid").reset_index(drop=True)
    dvs = C["dv"].sort_values(["duration_s", "uid"]); dv = dvs.iloc[np.linspace(0, len(dvs) // 2, n_dev).astype(int)].drop_duplicates("uid").reset_index(drop=True)
    Ct = dict(C, tr=tr, dv=dv, manifest_sha=sha_of(sorted(tr.uid)), dev_sha=sha_of(sorted(dv.uid)))
    LE, DE = 3, 7
    ov = dict(max_epochs=4, warmup_updates=4, local_ckpt_every_updates=LE, drive_ckpt_every_updates=DE, ckpt_every_minutes=1e9, log_every=1, max_batch_seconds=40.0,
              patience=99, max_session_hours=None)
    def mkP(name):
        P = dict(ckpt=root / name / "ckpt", logs=root / name / "logs", results=root / name / "results"); P.update(best=P["ckpt"] / "best", latest=P["ckpt"] / "latest")
        return P
    def go(P, ddir, interrupt=None):
        drv = DriveStore(ddir, fake=True); out = "finished"
        try: train(RUN_PRESETS[0], C=Ct, P=P, drive=drv, overrides=ov, interrupt_after_update=interrupt, probe=False)   # the optimizer actually used (ScaledAdam + Eden)
        except SimulatedDisconnect as e: out = str(e)
        drv.wait(); gc.collect(); torch.cuda.empty_cache(); return out
    def state(P): ck = torch.load(P["latest"] / "checkpoint.pt", map_location="cpu", weights_only=False); s = ck["state"]; del ck; return s
    log("---- reference run (uninterrupted)"); Pr = mkP("reference"); go(Pr, root / "drive_reference"); R = state(Pr)
    ref = {h["update"]: h for h in R["train_hist"]}; total = R["update"]; ends = {v["update"] for v in R["val_hist"]}
    lpts = {u for u in ref if u % LE == 0} | ends; dpts = {u for u in ref if u % DE == 0} | ends
    lastle = lambda pts, u: max([p for p in pts if p <= u], default=0)
    c1 = next((u for u in range(DE + 2, total) if u not in lpts | dpts), None)
    c2 = next((u for u in range(c1 + 2, total) if u not in lpts | dpts and lastle(lpts, u) > lastle(dpts, u) > lastle(dpts, c1)), None) if c1 else None
    if c2 is None: raise RuntimeError(f"resume test: the reference run has only {total} steps, too few to place the two interruptions")
    def expect(u):                       # saved position after step u
        return (ref[u]["epoch"], 0) if u in ends else (ref[u]["epoch"] - 1, ref[u]["batch"])
    Pb, Db = mkP("interrupted"), root / "drive_interrupted"; checks = []
    log(f"---- run A: killed after step {c1} (local checkpoint at step {lastle(lpts, c1)}, Drive at step {lastle(dpts, c1)})"); go(Pb, Db, c1)
    shutil.rmtree(Pb["ckpt"]); (Db / "latest.pt.tmp").write_bytes(os.urandom(1 << 20))
    log(f"---- runtime recycle simulated (local checkpoints deleted; incomplete upload planted) -> run B must resume from Drive at step {lastle(dpts, c1)}; killed after step {c2}")
    go(Pb, Db, c2)
    log(f"---- run C: must resume from the local checkpoint at step {lastle(lpts, c2)} (Drive has step {lastle(dpts, c2)}) and finish"); go(Pb, Db)
    F = state(Pb); res = F["resumes"]
    exp = [("Google Drive", lastle(dpts, c1)), ("local disk", lastle(lpts, c2))]
    checks.append(("resume sources and steps", [(r["source"], r["update"]) for r in res] == exp, f"{[(r['source'], r['update']) for r in res]} expected {exp}"))
    pos = [(r["epoch"], r["batch"]) for r in res]; epos = [expect(u) for _, u in exp]
    checks.append(("resumed epoch and batch position", pos == epos, f"{pos} expected {epos} (epoch index, batch index)"))
    upd = [h["update"] for h in F["train_hist"]]
    checks.append(("every optimizer step done exactly once", upd == list(range(1, total + 1)), f"{len(upd)} steps, reference {total}"))
    same_pos = all((h["epoch"], h["batch"]) == (ref[h["update"]]["epoch"], ref[h["update"]]["batch"]) for h in F["train_hist"])
    checks.append(("each step used the reference's epoch/batch", same_pos, ""))
    rel = {h["update"]: abs(h["l1"] - ref[h["update"]]["l1"]) / ref[h["update"]]["l1"] for h in F["train_hist"]}
    first = [rel[u + 1] for _, u in exp]
    checks.append(("first loss after each resume == reference", max(first) < 1e-2, f"relative differences {[f'{x:.1e}' for x in first]}"))
    checks.append(("all losses == reference", max(rel.values()) < 5e-2, f"max relative difference {max(rel.values()):.1e}"))
    checks.append(("final state == reference", F["epoch"] == R["epoch"] and F["update"] == R["update"] and bool(F["finished"])
                   and abs(F["val_hist"][-1]["val_l1"] - R["val_hist"][-1]["val_l1"]) / R["val_hist"][-1]["val_l1"] < 5e-2,
                   f"epochs {F['epoch']}/{R['epoch']}, steps {F['update']}/{R['update']}, final val L1 {F['val_hist'][-1]['val_l1']:.4f}/{R['val_hist'][-1]['val_l1']:.4f}"))
    checks.append(("incomplete upload ignored and removed", not (Db / "latest.pt.tmp").exists(), ""))
    bm = best_meta(Db / "best.pt"); vbest = min(v["val_l1"] for v in F["val_hist"])
    checks.append(("best.pt in the Drive folder holds the lowest validation loss", bm["val_l1"] <= vbest + ov.get("min_delta", CFG["min_delta"]) + 1e-9,
                   f"best.pt {bm['val_l1']:.5f}, lowest epoch val {vbest:.5f}"))
    for n, okk, info in checks: log(f"  [{'PASS' if okk else 'FAIL'}] {n}: {info}")
    rep = dict(passed=all(c[1] for c in checks), code_version=CODE_VERSION, time=time.strftime("%Y-%m-%d %H:%M:%S"), interrupts=[c1, c2], total_steps=total,
               checks=[dict(check=n, passed=bool(o), info=i) for n, o, i in checks])
    atomic_json(rep, paths()["logs"] / "resume_test.json")
    if rep["passed"]: shutil.rmtree(root); log("RESUME TEST PASSED (test folders removed)")
    else: log(f"RESUME TEST FAILED: training is blocked (test folders kept in {root})")
    return rep

# ####################################################################################################
# ---- plots + final report
# ####################################################################################################
def plots(P):
    """Training curves (local disk only). Columns are read with [...] (T.update would be the DataFrame.update METHOD).
    A plotting problem is reported but never stops training."""
    try:
        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
        ft, fv = P["logs"] / "train_history.csv", P["logs"] / "history.csv"
        if not fv.exists() or fv.stat().st_size < 40: return
        V = pd.read_csv(fv); T = pd.read_csv(ft) if ft.exists() and ft.stat().st_size > 10 else None
        if T is not None and not len(T): T = None
        best = V["best"].astype(str).str.lower().eq("true")
        fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))
        if T is not None: ax[0].plot(T["update"], T["l1"], alpha=0.5, label="train L1 (logged steps)")
        ax[0].plot(V["global_step"], V["train_loss"], "o-", label="train L1 (epoch)"); ax[0].plot(V["global_step"], V["val_loss"], "o-", label="validation L1 (dev-clean)")
        if best.any():
            b = V[best].iloc[-1]; ax[0].axvline(b["global_step"], c="green", ls=":", label=f"best (epoch {int(b['epoch'])})")
        ax[0].set(title="ASR KD loss", xlabel="global step", ylabel="L1"); ax[0].legend()
        if T is not None:
            ax[1].semilogy(T["update"], T["grad_norm"].clip(lower=1e-4)); ax[1].set(title="gradient norm (before clipping)", xlabel="global step")
            ax[2].plot(T["update"], T["lr"]); ax[2].set(title="learning rate", xlabel="global step")
        fig.tight_layout(); fig.savefig(P["results"] / "training_curves.png", dpi=110); plt.close(fig)
    except Exception as e:
        log(f"  (plot not updated: {type(e).__name__}: {e})")

def make_report(preset, C, P, drive=None, verify_best=True):
    import torch
    LATEST, BEST = P["latest"] / "checkpoint.pt", P["best"] / "best.pt"
    if not LATEST.exists(): log("no training state yet"); return None
    ck = torch.load(LATEST, map_location="cpu", weights_only=False); S = ck["state"]; del ck
    rc = json.load(open(P["logs"] / "run_config.json")); V = pd.DataFrame(S["val_hist"]); d = rc["data"]
    rep = dict(
        status=("COMPLETED: " + S["finished"]) if S["finished"] else
               f"INTERRUPTED / NOT FINISHED at epoch {S['epoch'] + 1}, batch {S['batch']}, global step {S['update']} (re-run the cell to resume)",
        training_completed=bool(S["finished"]),
        dataset=dict(train=f"LibriSpeech train-clean-100, reproducible '{d['train_subset']}' subset (seed {rc['seed']}, target {d['target_seconds']:.0f} s)",
                     train_utterances=d["train_utterances"], train_seconds=d["train_seconds"], train_speakers=d["train_speakers"],
                     validation=f"{d['validation']['split']}: {d['validation']['utterances']} utterances, {d['validation']['hours']} h",
                     test="test-clean not used in Stage 1", discarded=d["discarded"]),
        teacher=rc["teacher"], student=rc["student"],
        feature_dimensions=dict(student_input="80-D fbank @ 100 Hz", student_output="512-D @ 25 Hz", projection="512 -> 2048", teacher="WavLM-Large 1024-D @ 50 Hz -> 2048-D @ 25 Hz"),
        distillation_loss=rc["loss"], alignment=rc["alignment"],
        epochs_completed=S["epoch"], total_optimizer_steps=S["update"], best_validation=S["best"],
        last_epoch=dict(train_l1=float(V.train_l1.iloc[-1]), val_l1=float(V.val_l1.iloc[-1]), val_mean_baseline_l1=float(V.val_base_mean_l1.iloc[-1]),
                        val_median_baseline_l1=float(V.val_base_median_l1.iloc[-1]), val_cos=float(V.val_cos.iloc[-1])) if len(V) else None,
        overfitting_flags=[f"epoch {int(r.epoch)}: {r.overfit_flags}" for r in V.itertuples() if isinstance(r.overfit_flags, str) and r.overfit_flags] if len(V) else [],
        early_stopping=dict(counter=S["es_counter"], patience=rc["training"]["patience"]),
        training_time_hours=round(S["elapsed"] / 3600, 3), sessions=S["sessions"], resumes=S.get("resumes", []),
        checkpoints=dict(drive_best=str(drive.dir / "best.pt") if drive else None, drive_latest=str(drive.dir / "latest.pt") if drive else None,
                         local_best=str(BEST), local_latest=str(LATEST)),
        drive_uploads=dict(drive.done) if drive else None,
        not_paper_exact=["ENHANCEMENT, not the paper: teacher WavLM-Large instead of Whisper-large-v3",
                         "reduced-data pilot: 2 h subset of train-clean-100, not the paper's training data",
                         "optimizer/schedule: ScaledAdam + Eden as in the paper / icefall (identical to the Whisper v4 run)",
                         "k2 not used: Swoosh activations are a pure-PyTorch port of k2's implementation, verified numerically",
                         "frame pairing (student j <-> teacher j+offset) derived from measured frame centres; not verified against the authors' code",
                         "projection bias initialised to the mean teacher frame (engineering choice)",
                         "WavLM: every utterance encoded whole (no chunking); odd last 50 Hz frame dropped by the pairing"],
        note="Stage 1 = feature-level distillation; the CTC probe of this cell gives a frozen-encoder WER for the teacher comparison (not the final Stage 2 WER).")
    if verify_best and BEST.exists():
        dev = torch.device("cuda"); student, proj = new_models(C, dev); b = torch.load(BEST, map_location=dev, weights_only=False)
        student.load_state_dict(b["student"], strict=True); proj.load_state_dict(b["asr_proj"], strict=True)
        v = evaluate(student, proj, C, dev, amp_dtype()); del student, proj; torch.cuda.empty_cache()
        rep["final_evaluation_best_checkpoint"] = dict(split=CFG["val_split"], epoch=b["epoch"], global_step=b["update"], val_l1=v["val_l1"], stored_val_l1=b["val"]["val_l1"],
                                                       reload_matches=abs(v["val_l1"] - b["val"]["val_l1"]) < 2e-3, val_cos=v["val_cos"],
                                                       ratio_to_mean_baseline=v["val_ratio_mean"], ratio_to_median_baseline=v["val_ratio_median"])
    atomic_json(rep, P["results"] / "final_report.json"); plots(P)
    if drive: drive.write_json("final_report.json", rep)
    log("\n" + "=" * 90 + f"\nWavLM ASR STAGE 1 FINAL REPORT\n" + "=" * 90)
    for k in ["status", "dataset", "epochs_completed", "total_optimizer_steps", "best_validation", "last_epoch", "final_evaluation_best_checkpoint", "training_time_hours",
              "sessions", "resumes", "overfitting_flags", "checkpoints"]:
        if k in rep: log(f"{k:34s}: {rep[k]}")
    log(f"(full report: {P['results'] / 'final_report.json'}" + (f" and {drive.dir / 'final_report.json'}" if drive else "") + ")")
    return rep


# ####################################################################################################
# ---- CTC PROBE: does the WavLM-taught student carry more SPEECH-RECOGNITION information than the Whisper-taught one?
# ####################################################################################################
# Each student's encoder is FROZEN; only a small probe is trained on top of it, identically for every student:
#   student output (512-D @ 25 Hz) -> per-dimension standardisation (train statistics) -> each frame repeated twice (50 Hz, so that
#   CTC has room for repeated letters) -> 2-layer BiLSTM -> Linear -> 29 symbols (CTC blank, space, apostrophe, A-Z) -> CTC loss.
#   Data: the 10 h subset of train-clean-100 (it contains the 2 h KD subset) with its transcripts; evaluation: the whole dev-clean, greedy
#   decoding, WER and CER computed by this code. Fixed seed, fixed number of epochs (no model selection on dev-clean), same settings for all.
#   Students compared: WavLM-taught (this cell), Whisper-taught (your ASR v4 sanity_2h best.pt, read-only), random-init (control = no teacher).
VOCAB = ["<blank>", " ", "'"] + [chr(c) for c in range(ord("A"), ord("Z") + 1)]
CHAR2ID = {c: i for i, c in enumerate(VOCAB)}

def text_to_ids(t):
    t = " ".join(str(t).upper().split()); bad = sorted(set(t) - set(CHAR2ID))
    if bad: raise ValueError(f"unexpected characters {bad} in transcript {t[:60]!r}")
    return [CHAR2ID[c] for c in t]

def ctc_min_frames(ids):
    """Fewest frames a CTC path needs: one per label plus a blank between two equal neighbours."""
    return len(ids) + sum(1 for a, b in zip(ids, ids[1:]) if a == b)

def greedy_decode(best):
    """best: list of argmax symbol ids per frame -> text (repeats collapsed, blanks removed)."""
    out = []; prev = -1
    for k in best:
        if k != prev and k != 0: out.append(VOCAB[k])
        prev = k
    return " ".join("".join(out).split())

def edit_distance(a, b):
    if len(a) < len(b): a, b = b, a
    prev = list(range(len(b) + 1))
    for i, x in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, y in enumerate(b, 1): cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x != y))
        prev = cur
    return prev[-1]

def error_rates(refs, hyps, with_cer=True):
    refs = [" ".join(r.upper().split()) for r in refs]; hyps = [" ".join(h.split()) for h in hyps]
    we = sum(edit_distance(r.split(), h.split()) for r, h in zip(refs, hyps)); wn = sum(len(r.split()) for r in refs)
    R = dict(wer=we / max(wn, 1), word_errors=we, words=wn, utterances=len(refs))
    if with_cer:
        ce = sum(edit_distance(list(r), list(h)) for r, h in zip(refs, hyps)); cn = sum(len(r) for r in refs); R.update(cer=ce / max(cn, 1), char_errors=ce, chars=cn)
    return R

class _FbankDS:
    def __init__(self, df): self.r = df.to_dict("records")
    def __len__(self): return len(self.r)
    def __getitem__(self, i):
        r = self.r[i]; w = load_audio(r["relpath"], int(r["n_samples"])); return dict(uid=r["uid"], n=len(w), fb=fbank(w))
def _fb_collate(b):
    import torch, math
    from torch.nn.utils.rnn import pad_sequence
    return dict(uid=[x["uid"] for x in b], n=torch.tensor([x["n"] for x in b]),
                fb=pad_sequence([x["fb"] for x in b], batch_first=True, padding_value=math.log(1e-10)), fb_lens=torch.tensor([x["fb"].size(0) for x in b]))

def student_features(student, df, dev, amp):
    """Frozen student encoder outputs for every utterance of df -> {uid: float16 CPU tensor [T, 512]} (25 Hz)."""
    import torch
    student.eval(); out = {}
    bt = make_batches(df.n_samples.values, 0, CFG["seed"], False, 300.0, CFG["max_batch_utts"])
    dl = torch.utils.data.DataLoader(_FbankDS(df), batch_sampler=bt, num_workers=CFG["num_workers"], collate_fn=_fb_collate,
                                     prefetch_factor=4 if CFG["num_workers"] else None)
    with torch.no_grad():
        for b in dl:
            with torch.autocast(device_type="cuda", dtype=amp): enc, el = student(b["fb"].to(dev), b["fb_lens"].to(dev))
            enc = enc.float()
            if not torch.isfinite(enc).all(): raise FloatingPointError("non-finite student output")
            for k, u in enumerate(b["uid"]):
                L = int(el[k]); assert L == student_frames(int(b["n"][k])), (u, L); out[u] = enc[k, :L].half().cpu()
    assert len(out) == len(df), f"features for {len(out)} of {len(df)} utterances"
    return out

def build_probe(d_in):
    import torch, torch.nn as nn
    from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
    class CTCProbe(nn.Module):
        def __init__(s):
            super().__init__(); s.register_buffer("mu", torch.zeros(d_in)); s.register_buffer("sd", torch.ones(d_in))
            s.lstm = nn.LSTM(d_in, CFG["probe_hidden"], num_layers=CFG["probe_layers"], bidirectional=True, batch_first=True, dropout=CFG["probe_dropout"])
            s.drop = nn.Dropout(CFG["probe_dropout"]); s.out = nn.Linear(2 * CFG["probe_hidden"], len(VOCAB))
        def forward(s, x, lens):                         # x [B, T, 512] @ 25 Hz -> log-probs [B, 2T, 29] @ 50 Hz, lengths 2 * lens
            x = ((x.float() - s.mu) / s.sd).repeat_interleave(2, dim=1); L = lens * 2
            p = pack_padded_sequence(x, L.cpu(), batch_first=True, enforce_sorted=False)
            y, _ = s.lstm(p); y, _ = pad_packed_sequence(y, batch_first=True, total_length=x.size(1))
            return s.out(s.drop(y)).float().log_softmax(-1), L
    return CTCProbe()

def _probe_batches(uids, F, epoch, shuffle, max_sec):
    lens = np.array([len(F[u]) * HOP_T for u in uids])              # duration-equivalent lengths (samples) of the 25 Hz features
    return make_batches(lens, epoch, CFG["probe_seed"], shuffle, max_sec, 256)

def _probe_batch(uids, F, labels, idx, dev):
    import torch
    from torch.nn.utils.rnn import pad_sequence
    us = [uids[i] for i in idx]; x = pad_sequence([F[u].float() for u in us], batch_first=True).to(dev)
    lens = torch.tensor([len(F[u]) for u in us], device=dev)
    if labels is None: return us, x, lens, None, None
    tg = [labels[u] for u in us]
    return us, x, lens, torch.tensor([c for t in tg for c in t], device=dev, dtype=torch.long), torch.tensor([len(t) for t in tg], device=dev, dtype=torch.long)

def probe_decode(probe, uids, F, dev):
    import torch
    hyps = {}; probe.eval()
    with torch.no_grad():
        for idx in _probe_batches(uids, F, 0, False, CFG["probe_batch_seconds"]):
            us, x, lens, _, _ = _probe_batch(uids, F, None, idx, dev); lp, L = probe(x, lens); best = lp.argmax(-1).cpu()
            for k, u in enumerate(us): hyps[u] = greedy_decode(best[k, :int(L[k])].tolist())
    probe.train(); return hyps

def run_probe(name, Ftr, Fdv, tr_df, dv_df, dev):
    """Trains one CTC probe (identical settings and seed for every student) and evaluates it on dev-clean."""
    import torch, torch.nn.functional as Fn
    t0 = time.time(); set_seed(CFG["probe_seed"])
    labels = {u: text_to_ids(t) for u, t in zip(tr_df.uid, tr_df.text)}
    feasible = [u for u in tr_df.uid if 2 * len(Ftr[u]) >= ctc_min_frames(labels[u])]; n_inf = len(tr_df) - len(feasible)
    s = torch.zeros(D_S, dtype=torch.float64); s2 = torch.zeros(D_S, dtype=torch.float64); nfr = 0
    for u in feasible: x = Ftr[u].double(); s += x.sum(0); s2 += (x * x).sum(0); nfr += len(x)
    mu = s / nfr; sd = (s2 / nfr - mu * mu).clamp(min=1e-8).sqrt()
    probe = build_probe(D_S).to(dev); probe.mu.copy_(mu.float()); probe.sd.copy_(sd.float())
    opt = torch.optim.Adam(probe.parameters(), lr=CFG["probe_lr"])
    nb = len(_probe_batches(feasible, Ftr, 0, True, CFG["probe_batch_seconds"])); total = nb * CFG["probe_epochs"]; warm = max(1, min(300, total // 10))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda st: min(1.0, (st + 1) / warm) * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(1.0, st / total)))))
    hist = []; step = 0
    log(f"  [{name}] probe: {len(feasible):,} training utterances ({n_inf} too short for their transcript at 50 Hz, excluded), {nb} batches/epoch, "
        f"{CFG['probe_epochs']} epochs, BiLSTM {CFG['probe_layers']} x {CFG['probe_hidden']}, Adam {CFG['probe_lr']}")
    for ep in range(CFG["probe_epochs"]):
        tl = tn = 0.0
        for idx in _probe_batches(feasible, Ftr, ep, True, CFG["probe_batch_seconds"]):
            us, x, lens, tg, tl_ = _probe_batch(feasible, Ftr, labels, idx, dev)
            lp, L = probe(x, lens)
            loss = Fn.ctc_loss(lp.transpose(0, 1), tg, L, tl_, blank=0, reduction="sum", zero_infinity=True) / tl_.sum()
            if not torch.isfinite(loss): opt.zero_grad(set_to_none=True); continue
            opt.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(probe.parameters(), 5.0); opt.step(); sched.step(); step += 1
            tl += float(loss) * float(tl_.sum()); tn += float(tl_.sum())
        rec = dict(probe=name, epoch=ep + 1, step=step, train_ctc_per_char=tl / max(tn, 1), lr=sched.get_last_lr()[0])
        if (ep + 1) % 5 == 0 or ep + 1 == CFG["probe_epochs"]:
            hy = probe_decode(probe, list(dv_df.uid), Fdv, dev); rec["dev_wer_greedy"] = error_rates(list(dv_df.text), [hy[u] for u in dv_df.uid], with_cer=False)["wer"]
        hist.append(rec)
        log(f"  [{name}] epoch {ep + 1}/{CFG['probe_epochs']}: train CTC {rec['train_ctc_per_char']:.4f} per character"
            + (f" | dev-clean WER {rec['dev_wer_greedy'] * 100:.2f}%" if "dev_wer_greedy" in rec else "") + f" | {(time.time() - t0) / 60:.1f} min")
    hy = probe_decode(probe, list(dv_df.uid), Fdv, dev); er = error_rates(list(dv_df.text), [hy[u] for u in dv_df.uid])
    ex = [dict(uid=u, ref=r, hyp=hy[u]) for u, r in list(zip(dv_df.uid, dv_df.text))[:3]]
    del probe, opt; torch.cuda.empty_cache()
    return dict(name=name, dev_clean_wer=er["wer"], dev_clean_cer=er["cer"], errors=er, train_utterances=len(feasible), excluded_too_short=n_inf,
                epochs=CFG["probe_epochs"], steps=step, minutes=round((time.time() - t0) / 60, 2), examples=ex), hist

def load_student_ckpt(path):
    """A trained student from a best.pt (read-only): returns (student on CPU, info)."""
    import torch
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(ck, dict) or "student" not in ck: raise RuntimeError(f"{path} has no 'student' weights")
    if ck.get("zip_args") is not None and dict(ck["zip_args"]) != dict(ZIP_ARGS): raise RuntimeError(f"{path}: Zipformer arguments differ from this cell's")
    set_seed(CFG["seed"]); s, _ = build_models(1); s.load_state_dict(ck["student"], strict=True)
    v = ck.get("val") or {}
    info = dict(path=str(path), code_version=ck.get("code_version"), teacher=(ck.get("teacher") or {}).get("model"), epoch=ck.get("epoch"), update=ck.get("update"),
                kd_val_l1=v.get("val_l1"), kd_ratio_to_mean_baseline=v.get("val_ratio_mean"), kd_val_cos=v.get("val_cos"), manifest_sha=ck.get("manifest_sha"),
                file_sha1=file_sha1(path)[:16])
    del ck; return s, info

def probe_comparison(wavlm_best, drive=None):
    """Runs the CTC probe for: WavLM-taught student, Whisper-taught student (if its best.pt exists), random-init student. Resumable per student."""
    import torch, gc
    dev = torch.device("cuda"); amp = amp_dtype(); Pp = paths()["logs"] / "ctc_probe"; Pp.mkdir(parents=True, exist_ok=True)
    tr = load_manifest(CFG["probe_train_subset"]); dv = load_manifest("dev"); s2 = load_manifest(PRESETS[RUN_PRESETS[0]]["train_subset"])
    materialize(tr, "train-clean-100"); materialize(dv, CFG["val_split"])
    pcfg = {k: CFG[k] for k in CFG if k.startswith("probe_")}
    log(f"  probe data: {CFG['probe_train_subset']} = {len(tr):,} utterances / {tr.duration_s.sum() / 3600:.2f} h (contains the {len(s2):,}-utterance KD subset: "
        f"{set(s2.uid) <= set(tr.uid)}) | evaluation: dev-clean {len(dv):,} utterances | settings {pcfg}")
    students = [("WavLM-taught", wavlm_best)]
    wb = Path(CFG["whisper_student_ckpt"])
    if wb.exists(): students.append(("Whisper-taught", wb))
    else: log(f"  Whisper-taught student not found at {wb}: it is skipped (set CFG['whisper_student_ckpt'] to its best.pt)")
    students.append(("random-init (no teacher)", None))
    old = {}
    if drive is not None and (drive.dir / "final_report.json").exists():
        try: old = {r["name"]: r for r in json.load(open(drive.dir / "final_report.json")).get("results", [])}
        except Exception: old = {}
    results, hist_all = [], []
    if (Pp / "history.csv").exists():
        try: hist_all = pd.read_csv(Pp / "history.csv").to_dict("records")
        except Exception: hist_all = []
    for name, ck in students:
        if ck is None: set_seed(CFG["seed"]); st, _ = build_models(1); info = dict(path=None, teacher=None, note="Zipformer-M with random weights (seed %d), never trained" % CFG["seed"])
        else: st, info = load_student_ckpt(ck)
        if info.get("manifest_sha") and info["manifest_sha"] != sha_of(sorted(s2.uid)): log(f"  NOTE [{name}]: its KD training subset differs from this cell's 2 h subset")
        sig = sha_of([json.dumps(pcfg, sort_keys=True), info.get("file_sha1"), name, sha_of(sorted(tr.uid)), sha_of(sorted(dv.uid)), CODE_VERSION])
        if name in old and old[name].get("sig") == sig:
            log(f"  [{name}] already done (same checkpoint and probe settings): WER {old[name]['dev_clean_wer'] * 100:.2f}% -> reused"); results.append(old[name]); continue
        log(f"\n  ---- [{name}] {info.get('path') or 'random weights'} (KD teacher: {info.get('teacher')}, KD epoch {info.get('epoch')}, "
            f"KD ratio to mean baseline {info.get('kd_ratio_to_mean_baseline')})")
        st = st.to(dev); t0 = time.time(); Ftr = student_features(st, tr, dev, amp); Fdv = student_features(st, dv, dev, amp)
        st.cpu(); del st; torch.cuda.empty_cache(); log(f"  [{name}] frozen encoder features extracted in {(time.time() - t0) / 60:.1f} min")
        r, h = run_probe(name, Ftr, Fdv, tr, dv, dev); r.update(student=info, sig=sig); results.append(r)
        hist_all = [x for x in hist_all if x.get("probe") != name] + h
        del Ftr, Fdv; gc.collect()
        atomic_json(r, Pp / f"{sha_of([name])}.json"); pd.DataFrame(hist_all).to_csv(Pp / "history.csv", index=False)
        rep = dict(code_version=CODE_VERSION, updated=time.strftime("%Y-%m-%d %H:%M:%S"), probe_settings=pcfg,
                   data=dict(train=f"LibriSpeech train-clean-100 {CFG['probe_train_subset']} subset ({len(tr)} utterances)", evaluation=f"dev-clean ({len(dv)} utterances)"),
                   results=list(results), status="in progress (re-run the cell to continue with the remaining students)")
        if drive is not None: drive.write_json("final_report.json", rep); drive.write_text("history.csv", pd.DataFrame(hist_all).to_csv(index=False))
        log(f"  [{name}] dev-clean WER {r['dev_clean_wer'] * 100:.2f}% | CER {r['dev_clean_cer'] * 100:.2f}% | probe {r['minutes']:.1f} min | example: "
            f"REF {r['examples'][0]['ref'][:70]!r} / HYP {r['examples'][0]['hyp'][:70]!r}")
    rep = dict(code_version=CODE_VERSION, updated=time.strftime("%Y-%m-%d %H:%M:%S"), probe_settings=pcfg,
               data=dict(train=f"LibriSpeech train-clean-100 {CFG['probe_train_subset']} subset ({len(tr)} utterances)", evaluation=f"dev-clean ({len(dv)} utterances)"),
               results=results,
               reading=("lower WER = the frozen student carries more speech-recognition information. Same student architecture, same 2 h KD audio, same KD "
                        "recipe; only the teacher differs (the random-init row shows what the probe gets from an untrained encoder). This is a probe on a frozen "
                        "Stage-1 encoder, not the final MT2KD ASR WER (Stage 2 fine-tunes the encoder with a transducer)."))
    atomic_json(rep, Pp / "final_report.json")
    if drive is not None:
        drive.write_json("final_report.json", rep); drive.write_text("history.csv", pd.DataFrame(hist_all).to_csv(index=False))
        drive.write_json("config.json", dict(probe_settings=pcfg, students=[dict(name=n, checkpoint=str(c) if c else None) for n, c in students], code_version=CODE_VERSION))
    return rep

# ####################################################################################################
# ---- RUN: WavLM KD on 2 h (same as the Whisper v4 sanity_2h run) -> CTC probe comparison. After ANY interruption: run this cell again.
# ####################################################################################################
# CFG["teacher_layer"] = 20                # optional: another WavLM layer (new cache + new run; use a NEW drive folder name, see drive_dir)
# CFG["max_session_hours"] = 2.0           # optional: stop training after ~2 h per cell run (state saved locally + on Drive; re-run to continue)
# CFG["drive_api"] = False                 # optional: skip the Google sign-in pop-up (uploads not md5-confirmed; replaced checkpoints stay in the Drive trash)

import gc
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive as _gd; _gd.mount("/content/drive")
pip_if_missing("soundfile", "pandas", "matplotlib", "transformers")
import torch
for k in ("write_root", "local_audio", "local_features", "zipformer_dir"): assert_local(CFG[k])
assert RUN_PRESETS and all(p in PRESETS for p in RUN_PRESETS), RUN_PRESETS
def _step(t): log("\n" + "=" * 110 + f"\n{t}\n" + "=" * 110)
def _cache_ready(split, df):
    try: TeacherCache(find_cache(split), df.uid, local=False); return True
    except Exception: return False

_step("STEP 1/11  environment + Google Drive storage check (the KD run writes ONLY 5 files into " + drive_dir("<run>") + ")")
env = env_report()
drives = {p: DriveStore(drive_dir(p), use_api=CFG["drive_api"]) for p in RUN_PRESETS}
_s, _ = build_models(2 * WAVLM_DIM); _npar = sum(p.numel() for p in _s.parameters()) + sum(p.numel() for p in _.parameters()); del _s, _
_need = 0
for p, d in drives.items():
    if os.path.realpath(str(d.dir)) != os.path.normpath(str(d.dir)): log(f"  (the path goes through a Drive shortcut: files land in {os.path.realpath(str(d.dir))})")
    n_, keep_ = drive_need_bytes(_npar, d); _need += n_; d.status(expected_bytes=keep_)
    if d.candidates("latest.pt"): log(f"  [{p}] a training checkpoint exists on Drive: this run will RESUME from it")
drives[RUN_PRESETS[0]].check_space(_need, "best.pt + latest.pt (each replaced via a temporary copy)")
log(f"  Whisper-taught student for the comparison: {CFG['whisper_student_ckpt']} -> {'found' if Path(CFG['whisper_student_ckpt']).exists() else 'NOT FOUND (that row will be skipped)'}")

_step("STEP 2/11  dataset integrity + reproducible nested subsets of train-clean-100 (2 h for KD, 10 h for the probe) + full dev-clean (local disk)")
sel = prepare_manifests()
train_sets = {p: load_manifest(PRESETS[p]["train_subset"]) for p in RUN_PRESETS}
s2, dev = load_manifest("2h"), load_manifest("dev")
s_all = max(train_sets.values(), key=len)                          # the KD training set (2 h)
for p, df in train_sets.items():
    log(f"  [{p}] {len(df):,} utterances, {df.duration_s.sum():.1f} s ({df.duration_s.sum() / 3600:.3f} h), {df.speaker.nunique()} speakers")
log(f"  validation dev-clean: {len(dev):,} utterances, {dev.duration_s.sum():.0f} s")

need_T = not _verification_file().exists() or not ALIGN_F().exists() or not _cache_ready(CFG["val_split"], dev) or not _cache_ready("train-clean-100", s_all)
T = WavLMTeacher() if need_T else None
_step("STEP 3/11  WavLM-Large (frozen): preprocessing / frame rule / layer / pairing / long utterances / repeatability / precision on real audio")
if not _verification_file().exists(): teacher_check = verify_wavlm(T, s2, dev)
else: log("already verified in this runtime -> " + str(_verification_file()))

_step(f"STEP 4/11  WavLM teacher features on LOCAL disk: dev-clean + {len(s_all):,} training utterances "
      f"(~{sum(teacher_frames(n) for n in pd.concat([s_all, dev]).n_samples) * 4096 / 2**30:.1f} GiB; resumable, integrity-checked, regenerated after a recycle)")
materialize(dev, CFG["val_split"]); materialize(s_all, "train-clean-100")
if not _cache_ready(CFG["val_split"], dev): build_cache(T, dev, CFG["val_split"])
else: log("dev-clean features complete")
if not _cache_ready("train-clean-100", s_all): build_cache(T, s_all, "train-clean-100")
else: log("training features complete")

_step("STEP 5/11  teacher/student frame alignment, measured from the imported WavLM and Zipformer code")
if not ALIGN_F().exists(): alignment = measure_alignment(T, s2, dev)
else: alignment = load_alignment(); log(f"already measured: student j <-> teacher j+{alignment['offset']} ({alignment['residual_ms']:+.2f} ms)")
if not alignment.get("passed"): raise RuntimeError("the alignment measurement did not pass (see the lines above): training blocked")
if T is not None: T.enc.cpu(); del T
gc.collect(); torch.cuda.empty_cache()

_step("STEP 6/11  Swoosh (k2 replacement, pure PyTorch) numerical verification")
swoosh = verify_swoosh()
if not swoosh["passed"]: raise RuntimeError("Swoosh verification failed: training blocked")

_step("STEP 7/11  SMOKE TEST (21 checks incl. a real ScaledAdam step; any critical failure blocks training)")
try: _gate(context(RUN_PRESETS[0])); log("smoke test already passed in this runtime for this code, features and alignment -> skipped")
except Exception as e:
    log(f"(smoke test needed: {e})"); smoke = smoke_test(RUN_PRESETS[0])
    if not smoke["all_critical_passed"]: raise RuntimeError(f"smoke test critical failures {smoke['critical_failed']}: training blocked")

_step("STEP 8/11  RESUME TEST with ScaledAdam + Eden: deliberately interrupted short runs must continue at the right epoch / batch / step")
_rt = paths()["logs"] / "resume_test.json"
if _rt.exists() and json.load(open(_rt)).get("passed") and json.load(open(_rt)).get("code_version") == CODE_VERSION: log("already passed in this runtime -> skipped")
else:
    rt = resume_test(context(RUN_PRESETS[0])); gc.collect(); torch.cuda.empty_cache()
    if not rt["passed"]: raise RuntimeError("resume test failed: training blocked (see the lines above)")

reports = {}
for i_, PRESET in enumerate(RUN_PRESETS):
    pc = dict(CFG, **PRESETS[PRESET])
    _step(f"STEP 9/11  KD TRAINING '{PRESET}' (WavLM teacher) on the {pc['train_subset']} subset ({len(train_sets[PRESET]):,} utterances, max {pc['max_epochs']} epochs, "
          f"batches <= {pc['max_batch_seconds']:.0f} s, {pc['optimizer']}; Drive folder {drive_dir(PRESET)})")
    rep = train(PRESET, drive=drives[PRESET]); reports[PRESET] = rep; gc.collect(); torch.cuda.empty_cache()
    try:
        from IPython.display import Image, display; display(Image(str(paths(PRESET)["results"] / "training_curves.png")))
    except Exception: pass
    if not rep or not rep["training_completed"]:
        log(f"'{PRESET}' not finished: re-run this cell to continue it from the last checkpoint"); break
    fe = rep.get("final_evaluation_best_checkpoint") or {}; ratio = fe.get("ratio_to_mean_baseline")
    if ratio is None: ratio = float("nan")
    log(f"  [{PRESET}] best dev-clean L1 {fe.get('val_l1', float('nan')):.4f} | constant-mean baseline ratio {ratio:.3f} (1.0 = learned nothing; "
        f"Whisper v4 sanity_2h: 0.792) | cosine {fe.get('val_cos', float('nan')):.3f}")
    if "gate_ratio" in pc:
        if ratio < pc["gate_ratio"]: log(f"  GATE PASSED ({ratio:.3f} < {pc['gate_ratio']}): the student learns from WavLM")
        else: log(f"  NOTE: ratio {ratio:.3f} is not below {pc['gate_ratio']} (the student learned little from WavLM); the CTC probe still runs and shows what it means for ASR")

wavlm_best = paths(RUN_PRESETS[0])["best"] / "best.pt"
done_kd = bool(reports.get(RUN_PRESETS[0]) and reports[RUN_PRESETS[0]].get("training_completed"))
probe_rep = None
_step("STEP 10/11  CTC PROBE: WavLM-taught vs Whisper-taught vs random-init student (frozen encoders, same BiLSTM-CTC, 10 h with transcripts, WER on dev-clean)")
if not done_kd: log("KD training is not finished yet: the probe runs after it (re-run this cell)")
elif not wavlm_best.exists(): log(f"best.pt of the WavLM run not found at {wavlm_best}: re-run this cell")
else:
    pdrive = DriveStore(probe_drive_dir(), use_api=CFG["drive_api"])
    probe_rep = probe_comparison(wavlm_best, drive=pdrive); pdrive.wait()

_step("STEP 11/11  summary")
for PRESET, rep in reports.items():
    fe = (rep or {}).get("final_evaluation_best_checkpoint") or {}
    log(f"  KD {PRESET:10s}: {(rep or {}).get('status')} | best dev-clean L1 {fe.get('val_l1', float('nan')):.4f} | ratio to mean baseline "
        f"{fe.get('ratio_to_mean_baseline', float('nan')):.3f} | cosine {fe.get('val_cos', float('nan')):.3f} | {drive_dir(PRESET)}")
    drives[PRESET].status()
log("  (the KD L1 values of the WavLM and Whisper runs are on different teacher features and are NOT comparable; the probe WER below is)")
if probe_rep:
    log(f"\n  CTC probe on frozen students (dev-clean, greedy decoding, {CFG['probe_epochs']} epochs on {CFG['probe_train_subset']}):")
    log(f"  {'student':28s} {'WER':>8s} {'CER':>8s}   KD teacher / checkpoint")
    for r in probe_rep["results"]:
        log(f"  {r['name']:28s} {r['dev_clean_wer'] * 100:7.2f}% {r['dev_clean_cer'] * 100:7.2f}%   {(r.get('student') or {}).get('teacher')} / {(r.get('student') or {}).get('path')}")
    log("  " + probe_rep["reading"])
    log(f"  saved: {probe_drive_dir()}/final_report.json (+ history.csv, config.json)")
